<center>

<img src='https://micro.ce.sharif.edu/lib/tpl/writr/images/logo.svg' alt="SUT logo" width=500 height=300 align=center class="saturate" >


<br>
<font>
<div dir=ltr align=center>
<font color=0F5298 size=7>
Machine Learning <br>
<font color=2565AE size=5>
Computer Engineering Department <br>
Dr. Sharifi-Zarchi, Fall 2025 <br>
<font color=3C99D size=5>
Practical Assignment 5 - Natural Language Processing <br>
<font color=696880 size=4>
By: Sina Bayrami

</center>


# Notebook Overview — NER from BiLSTM to (Tiny)BERT

This notebook is a **hands-on, end-to-end practical lab** for a core NLP task: **Named Entity Recognition (NER)**.  
You will start from a strong non-Transformer baseline and gradually build up the ideas that lead to modern Transformer encoders and BERT-style pretraining—then you will compare those approaches fairly on the **same NER problem**.

The goal is not to “memorize architectures”, but to **learn what each ingredient adds**, why it matters, and how to evaluate models correctly.

## 1) What is NER and what are we trying to do?


**Named Entity Recognition (NER)** is a **token-level sequence labeling** task.

- **Input:** a sentence as a sequence of tokens  
  $$
  x = (x_1, x_2, \dots, x_T)
  $$
- **Output:** a label for each token  
  $$
  y = (y_1, y_2, \dots, y_T)
  $$
  where labels usually follow a BIO-style format (e.g., **B-PERSON**, **I-ORG**, **O**).

**What we want:** learn a model $f_\theta$ that predicts the correct tag for each token:
$$
\hat{y}_t = \arg\max_k \; p_\theta(y_t = k \mid x)
$$

NER is difficult because:
- Entities can span multiple tokens (e.g., “New York City”)
- Context matters (e.g., “Jordan” can be a person or a location)
- Most tokens are **not** entities → the label **O** is dominant

## 2) How we evaluate: Metrics (and why F1 is the main one)

In NER, **token accuracy** can be misleading because a model can predict **O** almost everywhere and still look “good”.

So we use two families of metrics:

### A) Token Accuracy (quick sanity metric)
Token accuracy is:
$$
\text{Acc}=\frac{\#\{t:\hat{y}_t=y_t\}}{T}
$$
But because **O** dominates, accuracy often hides failures on entities.

### B) Entity-level Precision / Recall / F1 (the main metric)
We care about **entities as spans**, not just individual tokens.

From a BIO tag sequence we extract a set of predicted entity spans $\hat{E}$ and gold spans $E$.  
A predicted entity is counted as **correct (TP)** only if:
- the span boundaries match exactly, and
- the entity type matches exactly

Define:
- $TP$: number of correctly predicted entities  
- $FP$: predicted entities not in gold  
- $FN$: gold entities missed by the model  

Then:
$$
P=\frac{TP}{TP+FP},\qquad
R=\frac{TP}{TP+FN}
$$
and the **F1-score** is the harmonic mean:
$$
F1 = \frac{2PR}{P+R}
$$
F1 heavily penalizes models that:
- miss entities (high FN → low recall),
- hallucinate entities (high FP → low precision),
- or get boundaries/types wrong (not counted as TP).

**In this notebook, the most important score is entity-level F1.**  
Token accuracy is mainly a secondary diagnostic.

## 3) What happens in each stage?

### Part 1 — Baseline: BiLSTM Tagger (sequential encoder)
Start with a strong classical baseline:
- a contextual encoder that reads tokens left-to-right and right-to-left
- predicts a tag per token

**What you learn:** a sequential model captures order naturally, but has limited parallelism and long-range interaction costs.

---

### Part 2 — Add Self-Attention on top of the baseline
You keep the baseline encoder and **add a single self-attention mechanism** after it.

**What you learn:** attention can mix global context and sometimes improve tagging quality; attention maps are also interpretable.

---

### Part 3 — Attention-only (remove the sequential encoder)
Now you remove the sequential component and try a model that relies on **self-attention only**, without any explicit notion of position.

**What you learn:** self-attention alone can mix token content, but **does not inherently know order**, so performance can degrade on tasks where order matters.

---

### Part 4 — Fix the order problem by adding Positional Encoding
You keep the attention-only design, but now inject **positional information** into the token representations.

**What you learn:** adding position makes attention usable for order-sensitive tasks, and you can directly compare Part 3 vs Part 4 as an ablation.

---

### Part 5 — Upgrade to Multi-Head Attention
You replace single attention with **multi-head attention**, so the model can represent different relation patterns in parallel.

**What you learn:** multiple heads can specialize (local vs long-range, boundaries vs separators, etc.), and you can visualize different heads.

---

### Part 6 — Build a deeper Transformer encoder (“mini-BERT brick”) + depth ablation
You turn the attention block into a standard encoder layer and stack layers to study **depth vs performance**.

**What you learn:** depth can help, but not always; optimization and overfitting constraints appear; ablations teach you what matters empirically.

---

### Part 7 — BERT-style inputs and embedding recipe
You transition from “generic sequence tagging inputs” to **BERT-style inputs**:
- special tokens ([CLS], [SEP], [MASK])
- token type (segment) IDs
- position IDs
- embedding composition used by BERT-like models

**What you learn:** how the input representation design supports pretraining and downstream fine-tuning.

---

### Part 8 — Pretraining objective: MLM (Masked Language Modeling)
You pretrain your encoder on **MLM**, where only a subset of tokens are predicted (masked positions).

**What you learn:** pretraining teaches general language/statistical structure without labels, and the masking strategy matters.

---

### Part 9 — Pretraining objective: NSP + joint MLM+NSP
You extend pretraining with a second objective: **Next Sentence Prediction (NSP)**, and train with MLM+NSP jointly.

**What you learn:** multi-objective pretraining, what NSP is trying to encourage, and why its benefit can be mixed.

---

### Part 10 — Fine-tune TinyBERT for NER (scratch vs pretrained initializations)
You fine-tune on NER and compare three conditions under a similar fine-tuning budget:
- random initialization (scratch)
- MLM-pretrained initialization
- MLM+NSP-pretrained initialization

**What you learn:** how pretraining affects downstream learning speed, stability, and final entity F1.

---

### Part 11 — Fine-tune a foundation model (pretrained DistilBERT/BERT)
You fine-tune a real pretrained Transformer model using a modern library pipeline.

**What you learn:** what large-scale pretraining buys you in practice (data efficiency + stronger performance), and how subword tokenization affects label alignment conceptually.

---

### Part 12 — Analysis Lab: attention interpretability + computational cost
You treat the final part like a mini research lab:
- visualize attention maps from a fine-tuned pretrained model
- inspect predictions vs gold labels
- measure how runtime scales with sequence length
- build a final comparison table across all approaches

**What you learn:** interpretability signals (with caution), and the practical limitation of attention’s quadratic scaling.

## 4) The full “sequence chain” of the notebook (the big picture)

You can summarize the notebook progression as:

1. **Reliable NER evaluation + reproducible training loop**  
2. **Sequential baseline (BiLSTM)**  
3. **Add attention (global mixing) while keeping sequence awareness**  
4. **Remove sequence awareness → observe failure**  
5. **Restore order with positional encoding → recover performance**  
6. **Scale attention with multi-head + depth → stronger encoder**  
7. **Adopt BERT-style input/embedding recipe**  
8. **Pretrain with MLM (self-supervision)**  
9. **Optionally add NSP and compare**  
10. **Fine-tune and measure the value of pretraining**  
11. **Compare against a real pretrained foundation model**  
12. **Interpret + analyze efficiency and limitations**

This is a controlled story of **how modern Transformer encoders emerge from specific design decisions**.

## 5) What we expect you to practice while completing TODOs

As you fill in TODO sections, your focus should be:
- correctness and reproducibility (your code must rerun cleanly)
- fair comparisons (avoid changing multiple factors at once)
- careful evaluation (use entity F1 as the main score)
- interpretation (attention maps and failure cases are part of the learning)

Whenever you change something important, document it briefly in a Markdown cell near the change.

## 6) Grading criteria

Your grade in this notebook will be based on:

1. **Correct implementation and rerunnability**  
   Your solutions must be correct and the notebook should run end-to-end when re-executed.

2. **Reasonable training progression**  
   Training curves should show a sensible learning trend (loss decreasing, metrics stabilizing).

3. **Reasonable final performance per part**  
   You do *not* need the absolute best score, but your results should not be drastically below the expected ranges shown in each section output.

4. **Quality of plots and visualizations**  
   Curves and attention heatmaps must be clearly plotted, labeled, and interpretable.

5. **Correct and well-explained answers to Concept Checks**  
   Method questions and interpretation questions must be answered accurately and with clear reasoning.

## 7) Notes for students (important)

- You are allowed to modify any code, **but your notebook will be re-run by the grader**.  
  Write clean, stable code and avoid hidden dependencies.

- Try to keep model comparisons under **equal conditions** as much as possible (same data splits, same evaluation logic, similar budgets) so the comparisons are fair.

- Do **not** change the core assignment identity:
  - keep the task as **NER**
  - keep the selected dataset
  - do not replace the overall model families with unrelated architectures

- You have flexibility in details:
  - hyperparameters, number of epochs, optimization choices, regularization, etc.
  - but if you change something important, explain why (briefly) in Markdown.

- If a section gives unexpected results (low F1, overfitting, worse than a previous model, etc.),
  you can still earn partial or even full credit for that section **if you diagnose the reason correctly and explain it clearly**.

---

# Install Dependencies

In [3]:
!pip -q install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [4]:
pip install -U "transformers<5" "pydantic<2.12"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.5 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.3
    Uninstalling pydantic-2.12.3:
      Successfully uninstalled pydantic-2.12.3
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingfac

In [5]:
!pip -q install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 45.4 MB/s eta 0:00:00


# Part 0 — Setup, Data Pipeline, and Evaluation Harness

In this part, **we build the experimental foundation** that every later section relies on. The main goal is not modeling yet—it is making sure we can **load the dataset**, **represent sequences with padding/masks**, and **evaluate NER correctly** (especially with **entity-level F1**, not just token accuracy).

Most later parts reuse the same interfaces designed here (dataset → dataloader → model → loss → metrics). So we should write Part 0 as if we are preparing a small, reliable research framework.

## What we should have by the end of Part 0

- A reproducible setup (seeds + device)
- A dataset object that returns token sequences and NER tags
- A collate function that pads inputs and labels correctly  
  (and does **not** let padded tokens influence loss/metrics)
- A label mapping (`label2id`, `id2label`) that is consistent and checked
- A reliable metric function that reports:
  - token accuracy (sanity only)
  - entity-level precision / recall / F1 (main score)

## Imports and Logging Hygiene

We import all libraries used throughout the notebook and suppress noisy warnings/logs so outputs stay readable and focused on the experiments.

In [6]:
import os
import warnings

# --- Reduce noisy HuggingFace logs ---
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()  # only show errors

# --- Reduce a common PyTorch DataParallel warning ---
warnings.filterwarnings(
    "ignore",
    message=r"Was asked to gather along dimension 0, but all input tensors were scalars.*",
    category=UserWarning,
)

# --- Standard library / utilities ---
from dataclasses import dataclass
from typing import List, Dict, Tuple
import random
import math
import time
import inspect
from collections import Counter

# --- Numerics / data ---
import numpy as np
import pandas as pd

# --- PyTorch ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# --- Datasets + evaluation ---
from datasets import load_dataset, DatasetDict
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from dataclasses import dataclass

# --- Visualization / progress ---
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Transformers (later parts use these) ---
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

## Configuration (CFG)

Centralizes the key hyperparameters and conventions (seed, batch size, vocabulary limits, padding/ignore rules) so experiments stay consistent and easy to modify.

In [7]:
@dataclass
class CFG:
    # Reproducibility / dataloading
    seed: int = 34
    batch_size: int = 32
    num_workers: int = 2

    # Vocabulary (for from-scratch / word-level baselines)
    max_vocab_size: int = 50_000
    min_freq: int = 2
    pad_token: str = "<pad>"
    unk_token: str = "<unk>"

    # Label padding rule for masked loss/metrics
    ignore_index: int = -100

cfg = CFG()
cfg



CFG(seed=34, batch_size=32, num_workers=2, max_vocab_size=50000, min_freq=2, pad_token='<pad>', unk_token='<unk>', ignore_index=-100)

## Reproducibility (Seeding) and Device Selection

Set random seeds and determinism settings so runs are comparable, then select the compute device (CPU/GPU) used throughout the notebook.

In [8]:
# TODO:
# Implement a seeding utility that makes experiments reproducible.
#
# Requirements:
# - Seed Python's `random`
# - Seed NumPy
# - Seed PyTorch (CPU) and CUDA (if available)
# - (Optional but recommended) set deterministic flags for cuDNN
#
# Notes:
# - Full determinism can be slower.
# - The goal is that re-running the notebook yields similar curves/metrics.

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# TODO: call your seeding function using cfg.seed
# seed_everything(cfg.seed)

# TODO: define the global device used by the notebook
# device = ...

seed_everything(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# TODO: print the chosen device for visibility
# print("Device:", device)

Device: cuda


## Load the OntoNotes5 NER Dataset (HuggingFace)

We load the train/validation/test splits and print basic sanity checks (sizes + available columns) to verify the dataset pipeline is correct.

In [9]:
BASE = "https://huggingface.co/datasets/tner/ontonotes5/resolve/refs/convert/parquet/ontonotes5"

data_files = {
    "train": f"{BASE}/train/0000.parquet",
    "validation": f"{BASE}/validation/0000.parquet",
    "test": f"{BASE}/test/0000.parquet",
}

ds = load_dataset("parquet", data_files=data_files)

print(ds)
print("Columns:", ds["train"].column_names)
print("Train size:", len(ds["train"]))
print("Valid size:", len(ds["validation"]))
print("Test size :", len(ds["test"]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ontonotes5/train/0000.parquet:   0%|          | 0.00/3.68M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/515k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/517k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8262
    })
})
Columns: ['tokens', 'tags']
Train size: 59924
Valid size: 8528
Test size : 8262


## Normalize the Dataset Schema (Ensure We Have `ner_tags`)

Different NER exports sometimes use different column names for labels. This cell standardizes the label column name to `ner_tags` so the rest of the notebook can stay consistent.

In [10]:
# Part 0 — Ensure the dataset has a consistent NER label column name: `ner_tags`

print("Columns:", ds["train"].column_names)

def ensure_ner_tags(ds, preferred: str = "ner_tags"):
    cols = ds["train"].column_names
    if preferred in cols:
        return ds

    # Common alternatives in parquet exports / NER datasets
    candidates = ["tags", "ner", "labels", "bio_tags", "ner_tag", "ner_labels"]
    for cand in candidates:
        if cand in cols:
            print(f"Renaming '{cand}' -> '{preferred}'")
            return ds.rename_column(cand, preferred)

    raise KeyError(f"Could not find NER label column. Available columns: {cols}")

ds = ensure_ner_tags(ds, preferred="ner_tags")
print("After normalize:", ds["train"].column_names)

Columns: ['tokens', 'tags']
Renaming 'tags' -> 'ner_tags'
After normalize: ['tokens', 'ner_tags']


## Define the Fixed OntoNotes5 Label Space (and Sanity-Check)

We will use the official OntoNotes5 BIO label mapping so that decoding, metrics, and model heads all agree on the exact label IDs. We also sanity-check that the dataset does not contain unknown tag IDs.

In [11]:
ONTONOTES_LABEL2ID = {
    "O": 0,
    "B-CARDINAL": 1,
    "B-DATE": 2,
    "I-DATE": 3,
    "B-PERSON": 4,
    "I-PERSON": 5,
    "B-NORP": 6,
    "B-GPE": 7,
    "I-GPE": 8,
    "B-LAW": 9,
    "I-LAW": 10,
    "B-ORG": 11,
    "I-ORG": 12,
    "B-PERCENT": 13,
    "I-PERCENT": 14,
    "B-ORDINAL": 15,
    "B-MONEY": 16,
    "I-MONEY": 17,
    "B-WORK_OF_ART": 18,
    "I-WORK_OF_ART": 19,
    "B-FAC": 20,
    "B-TIME": 21,
    "I-CARDINAL": 22,
    "B-LOC": 23,
    "B-QUANTITY": 24,
    "I-QUANTITY": 25,
    "I-NORP": 26,
    "I-LOC": 27,
    "B-PRODUCT": 28,
    "I-TIME": 29,
    "B-EVENT": 30,
    "I-EVENT": 31,
    "I-FAC": 32,
    "B-LANGUAGE": 33,
    "I-PRODUCT": 34,
    "I-ORDINAL": 35,
    "I-LANGUAGE": 36,
}

# Reverse mapping (id -> label string)
id2label = {v: k for k, v in ONTONOTES_LABEL2ID.items()}
label2id = ONTONOTES_LABEL2ID

max_id = max(id2label)
label_names = [id2label[i] for i in range(max_id + 1)]  # index == label_id
num_labels = len(label_names)

print("Num labels:", num_labels)
print("Max id:", max_id)
print("First 10:", label_names[:10])

# Sanity-check: dataset should not contain label IDs outside this mapping
all_ids = set()
for split in ["train", "validation", "test"]:
    for seq in ds[split]["ner_tags"]:
        all_ids.update(seq)

missing = sorted([i for i in all_ids if i not in id2label])

print("All dataset label ids are covered")

Num labels: 37
Max id: 36
First 10: ['O', 'B-CARDINAL', 'B-DATE', 'I-DATE', 'B-PERSON', 'I-PERSON', 'B-NORP', 'B-GPE', 'I-GPE', 'B-LAW']
All dataset label ids are covered


## Robust Label-Space Construction

Some dataset formats preserve label-name metadata, while others only store integer IDs. This cell builds a reliable `label_names / label2id / id2label` setup by first using metadata if available, and otherwise inferring what it can from the data—while remaining compatible with the fixed OntoNotes mapping.

In [12]:
def infer_label_ids_from_split(split, col: str = "ner_tags", max_scan: int | None = None):
    """
    Infer label information when metadata is missing.

    - If tags are strings -> return sorted unique strings (with "O" first if present).
    - If tags are ints    -> return sorted unique int IDs (mapping to names handled elsewhere).
    """
    n = len(split) if max_scan is None else min(len(split), max_scan)

    first_seq = split[0][col]
    if len(first_seq) == 0:
        raise ValueError("Empty tag sequence encountered.")

    # Case 1: tags are strings already
    if isinstance(first_seq[0], str):
        s = set()
        for i in range(n):
            s.update(split[i][col])
        names = sorted(s)
        if "O" in names:
            names = ["O"] + [x for x in names if x != "O"]
        return names

    # Case 2: tags are ints (metadata missing)
    if isinstance(first_seq[0], (int, np.integer)):
        s = set()
        for i in range(n):
            s.update(split[i][col])
        return sorted(int(x) for x in s)

    raise TypeError(f"Unknown tag type: {type(first_seq[0])}")


# 1) Try to read metadata (works when Sequence(ClassLabel) is preserved)
label_names = None
feat = ds["train"].features.get("ner_tags", None)

if feat is not None and hasattr(feat, "feature") and hasattr(feat.feature, "names"):
    label_names = list(feat.feature.names)

# 2) Otherwise infer from data
if label_names is None:
    inferred = infer_label_ids_from_split(ds["train"], col="ner_tags", max_scan=20_000)

    # If inferred is ints -> use the official OntoNotes mapping
    if len(inferred) > 0 and isinstance(inferred[0], int):
        label2id = ONTONOTES_LABEL2ID
        id2label = {v: k for k, v in label2id.items()}
        label_names = [id2label[i] for i in range(max(id2label) + 1)]
    else:
        # inferred is already list[str]
        label_names = inferred
        label2id = {name: i for i, name in enumerate(label_names)}
        id2label = {i: name for name, i in label2id.items()}

num_labels = len(label_names)
print("Num labels:", num_labels)
print("First labels:", label_names[:20])

Num labels: 37
First labels: ['O', 'B-CARDINAL', 'B-DATE', 'I-DATE', 'B-PERSON', 'I-PERSON', 'B-NORP', 'B-GPE', 'I-GPE', 'B-LAW', 'I-LAW', 'B-ORG', 'I-ORG', 'B-PERCENT', 'I-PERCENT', 'B-ORDINAL', 'B-MONEY', 'I-MONEY', 'B-WORK_OF_ART', 'I-WORK_OF_ART']


## Quick Data Sanity Check (Tokens + Decoded Tags)

Before building models, I always inspect a couple of raw examples. This cell prints tokens alongside their NER tags (decoded to strings if needed) so you can confirm the dataset is loaded correctly and the label mapping behaves as expected.

In [13]:
def show_sample(i: int = 0, split: str = "train") -> None:
    ex = ds[split][i]
    tokens = ex["tokens"]
    tags = ex["ner_tags"]

    # If tags are ints, decode them; if already strings, keep them.
    if len(tags) > 0 and isinstance(tags[0], int):
        tags = [id2label[int(t)] for t in tags]

    print("TOKENS:", tokens)
    print("TAGS  :", tags)


show_sample(34, "train")
print("-" * 80)
show_sample(43, "train")

TOKENS: ['Employers', 'must', 'deposit', 'withholding', 'taxes', 'exceeding', '$', '3,000', 'within', 'three', 'days', 'after', 'payroll', '--', 'or', 'pay', 'stiff', 'penalties', '--', 'and', 'that', "'s", 'a', 'big', 'problem', 'for', 'small', 'businesses', '.']
TAGS  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MONEY', 'O', 'B-DATE', 'I-DATE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
--------------------------------------------------------------------------------
TOKENS: ['Alice', 'Fixx', ',', 'who', 'runs', 'her', 'own', 'public', '-', 'relations', 'concern', 'in', 'New', 'York', ',', 'says', 'she', 'has', 'had', 'to', 'overhaul', 'her', 'pension', 'and', 'profit', '-', 'sharing', 'plans', 'three', 'times', 'in', 'the', 'past', 'three', 'years', '.']
TAGS  : ['B-PERSON', 'I-PERSON', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-GPE', 'I-GPE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-CARDINAL', 'O', 

## Build a Word-Level Vocabulary (for the from-scratch baselines)

In the early parts of this notebook, we will train non-Transformer models on **word IDs**.  
This cell builds a vocabulary from the training tokens and creates both mappings:

- `stoi`: string → integer ID  
- `itos`: integer ID → string  

Your implementation choices here (normalization, frequency thresholds, vocab cap, special tokens) will affect downstream results, so document what you decide.

In [14]:
# TODO: Build a word-level vocabulary from the training split.
#
# Requirements:
# - Input: a list of tokenized sentences (List[List[str]]) from ds["train"]["tokens"]
# - Output:
#   1) stoi: Dict[str, int] mapping tokens -> ids
#   2) itos: Dict[int, str] mapping ids -> tokens
#
# Constraints / hints:
# - Reserve IDs for at least:
#     cfg.pad_token (PAD) and cfg.unk_token (UNK)
# - Use cfg.max_vocab_size and cfg.min_freq to control vocabulary growth
# - Decide whether you want any normalization (optional):
#     lowercasing, digit masking, punctuation handling, etc.
# - Make sure your vocabulary is built ONLY from the training data (no leakage).
#
# After building:
# - Print vocab size and verify PAD/UNK ids exist.
# - Optionally print a few example ids to sanity-check the mapping.

# train_tokens = ds["train"]["tokens"]
# stoi, itos = ...
# print("Vocab size:", ...)
# print("PAD id:", ..., "UNK id:", ...)
# print("Top few ids:", ...)

from collections import Counter
from typing import List, Dict, Tuple

def build_vocab(
    tokenized_sentences: List[List[str]],
    cfg: CFG,
) -> Tuple[Dict[str, int], Dict[int, str]]:
    counter = Counter()

    for sent in tokenized_sentences:
        # Optional normalization
        sent = [t.lower() for t in sent]
        counter.update(sent)

    # Special tokens first
    stoi = {
        cfg.pad_token: 0,
        cfg.unk_token: 1,
    }

    # Filter by min_freq and max_vocab_size
    vocab_tokens = [
        tok for tok, freq in counter.most_common(cfg.max_vocab_size)
        if freq >= cfg.min_freq
    ]

    for tok in vocab_tokens:
        if tok not in stoi:
            stoi[tok] = len(stoi)

    itos = {i: s for s, i in stoi.items()}
    return stoi, itos

train_tokens = ds["train"]["tokens"]
stoi, itos = build_vocab(train_tokens, cfg)

print("Vocab size:", len(stoi))
print("PAD id:", stoi[cfg.pad_token], "UNK id:", stoi[cfg.unk_token])
print("Top few ids:", list(stoi.items())[:10])


Vocab size: 21354
PAD id: 0 UNK id: 1
Top few ids: [('<pad>', 0), ('<unk>', 1), ('the', 2), (',', 3), ('.', 4), ('of', 5), ('to', 6), ('and', 7), ('a', 8), ('in', 9)]


## Wrap the NER Data as a PyTorch Dataset (word-level IDs)

This cell defines a simple `Dataset` that converts each example into `(input_ids, labels)` for the from-scratch baselines, while still keeping the raw tokens for debugging and qualitative inspection.

In [15]:
class NERWordDataset(Dataset):
    """
    Word-level NER dataset:
    - Converts token strings into integer IDs using `stoi`
    - Returns aligned NER label IDs from the dataset
    - Keeps raw tokens for debugging/inspection
    """
    def __init__(self, hf_split, stoi: Dict[str, int]):
        self.split = hf_split
        self.stoi = stoi
        self.unk_id = stoi[cfg.unk_token]

    def __len__(self) -> int:
        return len(self.split)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ex = self.split[idx]
        tokens = ex["tokens"]
        labels = ex["ner_tags"]

        input_ids = [self.stoi.get(tok, self.unk_id) for tok in tokens]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "tokens": tokens,  # keep raw tokens for debugging
        }


train_ds = NERWordDataset(ds["train"], stoi)
valid_ds = NERWordDataset(ds["validation"], stoi)
test_ds  = NERWordDataset(ds["test"], stoi)

print(
    f"train length: {len(train_ds)}\n"
    f"validation length: {len(valid_ds)}\n"
    f"test length: {len(test_ds)}"
)

train length: 59924
validation length: 8528
test length: 8262


## Collate Function + DataLoaders (padding, masks, and label ignoring)

In this cell you will implement batching for variable-length token sequences: pad `input_ids`, build an `attention_mask`, and pad `labels` using an ignore index so padded positions do not contribute to loss or metrics. Then you will plug this collate function into PyTorch `DataLoader`s.

In [16]:
# TODO (students):
# Implement a collate function for variable-length NER batches.
#
# Requirements:
# 1) Pad `input_ids` to the max length in the batch using the PAD token id.
# 2) Build an `attention_mask` (1 for real tokens, 0 for padding).
# 3) Pad `labels` to the same max length using `cfg.ignore_index`
#    so that padded positions are ignored by the loss and by metrics.
# 4) Optionally return:
#    - the original tokens (for debugging)
#    - `lengths` (useful for RNN packing or analysis)
#
# Return a dict with at least:
#   {"input_ids": ..., "attention_mask": ..., "labels": ...}
#
# Tip: Keep tensor dtypes consistent (usually long for ids/masks/labels).

def ner_collate_fn(batch):
    batch_input_ids = [x["input_ids"] for x in batch]
    batch_labels = [x["labels"] for x in batch]

    lengths = torch.tensor([len(x) for x in batch_input_ids], dtype=torch.long)
    max_len = max(lengths).item()

    pad_id = stoi[cfg.pad_token]

    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels = torch.full((len(batch), max_len), cfg.ignore_index, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

    for i, (ids, labs) in enumerate(zip(batch_input_ids, batch_labels)):
        seq_len = len(ids)
        input_ids[i, :seq_len] = ids
        labels[i, :seq_len] = labs
        attention_mask[i, :seq_len] = 1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "lengths": lengths,
        "attention_mask": attention_mask,
    }



# TODO (students):
# Create train/valid/test DataLoaders using your collate function.
# Feel free to choose num_workers, pin_memory, shuffle, etc.
#
# Make sure the loader outputs match what your training loop expects.

# cfg.num_workers = ???  # set as appropriate for your environment

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=ner_collate_fn,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=ner_collate_fn,
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=ner_collate_fn,
)

# TODO (students):
# Take one batch and print shapes/types to sanity-check padding + masks.
# batch = next(iter(train_loader))
# {k: (v.shape if torch.is_tensor(v) else type(v)) for k, v in batch.items()}

batch = next(iter(train_loader))
{k: (v.shape, v.dtype) for k, v in batch.items()}


{'input_ids': (torch.Size([32, 68]), torch.int64),
 'labels': (torch.Size([32, 68]), torch.int64),
 'lengths': (torch.Size([32]), torch.int64),
 'attention_mask': (torch.Size([32, 68]), torch.int64)}

## Metrics Utilities (masking, decoding, and seqeval formatting)

In this cell you will implement the metric utilities used throughout the notebook. Your goal is to (1) correctly ignore padded positions, (2) convert label IDs into BIO tag strings, and (3) compute both token accuracy and entity-level precision/recall/F1 using `seqeval`.

In [17]:
# TODO (students):
# Implement helpers to compute NER metrics.
#
# Core requirements:
# 1) Convert (padded) label/pred id tensors into list-of-list BIO tag strings,
#    and IGNORE positions where the gold label equals cfg.ignore_index.
# 2) Compute token-level accuracy over non-ignored positions.
# 3) Compute entity-level precision / recall / F1 using seqeval.
#
# Notes:
# - Your output format matters: seqeval expects List[List[str]] for y_true/y_pred.
# - Keep the masking policy consistent across the entire notebook.
# - Optionally support a `verbose=True` mode that prints seqeval's classification report.

def ids_to_tags(
    label_ids: torch.Tensor,
    pred_ids: torch.Tensor,
) -> Tuple[List[List[str]], List[List[str]]]:

    y_true, y_pred = [], []

    for gold_seq, pred_seq in zip(label_ids.tolist(), pred_ids.tolist()):
        true_tags, pred_tags = [], []

        for g, p in zip(gold_seq, pred_seq):
            if g == cfg.ignore_index:
                continue
            true_tags.append(id2label[g])
            pred_tags.append(id2label[p])

        y_true.append(true_tags)
        y_pred.append(pred_tags)

    return y_true, y_pred


@torch.no_grad()
def compute_token_accuracy(label_ids: torch.Tensor, pred_ids: torch.Tensor) -> float:
    mask = label_ids != cfg.ignore_index
    correct = (label_ids == pred_ids) & mask
    return correct.sum().item() / mask.sum().item()

@torch.no_grad()
def compute_ner_metrics(
    label_ids: torch.Tensor,
    pred_ids: torch.Tensor,
    verbose: bool = False,
) -> Dict[str, float]:

    token_acc = compute_token_accuracy(label_ids, pred_ids)

    y_true, y_pred = ids_to_tags(label_ids, pred_ids)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    if verbose:
        print(classification_report(y_true, y_pred))

    return {
        "token_acc": token_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## Sanity-check baseline: label distribution + random predictor

In this cell you will build a quick sanity baseline to validate your data + metrics pipeline. The idea is to compute a simple label distribution from the training data and then generate random tag predictions for a batch. This baseline should score poorly in entity-level F1, but it helps confirm that your metric code runs end-to-end and that the label imbalance is real.


In [18]:
# TODO (students):
# 1) Compute the empirical distribution of NER labels on the training split.
#    (This is typically extremely imbalanced, with "O" dominating.)
# 2) Print a small summary of the most frequent labels.
# 3) Create a trivial baseline predictor (e.g., random sampling from the empirical distribution)
#    and evaluate it using your compute_ner_metrics() implementation.
#
# Notes:
# - Make sure you evaluate on a *real batch* from your DataLoader.
# - Your predictions must have the same shape as the padded label tensor.
# - This is a diagnostic step: don't overthink performance here.

def label_distribution(hf_split) -> np.ndarray:
    all_labels = []

    for example in hf_split:
        all_labels.extend(example["ner_tags"])

    counts = np.bincount(all_labels)
    probs = counts / counts.sum()
    return probs




# TODO: compute probs from ds["train"] and print the most common labels

# TODO: take one batch from train_loader, generate trivial predictions, and compute metrics
# metrics = compute_ner_metrics(label_ids, pred_ids, verbose=False)
# metrics
probs = label_distribution(ds["train"])

top_ids = np.argsort(probs)[::-1][:10]
for i in top_ids:
    print(f"{id2label[i]:10s}  {probs[i]:.4f}")



O           0.8628
I-ORG       0.0168
B-PERSON    0.0142
B-GPE       0.0142
I-DATE      0.0122
B-ORG       0.0118
I-PERSON    0.0102
B-DATE      0.0100
B-CARDINAL  0.0068
B-NORP      0.0063


In [19]:
print(ds["train"].column_names)


['tokens', 'ner_tags']


## Concept Checks (answer in Markdown, no code)

1. **Why do we use entity-level F1 as the main score for NER instead of token accuracy?**  
   Explain using the label imbalance and what “getting an entity correct” actually means.



Answer:NER labels are highly imbalanced (“O” dominates).
A model can get high token accuracy by predicting “O” everywhere while finding zero entities.Entity-level F1 only counts a prediction as correct if: the entity type, the start, the end, are all correct. that matches what "getting an entityright" actuall means.

2. **What is the purpose of `ignore_index` in token classification training?**  
   In our own words, what goes wrong if padded positions are not ignored?


Answer:Padding tokens are not real words.

If we don’t ignore them:the model learns from fake labels,loss and metrics,come inflated,shorter sentences get unfair extra weight ignore_index ensures only real tokens affect training and evaluation.

3. **Padding and masking:**  
   We will create both a padded `labels` tensor and an `attention_mask`.  
   - What does each one control?  
   - Where do we expect each one to be used later (loss vs model vs metrics)?



Answer:labels with ignore_index
Controls what counts for loss and metrics.

attention_mask
Controls what the model attends to during the forward pass.

Loss/metrics → labels
Model computation → attention_mask

4. **BIO-style tagging:**  
   What does it mean for a model to be “wrong” even if it predicts many correct token labels?  
   Hint: think about entity boundaries and types.



Answer:NER is about correct spans, not just tokens.

A model is wrong if it:breaks entity boundaries,predicts the wrong entity type,
produces invalid BIO sequences Even many correct tokens ≠ a correct entity.

# Part 1 — Baseline: BiLSTM Tagger (Sequence Encoder)

In this part we build our **strong non-Transformer baseline** for NER: a **BiLSTM tagger**.

Why start here? Because sequence labeling is one of the most natural settings for recurrent models:
- A BiLSTM reads the sentence **left-to-right and right-to-left**, so each token representation can use both past and future context.
- We predict a tag **for every token** (token-level classification), while carefully ignoring padded positions during training and evaluation.

This baseline will become our reference point for the rest of the notebook:
- Later, we will add attention on top of it.
- Then we will remove recurrence and try attention-only models.
- Finally, we will compare against pretrained Transformer encoders.

**Important:** In this part we focus on building a *clean experiment loop*: correct masking, fair metrics, stable training, and a clear report of both token accuracy and entity-level F1.

## What we will do in Part 1
By the end of this part, we will have:
1. A BiLSTM model that produces logits of shape `(B, T, num_labels)`.
2. A loss function that properly ignores padded labels using `ignore_index`.
3. A training loop with reasonable optimization defaults (and gradient clipping).
4. An evaluation loop that reports **loss + token accuracy + entity-level P/R/F1**.
5. A few qualitative examples (tokens + gold tags + predicted tags) to see failure modes.

## DataLoaders for Validation and Test
We instantiate deterministic (non-shuffled) DataLoaders for validation and test so that evaluation is stable and comparable across runs and across later model variants.

In [20]:
# Re-create loaders explicitly (train_loader is already defined in Part 0)
valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=ner_collate_fn,
)

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=ner_collate_fn,
)

print("Batches:", len(train_loader), len(valid_loader), len(test_loader))

Batches: 1873 267 259


## Baseline Model: BiLSTM Tagger (Token-Level Classifier)
We implement our first strong non-Transformer baseline: a bidirectional LSTM encoder followed by a per-token classifier. The key goal here is to produce contextual representations while handling padding correctly so that evaluation is fair and later comparisons remain meaningful.

In [21]:
# TODO: Implement a BiLSTM-based token classifier for NER.
#
# Requirements (keep the interface compatible with later parts):
# - Use an embedding layer with a padding index (pad_id).
# - Use a bidirectional LSTM encoder that reads the sequence and outputs contextual token representations.
# - Ensure padded tokens do not corrupt sequence modeling:
#   * Either use pack_padded_sequence / pad_packed_sequence, OR an equivalent masking strategy.
# - Apply dropout (where appropriate) and a linear classifier to produce per-token logits.
#
# The forward() method should accept:
#   input_ids: LongTensor of shape [B, T]
#   lengths:   LongTensor of shape [B] containing true lengths (no padding)
# and return:
#   logits: FloatTensor of shape [B, T, num_labels]
#
# Notes:
# - You may choose the embedding size, hidden size, number of layers, and dropout rate.
# - Keep the output shape exactly [B, T, num_labels] so our evaluation code works unchanged.

class BiLSTMTagger(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        emb_dim: int = 128,
        hidden_dim: int = 256,
        num_layers: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, emb_dim, padding_idx=pad_id
        )

        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        emb = self.embedding(input_ids)  # [B, T, E]

        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True)

        out = self.dropout(out)
        logits = self.classifier(out)  # [B, T, C]
        return logits


# TODO: Instantiate the model and move it to the chosen device.
# - vocab_size should match our word-level vocabulary.
# - pad_id should match the PAD token id in stoi.
vocab_size = len(stoi)
pad_id = stoi[cfg.pad_token]

# model = BiLSTMTagger(...).to(device)
# print(model)
num_labels = len(id2label)
model = BiLSTMTagger(
    vocab_size=len(stoi),
    num_labels=num_labels,
    pad_id=stoi[cfg.pad_token],
).to(device)

print(model)


BiLSTMTagger(
  (embedding): Embedding(21354, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=256, out_features=37, bias=True)
)


## Sanity Check: Forward Pass Shapes
We run a single batch through the model to verify that the forward pass works and the logits have the expected shape `[B, T, num_labels]`. This quick check saves a lot of debugging time later.

In [22]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
lengths = batch["lengths"].to(device)

logits = model(input_ids, lengths)

print("input_ids:", input_ids.shape)
print("logits   :", logits.shape, "(should be [B, T, num_labels])")

input_ids: torch.Size([32, 106])
logits   : torch.Size([32, 106, 37]) (should be [B, T, num_labels])


## Training & Evaluation Utilities (TODO)
We now define the core utilities that we will reuse throughout the notebook: **masked loss computation** (ignoring padded tokens), an **evaluation pass** that reports both token accuracy and entity-level P/R/F1, and a **single-epoch training loop** with optional gradient clipping. Keep the design clean and reusable—later parts will plug into the same API.

In [23]:
# TODO: Training + evaluation utilities (core loop)
#
# In this section, we need a minimal training/evaluation toolkit for token-level NER:
#
# 1) Loss computation with padding ignored
#    - logits: [B, T, C]
#    - labels: [B, T]
#    - Use cfg.ignore_index to ignore padded label positions.
#    - Decide whether to compute:
#        (a) sum loss over real tokens + token count, and/or
#        (b) mean loss per real token
#
# 2) Evaluation function (no gradients)
#    - Must return at least:
#        loss, token accuracy, entity-level precision/recall/F1
#    - Important: entity-level metrics must ignore padded tokens
#      and must be computed on tag strings (via ids_to_tags).
#
# 3) One-epoch training function
#    - Typical steps: zero_grad -> forward -> loss -> backward -> (optional) grad clip -> optimizer.step
#    - Track and return mean loss per real token (or an equivalent fair normalization).
#
# Design notes:
# - Keep signatures simple and reusable across later parts.
# - Be careful: averaging loss across batches is NOT the same as averaging over tokens
#   when sequence lengths differ. Prefer token-normalized loss.

criterion = nn.CrossEntropyLoss(ignore_index=cfg.ignore_index, reduction="sum")

def loss_sum_and_tokens(logits: torch.Tensor, labels: torch.Tensor):
    B, T, C = logits.shape
    loss = criterion(
        logits.view(B * T, C),
        labels.view(B * T),
    )
    num_tokens = (labels != cfg.ignore_index).sum().item()
    return loss, num_tokens

def loss_mean(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    loss_sum, num_tokens = loss_sum_and_tokens(logits, labels)
    return loss_sum / num_tokens

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0
    token_correct = 0

    y_true, y_pred = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)

        logits = model(input_ids, lengths)
        loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)

        total_loss += loss_sum.item()
        total_tokens += n_tokens

        preds = logits.argmax(dim=-1)

        # token accuracy (mask padding)
        mask = labels != cfg.ignore_index
        token_correct += ((preds == labels) & mask).sum().item()

        # entity metrics
        batch_true, batch_pred = ids_to_tags(labels.cpu(), preds.cpu())
        y_true.extend(batch_true)
        y_pred.extend(batch_pred)

    return {
        "loss": total_loss / total_tokens,
        "token_acc": token_correct / total_tokens,
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }




def train_one_epoch(model, loader, optimizer, grad_clip: float = 1.0):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)

        logits = model(input_ids, lengths)
        loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)

        loss = loss_sum / n_tokens
        loss.backward()

        if grad_clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        total_loss += loss_sum.item()
        total_tokens += n_tokens

    return total_loss / total_tokens



## Training Loop with Early Stopping (TODO)
We now run the full training procedure for the BiLSTM baseline: define an optimizer, iterate over epochs, evaluate on the validation set each epoch, and apply **early stopping based on entity-level F1**. We also store a compact training history for later plotting and reload the best checkpoint before final testing.

In [ ]:
# TODO: Training driver (optimizer + epochs + early stopping + checkpointing)
#
# Goal:
# - Create an optimizer (and optionally a scheduler)
# - Train for multiple epochs using train_one_epoch(...)
# - Evaluate each epoch using evaluate(...)
# - Track history for plots (loss + metrics)
# - Implement early stopping (recommended: monitor entity-level F1 on validation)
# - Restore the best model weights before moving on
#
# Requirements / suggestions:
# - Use AdamW or another optimizer of your choice.
# - Keep a "history" dict for plotting later.
# - Save "best_state" as a CPU copy of the state_dict for reliability.
# - Use a small tolerance when comparing floats (optional).
# - Early stopping patience should be configurable.

# --- optimizer / schedule ---
# optimizer = ...
# (optional) scheduler = ...

# --- hyperparameters ---
# EPOCHS = ...
# patience = ...

# --- bookkeeping ---
# best_f1 = ...
# best_state = ...
# bad_epochs = ...
# history = {"train_loss": [], "val_loss": [], "val_token_acc": [], "val_f1": []}

# --- training loop ---
# for epoch in range(1, EPOCHS + 1):
#     train_loss = train_one_epoch(...)
#     val_metrics = evaluate(...)
#     update history
#     print a clean log line
#     early stopping check on val F1
#     update best_state when improved

# --- restore best model ---
# model.load_state_dict(best_state)
# print final best validation F1

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

EPOCHS = 20
patience = 3

best_f1 = -1.0
best_state = None
bad_epochs = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_token_acc": [],
    "val_f1": [],
}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_metrics = evaluate(model, valid_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_token_acc"].append(val_metrics["token_acc"])
    history["val_f1"].append(val_metrics["f1"])

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['token_acc']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f}"
    )

    if val_metrics["f1"] > best_f1 + 1e-4:
        best_f1 = val_metrics["f1"]
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print("Early stopping.")
            break

model.load_state_dict(best_state)
print("Best val F1:", best_f1)


Epoch 01 | train_loss=0.5188 | val_loss=0.3619 | val_acc=0.8999 | val_f1=0.2851
Epoch 02 | train_loss=0.3261 | val_loss=0.3161 | val_acc=0.9091 | val_f1=0.3648
Epoch 03 | train_loss=0.2880 | val_loss=0.2909 | val_acc=0.9151 | val_f1=0.4007
Epoch 04 | train_loss=0.2649 | val_loss=0.2780 | val_acc=0.9188 | val_f1=0.4211
Epoch 05 | train_loss=0.2488 | val_loss=0.2720 | val_acc=0.9200 | val_f1=0.4341
Epoch 06 | train_loss=0.2358 | val_loss=0.2641 | val_acc=0.9215 | val_f1=0.4470
Epoch 07 | train_loss=0.2249 | val_loss=0.2601 | val_acc=0.9231 | val_f1=0.4604
Epoch 08 | train_loss=0.2145 | val_loss=0.2594 | val_acc=0.9233 | val_f1=0.4688
Epoch 09 | train_loss=0.2052 | val_loss=0.2563 | val_acc=0.9250 | val_f1=0.4798
Epoch 10 | train_loss=0.1956 | val_loss=0.2588 | val_acc=0.9243 | val_f1=0.4716
Epoch 11 | train_loss=0.1874 | val_loss=0.2586 | val_acc=0.9244 | val_f1=0.4757
Epoch 12 | train_loss=0.1790 | val_loss=0.2594 | val_acc=0.9245 | val_f1=0.4824
Epoch 13 | train_loss=0.1709 | val_loss=

## Visualize Learning Curves (TODO)
We plot the main training curves to check whether optimization is behaving sensibly: losses should generally decrease and validation metrics (especially **entity-level F1**) should stabilize. These plots are a key diagnostic tool before we trust any final test results.

In [ ]:
# TODO: Plot training curves
#
# Goal:
# - Visualize the training dynamics using the "history" dictionary.
#
# Minimum plots to include:
# 1) train_loss vs val_loss across epochs
# 2) val_f1 vs val_token_acc across epochs
#
# Plotting tips:
# - Label axes and include a legend.
# - Keep plots readable (reasonable figure size, clear labels).
# - Optionally mark the best epoch (based on val_f1).

# Example structure (you can change it):
plt.figure()
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(history["val_f1"], label="val_F1")
plt.plot(history["val_token_acc"], label="val_token_acc")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.show()


## Inspect Token-Level Predictions (TODO)
Numbers (F1/precision/recall) tell us *how good* a model is overall, but not *how it fails*. Here we inspect a few sequences by hand to see typical error patterns (boundary mistakes, type confusions, and “O”-dominance effects) before moving on to more complex architectures.


In [ ]:
@torch.no_grad()
def inspect_predictions(model, loader, n_sentences: int = 3):
    model.eval()
    batch = next(iter(loader))

    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"]
    lengths = batch["lengths"]

    logits = model(input_ids, lengths.to(device))
    preds = logits.argmax(dim=-1).cpu()

    for i in range(n_sentences):
        print("-" * 40)
        for j in range(lengths[i]):
            tok = itos[input_ids[i, j].item()]
            gold = id2label[labels[i, j].item()]
            pred = id2label[preds[i, j].item()]
            print(f"{tok:15s} {gold:8s} {pred:8s}")

inspect_predictions(model, valid_loader)



## Test-Set Evaluation
Here we run a single final evaluation on the held-out test split and report loss, token accuracy, and entity-level precision/recall/F1. This is the number we will compare against later parts (under similar training and evaluation conditions).

In [2]:
# Final evaluation on the held-out test set
test_metrics = evaluate(model, test_loader)

print(
    f"TEST | loss={test_metrics['loss']:.4f} | "
    f"token_acc={test_metrics['token_acc']:.4f} | "
    f"P={test_metrics['precision']:.4f} | "
    f"R={test_metrics['recall']:.4f} | "
    f"F1={test_metrics['f1']:.4f}"
)

NameError: name 'evaluate' is not defined

## Concept Checks (answer in Markdown, no code)

### Q1 — Why BiLSTM for NER?
What does bidirectionality buy us in NER specifically?  
Explain what information the forward LSTM and backward LSTM each contribute.


Answer: NER decisions depend on both left and right context.

Forward LSTM: uses past context
(“Dr.” → next token is likely a PERSON)

Backward LSTM: uses future context
(“Inc.” after a word → likely an ORGANIZATION)

Bidirectionality lets the model decide a token’s label using the full sentence, which is crucial for correct entity boundaries and types.

### Q2 — Padding and “ignore_index”
Why must we ignore padded positions when computing the loss?  
What would go wrong (in optimization and metrics) if we treated padding like a real label?

Padding tokens are not real data.

If we don’t ignore them:

the model is trained on fake labels

loss gradients become noisy and misleading

token accuracy becomes inflated (padding is easy to predict)

Ignoring padding ensures only real tokens influence learning and evaluation.

### Q3 — Token accuracy vs. entity F1
We will print both token accuracy and entity-level F1.  
Give one concrete reason token accuracy can look “good” even when the model is poor at entities.

Token accuracy can look good because:

most tokens are "O"

predicting "O" everywhere yields high accuracy

but detects zero entities

Entity F1 correctly penalizes missing or broken entity spans.

# Part 2 — Add Self-Attention on Top of the BiLSTM

In Part 1, we used a strong sequential baseline (BiLSTM) that naturally encodes left-to-right and right-to-left context.  
In this part, we keep that **sequence-aware backbone**, but we add a **single self-attention block** on top of the BiLSTM outputs.

The point of this design is very specific:

- The **BiLSTM** already injects order and local sequential structure.
- The **self-attention layer** then provides a direct mechanism for **global token-to-token mixing**.
- Because we did *not* remove the BiLSTM, we can test whether attention helps *beyond* what a strong sequential encoder already learns.

Conceptually, we compute queries, keys, and values from the BiLSTM features $X$, then build attention weights and mix values:

$$
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

$$
A = \operatorname{softmax}\!\left(\frac{QK^{\mathsf T}}{\sqrt{d}} + M\right)
$$

$$
\text{out} = AV
$$

where $M$ is a masking term that prevents padding tokens from being attended to.

At the end, we still do token classification as before, and we evaluate with the **same exact metrics** (token accuracy + entity-level precision/recall/F1).  
That consistency makes Part 2 a clean ablation against Part 1.

### What we should pay attention to in this part

- **Does attention improve entity-level F1** compared to the BiLSTM baseline?
- **Does it help more for certain entity types** (longer spans, rarer types, or boundary-sensitive cases)?
- **How sensitive is it to masking and dropout**, since attention can exploit artifacts if masking is wrong?
- **What do the attention maps look like**, and do they show patterns that are at least plausibly meaningful?

## Single-Head Self-Attention (Core Mechanism)

In this cell, we implement **single-head self-attention**: each token computes a weighted mixture of all other tokens using the $QK^\top$ similarity matrix.  
The key detail is **padding-aware masking**: padded tokens must not be attended to, or the model will learn misleading interactions.

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [27]:
import torch.nn as nn
# TODO: Implement a single-head self-attention module.
#
# Requirements (do NOT hardcode shapes):
# - Input: x of shape [B, T, D] where D = d_model
# - Learnable projections: Q = xWq, K = xWk, V = xWv
# - Attention scores: scores = (Q K^T) / sqrt(D)
# - Masking:
#     attention_mask is [B, T] with 1 for real tokens and 0 for padding.
#     We must prevent attending *to padding keys* by setting masked scores to a large negative value.
# - Softmax over the last dimension to get attention weights A: [B, T, T]
# - Output: y = A V projected back to [B, T, D]
#
# Output behavior:
# - If return_attn=True, return (y, attn_weights)
# - Otherwise return y only.
#
# Notes:
# - Be careful about broadcasting the mask to match [B, T, T].
# - Use numerically stable masking (large negative score before softmax).
# - Add dropout on attention weights (optional but recommended).
#
# You are free to structure the code differently (e.g., combine projections, store parameters differently, etc.)
# as long as the behavior matches the requirements.


class SingleHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False):
        B, T, D = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)

        if attention_mask is not None:

            mask = attention_mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        y = torch.matmul(attn_weights, V)
        y = self.out_proj(y)

        if return_attn:
            return y, attn_weights
        return y


## Single-Head Transformer Block (Residual + LayerNorm + FFN)

We now wrap single-head self-attention into a **Transformer-style block** by adding:  
1) a residual path + normalization around attention, and  
2) a position-wise **feed-forward network (FFN)** with another residual + normalization.

This gives us the smallest “Transformer brick” we can plug into an NER model without introducing multi-head attention yet.

In [28]:
# TODO: Implement a minimal Transformer-style block (single-head version).
#
# Requirements (keep it simple and readable):
# - Use the provided SingleHeadSelfAttention module.
# - Add a residual connection around attention.
# - Apply LayerNorm in a consistent way (choose a pattern and document it):
#     * Post-LN (as written in many classic tutorials), or
#     * Pre-LN (often used for stability in deeper stacks).
# - Add a position-wise feed-forward network (FFN), then another residual + normalization.
# - Support an `attention_mask` so padding tokens are handled correctly.
# - If `return_attn=True`, return both (output, attention_weights).
#
# Notes:
# - Keep tensor shapes consistent: input/output should both be [B, T, d_model].
# - Avoid changing the external API: forward(x, attention_mask=None, return_attn=False).
#
# After implementing, we will plug this block into an NER tagger and compare to Part 1.

import math

class SingleHeadTransformerBlock(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False):

        B, T, D = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)

        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(1)
            scores = scores.masked_fill(mask_expanded == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)

        output = self.out_proj(context)

        if return_attn:
            return output, attn_weights
        return output

## Hybrid Tagger: BiLSTM Encoder + Single-Head Attention Block

In this section, we combine a **sequence-aware BiLSTM** (strong classical baseline) with a **single-head Transformer block** on top.

The goal is to isolate one idea:  
**What changes when we add a global mixing mechanism (self-attention) after a sequential encoder?**

We will keep the training/evaluation protocol consistent with Part 1, so any gains or failures can be attributed to this architectural change—not to changes in the experiment setup.

In [29]:
# TODO: Implement a hybrid NER tagger that combines:
#   (1) word embeddings
#   (2) a BiLSTM encoder (sequence-aware baseline feature extractor)
#   (3) a projection into a "model dimension" d_model
#   (4) ONE Transformer-style block (single-head attention + FFN)
#   (5) a token-level classifier over NER labels
#
# Requirements:
# - Input: input_ids [B, T], lengths [B], attention_mask [B, T] (1 real, 0 pad)
# - Output: logits [B, T, num_labels]
# - Use packing/unpacking for the LSTM so padded tokens do not waste compute.
# - Include dropout in reasonable places (embedding / between modules / before classifier).
# - If return_attn=True: return (logits, attn_weights) so we can visualize attention later.
#
# Design freedom (choose and document your choices):
# - Where to apply dropout (embedding output, LSTM output, after attention block, etc.)
# - Whether to apply projection before/after LSTM (but keep shapes consistent)
# - Whether to share dropout modules or use separate ones
#
# NOTE:
# - Keep tensor shapes consistent: logits must align with padded batch length T.
# - attention_mask should be forwarded into the Transformer block.

class BiLSTMTransformerTagger(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        emb_dim: int = 128,
        lstm_hidden: int = 192,
        d_model: int = 192,
        lstm_layers: int = 1,
        dropout: float = 0.4,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)

        self.lstm = nn.LSTM(
            emb_dim,
            lstm_hidden // 2,
            num_layers=lstm_layers,
            bidirectional=True,
            batch_first=True
        )

        self.proj = nn.Linear(lstm_hidden, d_model)

        # Transformer Block
        self.transformer = SingleHeadTransformerBlock(d_model, dropout)

        # Classifier
        self.classifier = nn.Linear(d_model, num_labels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, lengths, attention_mask=None, return_attn=False):
        # 1. Embed
        x = self.embedding(input_ids)
        x = self.dropout(x)

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(packed_out, batch_first=True)

        h = self.proj(lstm_out)

        if return_attn:
            h, weights = self.transformer(h, attention_mask, return_attn=True)
        else:
            h = self.transformer(h, attention_mask)

        h = self.dropout(h)
        logits = self.classifier(h)

        if return_attn:
            return logits, weights
        return logits

## Loss on Real Tokens Only (Mask-Aware)

In token classification, sequences are padded to a common length.  
Here we define the loss so that **padding does not contribute** (using `ignore_index`) and we can report loss **normalized by the number of real tokens**.

In [30]:
def loss_sum_and_tokens(logits: torch.Tensor, labels: torch.Tensor):
    """
    logits: [B, T, C], labels: [B, T]
    returns: (loss_sum over real tokens, num_real_tokens)
    """
    B, T, C = logits.shape
    loss_sum = F.cross_entropy(
        logits.view(B * T, C),
        labels.view(B * T),
        ignore_index=cfg.ignore_index,
        reduction="sum",
    )
    n_tokens = (labels != cfg.ignore_index).sum().item()
    return loss_sum, n_tokens


def loss_mean(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)
    return loss_sum / max(n_tokens, 1)

## Training & Evaluation Loops (Attention-Aware)

With this cell we can practice writing a clean training loop and a fair evaluation loop.  
The key requirement is **mask correctness**: padding must not affect loss or metrics, and attention must respect the `attention_mask`.

In [31]:
def train_one_epoch_attn(model, loader, optimizer, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, lengths, attention_mask=attention_mask)

        loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)

        loss = loss_sum / max(n_tokens, 1)
        loss.backward()

        if grad_clip:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        total_loss += loss_sum.item()
        total_tokens += n_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_attn(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    token_correct = 0

    y_true_all = []
    y_pred_all = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, lengths, attention_mask=attention_mask)
        preds = logits.argmax(dim=-1)

        loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)
        total_loss += loss_sum.item()
        total_tokens += n_tokens

        mask = labels != cfg.ignore_index
        token_correct += ((preds == labels) & mask).sum().item()

        batch_true, batch_pred = ids_to_tags(labels.cpu(), preds.cpu())
        y_true_all.extend(batch_true)
        y_pred_all.extend(batch_pred)

    precision = precision_score(y_true_all, y_pred_all)
    recall = recall_score(y_true_all, y_pred_all)
    f1 = f1_score(y_true_all, y_pred_all)

    return {
        "loss": total_loss / max(total_tokens, 1),
        "token_acc": token_correct / max(total_tokens, 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

## Initialize the Part 2 Model (BiLSTM + Single-Head Attention)

In this cell we define the **Part 2 model instance** and choose its main capacity knobs (embedding size, hidden size, attention dimension, dropout, etc.).  
We keep the **input/output interface** fixed so evaluation stays comparable, but we leave the architectural choices to you.

In [32]:
globals().keys()


dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', '_exit_code', '_i2', '_i3', '_i4', 'os', 'warnings', 'hf_logging', 'dataclass', 'List', 'Dict', 'Tuple', 'random', 'math', 'time', 'inspect', 'Counter', 'np', 'pd', 'torch', 'nn', 'F', 'Dataset', 'DataLoader', 'pack_padded_sequence', 'pad_packed_sequence', 'load_dataset', 'DatasetDict', 'f1_score', 'precision_score', 'recall_score', 'classification_report', 'plt', 'tqdm', 'AutoTokenizer', 'AutoModelForTokenClassification', 'DataCollatorForTokenClassification', 'TrainingArguments', 'Trainer', '_i5', 'CFG', 'cfg', '_5', '_i6', 'seed_everything', 'device', '_i7', 'BASE', 'data_files', 'ds', '_i8', 'ensure_ner_tags', '_i9', 'ONTONOTES_LABEL2ID', 'id2label', 'label2id', 'max_id', 'label_names', 'num_labels', 'all_ids', 'split', 'seq', 'missing', '_i10', 'infer_label_ids_from_sp

In [1]:
attn_model = BiLSTMTransformerTagger(
    vocab_size=len(stoi),
    num_labels=num_labels,
    pad_id=stoi[cfg.pad_token],
    emb_dim=128,
    lstm_hidden=192,
    d_model=192,
    lstm_layers=1,
    dropout=0.4,
)

attn_model = attn_model.to(device)


NameError: name 'BiLSTMTransformerTagger' is not defined

## Train Part 2 and Early-Stop on Validation F1

In this cell we train the **BiLSTM + single-head attention** tagger under a controlled setup.  
We track both optimization signals (loss) and NER signals (entity-level precision/recall/F1), and we **keep the best model** according to validation performance (typically **entity-level F1**).

In [ ]:
import copy

optimizer = torch.optim.AdamW(
    attn_model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

num_epochs = 20
patience = 3
best_f1 = -1.0
patience_ctr = 0

history2 = {
    "train_loss": [],
    "val_loss": [],
    "val_token_acc": [],
    "val_f1": [],
}

best_state = None

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch_attn(
        attn_model, train_loader, optimizer
    )

    val_metrics = evaluate_attn(attn_model, valid_loader)

    history2["train_loss"].append(train_loss)
    history2["val_loss"].append(val_metrics["loss"])
    history2["val_token_acc"].append(val_metrics["token_acc"])
    history2["val_f1"].append(val_metrics["f1"])

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f}"
    )

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        patience_ctr = 0
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in attn_model.state_dict().items()
        }
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print("Early stopping triggered.")
            break

# reload best checkpoint
attn_model.load_state_dict(best_state)
attn_model.to(device)


Epoch 01 | train_loss=0.6311 | val_loss=0.4658 | val_f1=0.1483
Epoch 02 | train_loss=0.4669 | val_loss=0.3820 | val_f1=0.2684
Epoch 03 | train_loss=0.4201 | val_loss=0.3537 | val_f1=0.3173
Epoch 04 | train_loss=0.3973 | val_loss=0.3369 | val_f1=0.3506
Epoch 05 | train_loss=0.3815 | val_loss=0.3218 | val_f1=0.3631
Epoch 06 | train_loss=0.3690 | val_loss=0.3150 | val_f1=0.3675
Epoch 07 | train_loss=0.3588 | val_loss=0.3053 | val_f1=0.3821


In [ ]:
##################
#batch = next(iter(train_loader))
#input_ids, labels, lengths, attention_mask = batch

#print(type(input_ids))        # should be <class 'torch.Tensor'>
#print(input_ids.shape)        # [B, T]
#print(type(labels), labels.shape)


## Visualize Training Curves (Part 2)

We plot how the model evolves over epochs.  
Loss curves help diagnose optimization and overfitting, while **entity-level F1** is the key signal for NER quality (token accuracy is only a secondary diagnostic).

In [ ]:

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(history2['train_loss'], label='Train Loss', marker='o', color='royalblue')
plt.plot(history2['val_loss'], label='Val Loss', marker='o', color='darkorange')
plt.title('Part 2: BiLSTM + Self-Attention - Loss Dynamics')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
plt.plot(history2['val_token_acc'], label='Val Token Acc', color='green', linestyle='--')
plt.plot(history2['val_f1'], label='Val Entity F1', color='red', marker='s')
plt.title('Part 2: BiLSTM + Self-Attention - Validation Metrics')
plt.xlabel('Epochs')
plt.ylabel('Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Test Evaluation and Quick Baseline Comparison (Part 2)

We evaluate the attention-augmented model on the **held-out test set** and report the full NER metric suite.  
If Part 1 results are available, we also print a short comparison—primarily focusing on **entity-level F1** to judge whether attention improves span-level predictions.

In [ ]:


test_metrics_attn = evaluate_attn(attn_model, test_loader)

print(f"Test Loss:      {test_metrics_attn['loss']:.4f}")
print(f"Test Token Acc: {test_metrics_attn['token_acc']:.4f}")
print(f"Test Precision: {test_metrics_attn['precision']:.4f}")
print(f"Test Recall:    {test_metrics_attn['recall']:.4f}")
print(f"Test F1 Score:  {test_metrics_attn['f1']:.4f}")
print("-" * 30)

if 'test_metrics' in locals():
    print(f"Baseline (Part 1) F1: {test_metrics['f1']:.4f}")
    print(f"Attention (Part 2) F1: {test_metrics_attn['f1']:.4f}")
    diff = test_metrics_attn['f1'] - test_metrics['f1']
    print(f"Improvement:           {diff:+.4f}")

## Attention Map Visualization on a Single Sentence (Part 2)

We take one validation sentence, run the attention-augmented model in a mode that returns **attention weights**, and visualize the resulting **token-to-token attention matrix**.  
This helps us inspect (qualitatively) which tokens attend to which others—while remembering that attention is not a guaranteed explanation, just a useful signal to explore.

In [ ]:
import seaborn as sns
import torch

example_idx = 5
example_item = test_loader.dataset[example_idx]

batch = ner_collate_fn([example_item])

input_ids = batch['input_ids'].to(device)
lengths = batch['lengths'].to(device)
attention_mask = batch['attention_mask'].to(device)

attn_model.eval()
with torch.no_grad():
    logits, attn_weights = attn_model(
        input_ids,
        lengths,
        attention_mask=attention_mask,
        return_attn=True
    )

L = lengths.item()

preds_ids = logits.argmax(dim=-1).squeeze(0).cpu().numpy()
tokens_list = [itos[idx.item()] for idx in input_ids[0][:L]]

preds_list = [id2label[idx] for idx in preds_ids[:L]]

print("Tokens:       ", tokens_list)
print("Predicted Tags:", preds_list)
attn_mat = attn_weights.squeeze(0).cpu().numpy()

attn_mat_cropped = attn_mat[:L, :L]

plt.figure(figsize=(10, 8))
sns.heatmap(
    attn_mat_cropped,
    xticklabels=tokens_list,
    yticklabels=tokens_list,
    cmap="viridis",
    annot=False,
    cbar=True
)
plt.title(f"Self-Attention Weights (Test Ex {example_idx})")
plt.xlabel("Key (Source)")
plt.ylabel("Query (Target)")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Concept Checks (answer in Markdown, no code)

1. **Why is “BiLSTM + attention” a cleaner experiment than “attention-only” at this stage?**  
   What capability does the BiLSTM provide that attention alone does not automatically guarantee?



Answer:BiLSTMs inherently capture sequence order and local proximity. Attention is permutation-invariant and relies entirely on positional encodings to understand order. BiLSTM guarantees a sequential backbone.

2. In the attention formula, we scale by $\sqrt{d}$.  
   **What numerical issue is this scaling trying to reduce?**  
   (Explain intuitively, not with a proof.)


Answer:As dimensionality $d$ increases, the dot product values grow large in magnitude. This pushes the Softmax function into "flat" regions where the gradient is extremely small (vanishing gradients), making the model stop learning. Scaling keeps the variance of the scores stable.

3. Self-attention produces a matrix $A \in \mathbb{R}^{T \times T}$ (per example).  
   **What do rows and columns represent in that matrix?**  
   If token $i$ strongly attends to token $j$, what does that mean operationally?


Answer: Represent the Query (the token looking for context).Columns j: Represent the Key (the tokens being looked at).Operational Meaning: If $i$ attends strongly to $j$, the model is "copying" or aggregating features from token $j$ into the new representation of token $i$.

4. Suppose Part 2 improves token accuracy but barely changes entity-level F1.  
   **Give one realistic reason this could happen in NER.**


Answer: NER datasets are dominated by "O" (Outside) tags (often 90%+ of the data). A model can increase token accuracy by simply getting better at identifying "O" tags, while still failing to correctly identify the boundaries or types of actual entities, which is what F1 measures.

# Part 3 — Attention-Only Tagger (No Positional Encoding)

In this part we intentionally **remove the sequential encoder** (BiLSTM) and try to solve NER using **self-attention only**.  
This is a controlled stress test: attention is excellent at mixing information across tokens, but **order is not “built in”** unless we explicitly provide it.

We keep the experimental setup as consistent as possible (same dataset, same evaluation metrics, same masking rules), and we focus on one central question:

> **What breaks when we remove sequence awareness, and why?**

We will:
- implement an attention-only encoder block (self-attention + feed-forward + layer norm + residuals),
- train it on NER,
- evaluate it with **entity-level precision/recall/F1** (not just token accuracy),
- and run interpretability/diagnostic checks that probe whether the model understands token order.

## Single-Head Self-Attention (Core Module)

In this cell we implement the fundamental attention mechanism used throughout the Transformer family.  
We will compute attention scores, apply optional scaling for stability, correctly mask padded tokens, and return either the transformed sequence or (optionally) the attention weight matrix for visualization.

In [ ]:
import math

class SingleHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False):

        B, T, D = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)


        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)

        if attention_mask is not None:

            mask_expanded = attention_mask.unsqueeze(1)
            scores = scores.masked_fill(mask_expanded == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, V)

        output = self.out_proj(context)

        if return_attn:
            return output, attn_weights
        return output

## Attention-Only Tagger Skeleton

In this cell we define an **attention-only** NER model: it must map token IDs to per-token label logits while handling padding correctly.  
We keep the design intentionally open-ended so we can later study how “attention without order information” behaves under different reasonable implementations.

In [ ]:
class AttentionOnlyTagger(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        d_model: int = 128,
        dropout: float = 0.2,
        use_scaling: bool = True,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.transformer = SingleHeadTransformerBlock(d_model, dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor = None,
        attention_mask: torch.Tensor = None,
        return_attn: bool = False,
    ):
        x = self.embedding(input_ids)
        x = self.dropout(x)


        if return_attn:
            x, weights = self.transformer(x, attention_mask, return_attn=True)
        else:
            x = self.transformer(x, attention_mask)

        logits = self.classifier(x)

        if return_attn:
            return logits, weights
        return logits

## Evaluation for Attention-Based Taggers

We define a dedicated evaluation function for attention-based NER models.  
The key requirement is **fairness**: we must compute **exactly the same metrics** as earlier parts (especially entity-level F1) while correctly ignoring padded tokens.

In [ ]:
@torch.no_grad()
def evaluate_attn(model, loader):
    model.eval()
    total_loss, total_tokens, token_correct = 0.0, 0, 0
    y_true_all, y_pred_all = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, lengths, attention_mask=attention_mask)
        loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)

        total_loss += loss_sum.item()
        total_tokens += n_tokens

        preds = logits.argmax(dim=-1)
        mask = labels != cfg.ignore_index
        token_correct += ((preds == labels) & mask).sum().item()

        batch_true, batch_pred = ids_to_tags(labels.cpu(), preds.cpu())
        y_true_all.extend(batch_true)
        y_pred_all.extend(batch_pred)

    return {
        "loss": total_loss / max(total_tokens, 1),
        "token_acc": token_correct / max(total_tokens, 1),
        "precision": precision_score(y_true_all, y_pred_all),
        "recall": recall_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all),
    }



## One-Epoch Training Loop for Attention Models

We implement a single-epoch training loop that mirrors earlier parts, but now passes an **attention mask** and enforces **padding-aware loss**.  
We keep the structure minimal so we can later compare different training choices without changing the rest of the notebook.

In [ ]:
def train_one_epoch_attn(model, loader, optimizer, grad_clip: float = 1.0):
    model.train()
    total_loss, total_tokens = 0.0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        lengths = batch["lengths"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, lengths, attention_mask=attention_mask)
        loss_sum, n_tokens = loss_sum_and_tokens(logits, labels)

        loss = loss_sum / max(n_tokens, 1)
        loss.backward()

        if grad_clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss_sum.item()
        total_tokens += n_tokens

    return total_loss / max(total_tokens, 1)

## Part 3 Setup: Attention-Only Model + Optimizer

We instantiate our **attention-only** NER tagger and choose an optimizer configuration.  
We keep this cell flexible on purpose so we can later study how architecture and optimization choices affect performance.

In [ ]:
# TODO: Instantiate the Part 3 model (attention-only tagger) and its optimizer.

# Requirements:
# - Use the same `stoi`, `num_labels`, and `cfg.pad_token` as earlier parts.
# - Move the model to `device`.
# - Create an optimizer (e.g., AdamW) for model parameters.

pad_id = stoi[cfg.pad_token]

attn_model = AttentionOnlyTagger(
    vocab_size=len(stoi),
    num_labels=num_labels,
    pad_id=pad_id,
    d_model=128,
    dropout=0.2,
    use_scaling=True
).to(device)

optimizer = torch.optim.AdamW(attn_model.parameters(), lr=1e-3, weight_decay=1e-4)

# TODO: Choose architecture hyperparameters (examples: d_model, dropout, scaling).
# attn_model = AttentionOnlyTagger(
#     vocab_size=...,
#     num_labels=...,
#     pad_id=...,
#     d_model=...,
#     dropout=...,
#     use_scaling=...,
# ).to(device)

# TODO: Print the model (optional but useful for sanity checking).
# print(attn_model)

# TODO: Choose optimizer + hyperparameters (lr, weight_decay, etc.)
# optimizer = torch.optim.AdamW(attn_model.parameters(), lr=..., weight_decay=...)

## Part 3 Training: Validation-Selected Checkpointing

We train the attention-only model and select checkpoints using **validation entity-level F1**.  
The goal is to keep the training protocol comparable across parts, while still allowing experimentation with stopping rules and optimization details.

In [ ]:
# TODO: Train the attention-only model with validation-based selection.

# Requirements:
# - Use `train_one_epoch_attn(...)` and `evaluate_attn(...)` from earlier cells.
# - Track training/validation curves in a `history` dict (for plotting later).
# - Save the best model based on validation entity-level F1 (recommended for NER).
# - Restore the best checkpoint at the end.

# TODO: Choose training budget + early stopping policy
# EPOCHS = ...
# patience = ...

# TODO: Initialize tracking variables
# best_f1 = ...
# best_state = ...
# bad_epochs = ...
# history3 = {"train_loss": [], "val_loss": [], "val_token_acc": [], "val_f1": []}

# TODO: Main training loop
# for epoch in range(1, EPOCHS + 1):
#     train_loss = ...
#     val_metrics = ...
#
#     # Update history
#     # history3[...].append(...)
#
#     # Print a clean summary line (loss, token_acc, P/R/F1)
#
#     # Early stopping + checkpointing:
#     # - if val F1 improves: save best_state, reset bad_epochs
#     # - else: increment bad_epochs and stop when it reaches patience

# TODO: Restore best checkpoint
# attn_model.load_state_dict(best_state)
# print("Loaded best model with val_F1 =", best_f1)
EPOCHS = 20
patience = 5
best_f1 = -1.0
best_state = None
bad_epochs = 0
history3 = {"train_loss": [], "val_loss": [], "val_token_acc": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_attn(attn_model, train_loader, optimizer)
    val_metrics = evaluate_attn(attn_model, valid_loader)

    history3["train_loss"].append(train_loss)
    history3["val_loss"].append(val_metrics["loss"])
    history3["val_token_acc"].append(val_metrics["token_acc"])
    history3["val_f1"].append(val_metrics["f1"])

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['token_acc']:.4f}")

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_state = attn_model.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

attn_model.load_state_dict(best_state)

## Training Curves for Part 3

We visualize optimization and generalization by plotting **loss** (train vs validation) and **validation metrics** (entity-level F1 vs token accuracy) across epochs.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
epochs = range(1, len(history3["train_loss"]) + 1)
plt.plot(epochs, history3["train_loss"], label="Train Loss", marker='o', linestyle='-', alpha=0.7)
plt.plot(epochs, history3["val_loss"], label="Val Loss", marker='o', linestyle='-', alpha=0.7)
plt.title("Part 3: Attention-Only Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss (per token)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(epochs, history3["val_token_acc"], label="Val Token Acc", marker='s', color='green', alpha=0.7)
plt.plot(epochs, history3["val_f1"], label="Val Entity F1", marker='^', color='red', alpha=0.7)
plt.title("Part 3: Attention-Only Validation Metrics")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## Final Evaluation on the Test Set (Part 3)

We run a **held-out test evaluation** using the same masked loss and NER span-level metrics as before, so we can compare Part 3 fairly against earlier architectures.


In [ ]:
# TODO: Evaluate the final Part 3 model on the test set and print the results.
#
# Requirements:
# - Run your evaluation pipeline on `test_loader` using the best/selected model checkpoint.
# - Report (at minimum):
#   1) average test loss (masked over real tokens)
#   2) token-level accuracy (masked)
#   3) entity-level precision, recall, and F1 (BIO span-based)
#
# Notes:
# - Do NOT assume a specific return type for your evaluation function.
#   Use whatever structure you designed earlier (dict, namedtuple, etc.).
# - Print the metrics in a clean, readable single-line or multi-line format.
# - Keep the same metric definitions as Parts 0–2 so comparisons are fair.

test_results_p3 = evaluate_attn(attn_model, test_loader)

print(f"{'Metric':<15} | {'Value':<10}")
print("-" * 28)
print(f"{'Test Loss':<15} | {test_results_p3['loss']:.4f}")
print(f"{'Token Accuracy':<15} | {test_results_p3['token_acc']:.4f}")
print(f"{'Precision':<15} | {test_results_p3['precision']:.4f}")
print(f"{'Recall':<15} | {test_results_p3['recall']:.4f}")
print(f"{'Entity F1':<15} | {test_results_p3['f1']:.4f}")
print("-" * 28)

if 'final_comparison' not in locals():
    final_comparison = {}
final_comparison["Part 3: Attn Only"] = test_results_p3['f1']


## Attention Map Visualization (No Positional Encoding)

We visualize a **single-head self-attention heatmap** on a real validation sentence to qualitatively inspect what information the model is using when it has **no explicit positional encoding**.

In [ ]:

batch = next(iter(valid_loader))
input_ids = batch['input_ids'].to(device)
lengths = batch['lengths'].to(device)
mask = batch['attention_mask'].to(device)

attn_model.eval()
with torch.no_grad():
    logits, attn_weights = attn_model(input_ids, lengths, attention_mask=mask, return_attn=True)

idx = 0
L = min(lengths[idx].item(), 25)
tokens = [itos[i.item()] for i in input_ids[idx][:L]]
matrix = attn_weights[idx, :L, :L].cpu().numpy()

plt.figure(figsize=(10, 8))
sns.heatmap(matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
plt.title("Part 3: Attention Map (No Positional Encoding)")
plt.xticks(rotation=45)
plt.show()

## Permutation Sanity Check (Order Sensitivity)

We run a small diagnostic where we **shuffle token order** in a short window and compare the model’s behavior before vs. after shuffling. The purpose is to test whether, **without positional encoding**, the model can reliably represent *order-dependent* structure in NER.

In [ ]:
# TODO: Permutation sanity check (order sensitivity diagnostic).
#
# Goal:
# Show (empirically) that an attention-only model WITHOUT positional encoding
# is largely insensitive to token order (beyond token identity / content),
# by comparing behavior before vs. after shuffling tokens.
#
# What to implement:
# 1) Select ONE example (batch-of-1 is easiest) from a loader and choose a token window length L.
# 2) Run the model on the original input and extract:
#    - predicted tags (optional but recommended)
#    - attention matrix (if your model can return it)
# 3) Create a permuted version of the first L tokens (and permute the mask consistently).
# 4) Run the model on the permuted input and extract the same outputs.
# 5) Compare results in a way that makes the invariance (or lack of it) visible.
#
# You get to choose the comparison method:
# - Compare logits/predictions before vs. after permutation
# - Compare attention maps (raw) and/or attention maps re-indexed back to original order
# - Print the permutation and the token lists to keep the experiment interpretable
#
# Rules:
# - Do NOT change the model architecture here (the point is to probe it).
# - Keep the test small and readable (e.g., L <= 20).
#
# Expected output:
# - A short, clear qualitative comparison (plots and/or prints) that supports a conclusion
#   about whether the model is order-sensitive without positional information.

idx = 0
orig_seq = input_ids[idx:idx+1, :L]
orig_mask = mask[idx:idx+1, :L]

with torch.no_grad():
    orig_logits = attn_model(orig_seq, None, attention_mask=orig_mask)

perm = torch.randperm(L)
perm_seq = orig_seq[:, perm]
perm_mask = orig_mask[:, perm]

with torch.no_grad():
    perm_logits = attn_model(perm_seq, None, attention_mask=perm_mask)

token_val = itos[orig_seq[0, 0].item()]
new_pos = (perm == 0).nonzero(as_tuple=True)[0].item()

print(f"Token '{token_val}' original logit sample: {orig_logits[0, 0, :3]}")
print(f"Token '{token_val}' permuted logit sample: {perm_logits[0, new_pos, :3]}")
print("\nConclusion: If the logits match exactly, the model is order-insensitive (Permutation Equivariant).")

## Concept Checks (answer in Markdown, no code)

1. **Why is this an important ablation?**  
   What specific capability do we lose by removing BiLSTM-style recurrence?


Answer=Removing the BiLSTM while keeping only the attention mechanism is a critical ablation because it isolates the inductive bias of order.

The Loss of Recurrence: BiLSTMs process tokens sequentially, meaning the representation of a word is inherently built upon the history of the words before and after it. This creates a natural "neighborhood" effect where local context is prioritized.

The Capability Gap: By removing the BiLSTM, you lose the innate sense of sequence. Without recurrence or positional encodings, self-attention treats the sentence like a "bag of words." It can see that the word "York" is in the sentence, but it doesn't automatically know if it came before "New" or after "City." This ablation proves whether the attention mechanism can survive on "content" alone (word identity) without "context" (word position).

2. **Self-attention mixes tokens — so why might order still be missing?**  
   In plain language, explain why “mixing” is not the same as “knowing positions.”

Answer=Think of self-attention as a blender rather than a timeline.

Mixing: Imagine a recipe. Self-attention looks at all the ingredients (tokens) in the bowl. It decides that "Flour" needs to interact with "Sugar" to make a cake. It "mixes" their information together based on how well they match.

Missing Positions: If you shuffle the order of the ingredients before putting them in the blender, the final mixture (the output) remains exactly the same. The blender doesn't care if the sugar was poured in first or last; it only cares that the sugar is present.

In Plain Language: Mixing tells you what is related to what, but it doesn't tell you where they were standing. Without positional encodings, "The dog bit the man" and "The man bit the dog" result in the exact same mathematical "mixture" for the word "bit."

3. **Interpretation caution:**  
   If we visualize a single-head attention map, why is it risky to conclude that the model “understands” syntax or entities from the heatmap alone?

Answer= It is tempting to look at a heatmap and say, "Look! The word 'New' is attending to 'York,' so the model understands the entity 'New York'!" This is risky for several reasons:

Correlation vs. Causation: Just because a model focuses on a token doesn't mean it "understands" it in a human sense. It might simply be attending to "York" because it’s a high-frequency word or because of a statistical artifact in the embedding space.

The "Single Head" Limitation: In models with multiple attention heads (or even a single one), a head might be specialized for something purely mechanical, like looking at the very next token or the [SEP] token, rather than semantic syntax.

Visualization is Not Decision-Making: Attention weights show the flow of information, but they don't show the logic applied to that information in the later Feed-Forward layers. A model could attend to the correct word but still produce the wrong label because the subsequent processing logic is flawed.

# Part 4 — Adding Positional Encoding (Fixing the Order Blindness)

In Part 3 we intentionally built an **attention-only** tagger *without* any explicit notion of token position.  
That setup is useful because it exposes a core limitation: **self-attention can mix information globally, but it does not inherently know token order** unless we give it some positional signal.

In this part we make **one minimal, controlled change**:

- We keep the attention-only architecture the same.
- We add **sinusoidal positional encoding** to the token embeddings before attention.

Everything else (data, masking, loss, optimizer style, evaluation metrics) should remain as consistent as possible so that we can interpret changes in performance as “the effect of position”.

By the end of Part 4, we should be able to answer a simple experimental question:

> Does injecting positional information recover performance and make attention behave more “structure-aware” on an order-sensitive task like NER?

## What we should pay attention to in this part

- **Fair comparison:** We should not change multiple knobs at once.  
- **Evaluation:** We still care most about **entity-level F1**, not token accuracy.
- **Interpretability (with caution):** Attention maps may look different with positional encoding, but we should treat them as *signals*, not proofs.

## Sinusoidal Positional Encoding

We add a **fixed sinusoidal positional signal** to token embeddings so the attention-only model can represent **token order**.  
In this cell, we implement a reusable module that precomputes a position table and adds the correct prefix for a given sequence length.

In [ ]:
import math

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor):
        return x + self.pe[:, :x.size(1), :]

## Attention-Only Tagger + Positional Encoding (Part 4)

We repeat the Part 3 attention-only tagger, but we inject **sinusoidal positional information** into the embeddings so the model can represent **token order**.  
This is an ablation-style step: we keep everything else as comparable as possible and test whether adding position restores NER performance.

In [ ]:
class AttentionOnlyTaggerWithPE(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        d_model: int = 128,
        dropout: float = 0.2,
        max_len: int = 512,
        use_scaling: bool = True,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=max_len)

        self.transformer = SingleHeadTransformerBlock(d_model, dropout=dropout)

        self.classifier = nn.Linear(d_model, num_labels)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor = None,
        attention_mask: torch.Tensor = None,
        return_attn: bool = False
    ):
        x = self.embedding(input_ids)
        x = self.pos_encoder(x)
        x = self.dropout(x)

        if return_attn:
            x, attn = self.transformer(x, attention_mask=attention_mask, return_attn=True)
        else:
            x = self.transformer(x, attention_mask=attention_mask)
        logits = self.classifier(x)

        if return_attn:
            return logits, attn
        return logits

## Notice: We Reuse the Part 3 Training/Evaluation Code

**Do not re-implement `train_one_epoch_attn(...)` and `evaluate_attn(...)` in Part 4.**

In Part 4, we intentionally keep the **same training loop and the same evaluation harness** from Part 3.  
This is a controlled experiment: the *only* conceptual change we want to test is **adding positional encoding**.

## Initialize the Positional-Encoding Model

We fix the random seed again and instantiate the **attention-only tagger with sinusoidal positional encoding**.  
Keeping the initialization protocol consistent helps us attribute any performance changes to **positional information**, not to randomness or a different setup.

In [ ]:
seed_everything(cfg.seed)

pe_model = AttentionOnlyTaggerWithPE(
    vocab_size=len(stoi),
    num_labels=num_labels,
    pad_id=stoi[cfg.pad_token],
    d_model=128,      # keep consistent with Part 3 unless we explicitly ablate this
    dropout=0.2,
    max_len=512,
    use_scaling=True,
).to(device)

## Train the PE Model (Early Stopping on Entity F1)

We now train the **attention-only model with sinusoidal positional encoding** under the same evaluation protocol as Part 3.  
Your job is to implement a clean training loop, track the key metrics, and use **early stopping based on validation entity-level F1** so the comparison stays fair and meaningful.

In [ ]:
# Optimizer and training configuration (you may tune these, but keep Part 3 vs Part 4 comparisons fair)
optimizer = torch.optim.AdamW(pe_model.parameters(), lr=5e-4, weight_decay=0.01)

EPOCHS = 7
patience = 2  # early stopping patience (based on validation entity-level F1)

# Track the best checkpoint (by validation F1)
best_f1_pe = -1.0
best_state_pe = None
bad_epochs = 0

# Keep a lightweight training history for plotting/diagnostics
history4 = {"train_loss": [], "val_loss": [], "val_token_acc": [], "val_f1": []}

# TODO:
# Train the positional-encoding model for up to `EPOCHS` epochs.
#
# Requirements:
# 1) Each epoch should:
#    - run one full training epoch (reuse the helper you used in Part 3)
#    - evaluate on the validation split (reuse the helper you used in Part 3)
#    - append metrics to `history4`
#    - print a concise progress line (epoch, train_loss, val_loss, val_token_acc, val_F1)
#
# 2) Early stopping:
#    - monitor validation entity-level F1 (NOT token accuracy)
#    - if F1 improves (with a small tolerance), save the model weights into `best_state_pe`
#    - otherwise increment `bad_epochs`
#    - stop when `bad_epochs >= patience`
#
# 3) After the loop:
#    - load `best_state_pe` back into `pe_model`
#    - print the best validation F1 you achieved
#
# Notes:
# - Keep the comparison fair: avoid changing multiple factors at once (e.g., EPOCHS + model size + optimizer).
# - If you change any training detail, document it briefly in a Markdown cell near the change.

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_attn(pe_model, train_loader, optimizer)
    val_metrics = evaluate_attn(pe_model, valid_loader)

    history4["train_loss"].append(train_loss)
    history4["val_loss"].append(val_metrics["loss"])
    history4["val_token_acc"].append(val_metrics["token_acc"])
    history4["val_f1"].append(val_metrics["f1"])

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
          f"Acc: {val_metrics['token_acc']:.4f} | F1: {val_metrics['f1']:.4f}")

    if val_metrics["f1"] > best_f1_pe:
        best_f1_pe = val_metrics["f1"]
        best_state_pe = pe_model.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

pe_model.load_state_dict(best_state_pe)
print(f"Training finished. Best Val F1: {best_f1_pe:.4f}")

## Test Evaluation (Report Final Metrics)

We report **test-set** loss, token accuracy, and entity-level precision/recall/F1 for the positional-encoding model.  
This is the number we will later compare against Part 3 to see how much **positional information** changes performance.

In [ ]:
test_metrics4 = evaluate_attn(pe_model, test_loader)

print(
    f"PART 4 TEST | "
    f"loss={test_metrics4['loss']:.4f} | "
    f"token_acc={test_metrics4['token_acc']:.4f} | "
    f"P={test_metrics4['precision']:.4f} | "
    f"R={test_metrics4['recall']:.4f} | "
    f"F1={test_metrics4['f1']:.4f}"
)

## Plot Learning Curves (Diagnostics)

We visualize our Part 4 training dynamics to diagnose optimization and generalization:  
loss curves tell us whether training is stable, and score curves (especially entity-level F1) tell us whether improvements translate to better NER performance.

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history4["train_loss"], label="Train Loss")
plt.plot(history4["val_loss"], label="Val Loss")
plt.title("Part 4: Loss Curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history4["val_f1"], label="Val F1", color='red')
plt.plot(history4["val_token_acc"], label="Val Acc", color='green')
plt.title("Part 4: Validation Metrics")
plt.legend()
plt.show()

## Compare With vs Without Positional Encoding

Here we make a *fair* visual comparison between Part 3 (attention-only **without** positional encoding) and Part 4 (the **same** model but **with** positional encoding).  
We compare validation curves (loss and at least one NER score such as entity-level F1) across epochs to see whether adding positional information improves learning dynamics and final tagging quality.


In [ ]:
plt.figure(figsize=(10, 6))
# Handle potentially different history lengths due to early stopping
plt.plot(history3["val_f1"], label="Part 3 (No PE)", linestyle='--')
plt.plot(history4["val_f1"], label="Part 4 (Sinusoidal PE)", linewidth=2)
plt.title("Comparison: Validation F1 with vs. without Positional Encoding")
plt.xlabel("Epoch")
plt.ylabel("Entity F1 Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Attention Map Comparison: No PE vs With PE

In this cell we compare *interpretability signals* across two nearly identical models:  
the attention-only tagger **without** positional encoding (Part 3) and the same architecture **with** sinusoidal positional encoding (Part 4).

We visualize a single sentence’s attention matrix from each model (cropped to real token length) and compare patterns qualitatively.  
The goal is not to “prove” correctness from attention, but to develop intuition about how positional information can reshape attention structure.

In [ ]:
# 1. Get batch and example
batch = next(iter(valid_loader))
idx = 0
L = min(batch['lengths'][idx].item(), 20)
input_ids = batch['input_ids'][idx:idx+1].to(device)
mask = batch['attention_mask'][idx:idx+1].to(device)

# 2. Get Attention from both models
attn_model.eval() # Part 3 model
pe_model.eval()   # Part 4 model
with torch.no_grad():
    _, attn3 = attn_model(input_ids, None, attention_mask=mask, return_attn=True)
    _, attn4 = pe_model(input_ids, None, attention_mask=mask, return_attn=True)

# 3. Plot
tokens = [itos[i.item()] for i in input_ids[0, :L]]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

sns.heatmap(attn3[0, :L, :L].cpu(), xticklabels=tokens, yticklabels=tokens, ax=ax1, cmap="viridis")
ax1.set_title("Part 3 (No PE)")

sns.heatmap(attn4[0, :L, :L].cpu(), xticklabels=tokens, yticklabels=tokens, ax=ax2, cmap="viridis")
ax2.set_title("Part 4 (With PE)")

plt.tight_layout()
plt.show()

## Concept Checks (answer in Markdown, no code)

### Q1 — Why positional encoding is needed
Self-attention computes attention weights using content-based similarity.  
Why does this make the model *potentially invariant* to token order when we remove any positional signal?

Answer=Self-attention is permutation equivariant. If you shuffle the input tokens, the output vectors remain identical in content; they simply move to the new shuffled positions. Because the model has no internal mechanism to distinguish index $i$ from index $j$, it treats the sequence as a bag-of-words. In NER, this means it cannot tell the difference between "New York" and "York New."

### Q2 — What sinusoidal positional encoding provides
Sinusoidal positional encoding is a deterministic function of position index.  
What property makes it reasonable for sequence models, and what kinds of information can it inject into embeddings?

Answer= Sinusoidal encoding uses sine and cosine functions of different frequencies to provide:Unique Time-stamps: Every position gets a unique, deterministic vector.Relative Positioning: Due to trigonometric identities, the encoding at $pos+k$ is a linear function of $pos$. This allows the model to learn that tokens close to each other should interact differently than tokens far apart.

### Q3 — Controlled ablation logic
If we keep everything the same except positional encoding, what conclusions are we allowed to draw from:
- a large improvement in entity F1?
- a small/no improvement?
- a drop in performance?

Answer= Large improvement: Proves that word order is a primary feature for identifying entities (the expected result for NER).

Small/No improvement: Suggests the task can be solved by local keywords or global context regardless of order.

Performance drop: Suggests the encoding is acting as noise or the model is overfitting to specific sequence positions.

### Q4 — Attention-map comparison interpretation
When we compare attention maps between Part 3 and Part 4:
- What qualitative patterns would we *expect* to change if positional information is being used?
- Why is it still risky to claim “the model learned X” based purely on attention heatmaps?


Answer=Expected Patterns: Without PE, you often see "vertical bands" (global focus). With PE, you expect more diagonal density, showing the model has developed a sense of local proximity.

Risks: Attention weights don't always equal "importance." A model might attend to a token for mechanical reasons (like a delimiter) rather than semantic understanding. Additionally, much logic happens in the Feed-Forward layers, which are invisible in a heatmap.

# Part 5 — Multi-Head Attention

In Parts 3–4 we saw that **self-attention can mix global context**, but it also needs **positional information** to work well on order-sensitive tasks like NER.  
Now we take the next step toward real Transformer encoders: **multi-head self-attention**.

The key idea is simple:

- Instead of learning *one* attention pattern, we learn **multiple attention heads in parallel**.
- Each head can focus on different relationships (local neighborhoods, long-range dependencies, boundary cues, etc.).
- We then **combine** those heads to form a richer representation.

In this part we will:
- implement **multi-head self-attention** (single layer first),
- wrap it into a standard **encoder-layer block** (attention + feed-forward + residuals + layer norm),
- stack multiple layers,
- train and compare against our earlier attention models,
- and finally **inspect attention heads** to build intuition about what different heads might be doing.


## Multi-Head Self-Attention (TODO)

In this cell we implement **multi-head self-attention**: multiple attention heads run in parallel on different learned subspaces, then their outputs are concatenated and projected back to \(d_{\text{model}}\). The key technical requirements are correct **tensor shapes**, correct **scaled dot-product attention**, and correct **padding masking** so pad tokens do not receive attention.

In [ ]:
# TODO (Part 5): Implement Multi-Head Self-Attention (MHA)
#
# Goal:
#   Write a PyTorch module that takes a sequence representation x and returns
#   a transformed representation using multi-head scaled dot-product attention.
#
# Required interface:
#   class MultiHeadSelfAttention(nn.Module):
#       def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1): ...
#       def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None,
#                   return_attn: bool = False): ...
#
# Inputs:
#   x:              [B, T, d_model]
#   attention_mask: [B, T] where 1 indicates a real token and 0 indicates padding
#
# Outputs:
#   y:    [B, T, d_model]
#   attn: (optional) attention weights with shape [B, H, T, T]
#
# Design requirements (you decide how to implement, but must satisfy behavior):
#   1) Split the model dimension into H heads:
#        d_head = d_model / H   (so d_model must be divisible by num_heads)
#   2) Compute per-head Q, K, V projections (any equivalent efficient form is fine).
#   3) Compute scaled dot-product attention:
#        scores = (Q @ K^T) / sqrt(d_head)
#      IMPORTANT: use correct transpose for the last two dimensions of K.
#   4) Apply padding mask so padded KEYS do not receive attention.
#      (Mask should affect the "columns" of the [T x T] score matrix.)
#   5) Softmax over the last dimension to get attention probabilities.
#   6) Multiply attention by V, then combine heads back to [B, T, d_model].
#   7) Apply output projection (and dropout if you want).
#   8) If return_attn=True, return (y, attn); otherwise return y.
#
# Notes:
#   - There are multiple correct implementations. We care about correctness,
#     shapes, and masking behavior—not matching a specific code style.
#   - Choose stable masking (e.g., a very negative fill value) to prevent pads
#     from influencing attention.
#
# Hint checklist (shapes):
#   - After projection, typical head shapes are [B, H, T, d_head]
#   - scores should become [B, H, T, T]
#   - attention_mask must broadcast to scores correctly
#
# Write your implementation below.

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False):
        B, T, C = x.shape
        H = self.num_heads

        q = self.q_proj(x).view(B, T, H, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, self.d_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # 4. Softmax + Dropout
        attn_weights = torch.softmax(scores, dim=-1)
        attn_probs = self.dropout(attn_weights)
        context = torch.matmul(attn_probs, v)
        context = context.transpose(1, 2).contiguous().view(B, T, C)

        y = self.out_proj(context)

        if return_attn:
            return y, attn_weights
        return y


## Transformer Encoder Layer (MHA + FFN) (TODO)

In this cell we build a **Transformer encoder layer**: multi-head self-attention followed by a position-wise feed-forward network, each wrapped with **residual connections**, **dropout**, and **layer normalization**. This layer becomes the reusable “brick” we will stack to form deeper Transformer encoders.

In [ ]:
# TODO (Part 5): Implement a Transformer Encoder Layer using Multi-Head Attention
#
# Goal:
#   Build a standard Transformer-style encoder layer:
#     1) Multi-Head Self-Attention sublayer
#     2) Feed-Forward Network (FFN) sublayer
#   Each sublayer should use:
#     - Residual connection
#     - Dropout
#     - LayerNorm
#
# Required interface:
#   class TransformerEncoderLayerMHA(nn.Module):
#       def __init__(self, d_model: int, num_heads: int, dropout: float = 0.2, ff_mult: int = 4): ...
#       def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False): ...
#
# Inputs:
#   x:              [B, T, d_model]
#   attention_mask: [B, T] (1 for real tokens, 0 for padding)
#
# Outputs:
#   x_out: [B, T, d_model]
#   attn:  (optional) attention weights from the MHA sublayer (shape depends on your MHA)
#
# Design requirements:
#   - The attention sublayer should call your MultiHeadSelfAttention.
#   - Apply residual + dropout + layernorm around attention.
#   - Then apply FFN (2 linear layers with an activation, e.g. GELU) and again residual + dropout + layernorm.
#   - If return_attn=True, return (x_out, attn); otherwise return x_out.
#
# Notes:
#   - You may choose "Pre-LN" or "Post-LN" style, but be consistent.
#   - Keep shapes consistent throughout.

class TransformerEncoderLayerMHA(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.2, ff_mult: int = 4):
        super().__init__()
        self.mha = MultiHeadSelfAttention(d_model, num_heads, dropout)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False):

        if return_attn:
            attn_out, attn_weights = self.mha(x, attention_mask, return_attn=True)
        else:
            attn_out = self.mha(x, attention_mask)

        x = self.norm1(x + self.dropout1(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        if return_attn:
            return x, attn_weights
        return x

## Multi-Head Attention-Only Tagger (Stacked Encoder) (TODO)

In this cell we assemble the **Part 5 model**: token embeddings, optional positional encoding, a **stack of Transformer encoder layers with multi-head self-attention**, and a final per-token classifier for NER. We also add a hook to optionally return attention maps from a chosen layer for interpretability experiments later in the section.

In [ ]:
# TODO (Part 5): Implement a multi-layer, multi-head attention-only NER tagger
#
# Goal:
#   Build an encoder that uses:
#     - Token embeddings
#     - (Optional) positional encoding
#     - A stack of TransformerEncoderLayerMHA blocks
#     - A token-level classifier to predict NER tags
#
# Required interface:
#   class MultiHeadAttentionOnlyTagger(nn.Module):
#       def __init__(...): ...
#       def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, return_attn_layer: int = -1): ...
#
# Inputs:
#   input_ids:       [B, T]
#   attention_mask:  [B, T] (1 real tokens, 0 padding)
#   return_attn_layer:
#       - If set to a valid layer index, the forward pass should also return that layer's attention map.
#       - If set to -1, no attention map is returned.
#
# Outputs:
#   logits: [B, T, num_labels]
#   (optional) attn: attention weights from the requested layer (shape depends on your MHA implementation)
#
# Design degrees of freedom (you choose):
#   - Whether to scale embeddings by sqrt(d_model)
#   - Where/how to apply dropout
#   - Whether positional encoding is sinusoidal or learned (keep interface compatible)
#   - How you manage "return_attn_layer" (only one layer’s attention is needed)
#
# Notes:
#   - Keep the implementation consistent with your Part 4 positional encoding (if you reuse it).
#   - Make sure masking is respected inside attention.

class MultiHeadAttentionOnlyTagger(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        d_model: int = 192,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.2,
        max_len: int = 512,
        use_positional_encoding: bool = True,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)

        self.pos_encoding = None
        if use_positional_encoding:
            self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            TransformerEncoderLayerMHA(d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(d_model, num_labels)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor = None,      # <--- ADD THIS HERE
        attention_mask: torch.Tensor = None,
        return_attn_layer: int = -1,
    ):
        x = self.embedding(input_ids)
        if self.pos_encoding:
            x = self.pos_encoding(x)
        x = self.dropout(x)

        saved_attn = None
        for i, layer in enumerate(self.layers):
            # Pass only the mask to the layers
            if i == return_attn_layer:
                x, saved_attn = layer(x, attention_mask=attention_mask, return_attn=True)
            else:
                x = layer(x, attention_mask=attention_mask)

        logits = self.classifier(x)

        if return_attn_layer >= 0:
            return logits, saved_attn
        return logits

## Train the Multi-Head Attention Model (TODO)

In this cell we run the full **training + validation** loop for our multi-head encoder, track **loss and NER metrics**, and apply **early stopping** based on **validation entity-level F1**. At the end, we reload the best checkpoint so evaluation and visualization always reflect the best-performing model from this run.

In [ ]:
# TODO (Part 5): Train the Multi-Head Attention model + early stopping + checkpointing
#
# In this cell we will:
#   1) Instantiate your MultiHeadAttentionOnlyTagger (mha_model)
#   2) Print the number of trainable parameters (optional but recommended)
#   3) Define an optimizer (and any scheduler you want)
#   4) Run a training loop that tracks validation metrics each epoch
#   5) Implement early stopping based on validation entity-level F1
#   6) Save the best model state and reload it at the end
#
# Important:
#   - Reuse your Part 3/4 utilities (train_one_epoch_attn, evaluate_attn, loss_mean, etc.) if you wrote them.
#   - If those utilities were TODOs in your notebook version, do NOT paste the full solutions here.
#     Instead call your own implementations from earlier cells.
#   - Keep comparisons fair: same splits, same metric definitions, and similar budgets across parts.

def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

mha_model = MultiHeadAttentionOnlyTagger(
    vocab_size=len(stoi),
    num_labels=num_labels,
    pad_id=stoi[cfg.pad_token],
    d_model=128,
    num_heads=4,
    num_layers=2
).to(device)

print(f"Trainable params: {count_params(mha_model):,}")

optimizer = torch.optim.AdamW(mha_model.parameters(), lr=5e-4, weight_decay=0.01)

EPOCHS = 10
patience = 3
best_f1 = -1.0
best_state = None
bad_epochs = 0
history = {"train_loss": [], "val_loss": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_attn(mha_model, train_loader, optimizer)
    val_metrics = evaluate_attn(mha_model, valid_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics['loss'])
    history["val_f1"].append(val_metrics['f1'])

    print(f"Epoch {epoch} | Loss: {train_loss:.4f} | Val F1: {val_metrics['f1']:.4f}")

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        best_state = mha_model.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience: break

mha_model.load_state_dict(best_state)

# TODO: instantiate mha_model with the chosen hyperparameters
# mha_model = MultiHeadAttentionOnlyTagger(...).to(device)

# TODO: print parameter count (optional)
# print("Trainable params:", f"{count_params(mha_model):,}")

# TODO: define optimizer (and optional scheduler)
# optimizer = ...

# TODO: set training hyperparameters (epochs, patience, grad clipping, etc.)
# EPOCHS = ...
# patience = ...

# TODO: initialize early-stopping bookkeeping and history dict
# best_f1 = ...
# best_state = ...
# history = ...

# TODO: training loop:
# for epoch in range(...):
#     train_loss = ...
#     val_metrics = ...
#     log metrics
#     update best checkpoint based on val F1
#     early stop if no improvement

# TODO: load best checkpoint
# mha_model.load_state_dict(best_state)
# print("Loaded best MHA model with val_F1 =", best_f1)

## Plot Curves and Test Evaluation (TODO)

In this cell we visualize the **training dynamics** (loss and validation metrics), then evaluate the best multi-head model on the **test set** using the same NER metric pipeline as earlier parts. Finally, we optionally print side-by-side comparisons against previous baselines to make the ablation story clear.

In [ ]:
# TODO (Part 5): Plot training curves + evaluate on test + compare with earlier parts
#
# In this cell we will:
#   1) Plot the training/validation loss curves from your history dict
#   2) Plot validation F1 vs token accuracy (optional but recommended)
#   3) Evaluate the final/best MHA model on the test set
#   4) Optionally compare the test F1 to earlier parts (e.g., single-head with/without PE)
#
# Notes:
#   - This cell assumes you stored training logs in a dictionary (e.g., hist5 or history).
#   - It also assumes you have evaluate_attn(...) implemented (or an equivalent evaluator).
#   - Keep plotting readable: label axes, include legends, and avoid clutter.
#   - For comparisons, only compare runs that used the same splits + same metric logic.

# TODO: pick your history container name
# history = hist5  # example

# TODO: plot loss curves

# TODO: plot validation metrics curves

# TODO: evaluate on test set
# mha_test = evaluate_attn(mha_model, test_loader)
# print metrics in a clean format

# TODO (optional): compare to Part 4 / previous models if you stored their test metrics
# if "..." in globals(): print(...)

import matplotlib.pyplot as plt

# --- 1) Plot Loss and Metrics for Part 5 ---
plt.figure(figsize=(14, 5))

# Plot Losses
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="MHA Train Loss", color='blue', alpha=0.7)
plt.plot(history["val_loss"], label="MHA Val Loss", color='orange', linestyle='--', alpha=0.7)
plt.title("Part 5: Multi-Head Attention Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

# Plot Validation F1 vs Token Acc
plt.subplot(1, 2, 2)
plt.plot(history["val_f1"], label="Val Entity F1", color='red', marker='o')
if "val_token_acc" in history:
    plt.plot(history["val_token_acc"], label="Val Token Acc", color='green', marker='s')
plt.title("Part 5: MHA Validation Metrics")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("FINAL TEST EVALUATION: PART 5 (MULTI-HEAD ATTENTION)")
print("="*50)

mha_test = evaluate_attn(mha_model, test_loader)

for metric, value in mha_test.items():
    print(f"{metric.capitalize():<15}: {value:.4f}")

print("\n" + "="*50)
print("CROSS-PART F1 COMPARISON (TEST SET)")
print("="*50)

comparisons = {
    "Part 3 (Single Head, No PE)": locals().get('test_results_p3', {}).get('f1', 'N/A'),
    "Part 4 (Single Head, With PE)": locals().get('test_metrics4', {}).get('f1', 'N/A'),
    "Part 5 (Multi-Head, With PE)": mha_test['f1']
}

for model_name, f1_val in comparisons.items():
    if isinstance(f1_val, float):
        print(f"{model_name:<30}: {f1_val:.4f}")
    else:
        print(f"{model_name:<30}: {f1_val}")

## Multi-Head Attention Maps (Interpretability) — TODO

In this cell we visualize **multi-head self-attention** on a single sentence to see whether different heads appear to specialize (e.g., local context vs. long-range links). We treat the plots as **diagnostics**, not as final explanations: we focus on correct token alignment, masking/cropping padded positions, and careful interpretation.


In [ ]:
# TODO (Part 5): Multi-head attention visualization (interpretability mini-lab)
#
# Goal:
#   Visualize attention weights for several heads on ONE example sentence.
#
# Requirements (keep it general):
#   - Choose one example (from validation is recommended).
#   - Identify the true (unpadded) token span length and crop all visuals to it.
#   - Run the trained model in evaluation mode and obtain:
#       (a) token-level predictions (optional, for qualitative inspection)
#       (b) attention weights from ONE chosen layer
#   - Plot attention heatmaps for multiple heads (e.g., a small subset).
#
# What we should be careful about:
#   - Token alignment: axis tick labels must match the exact tokens being attended over.
#   - Padding: do not visualize padding rows/columns (crop before plotting).
#   - Interpretation: attention is a diagnostic signal, not a proof of explanation.
#
# Deliverable:
#   - A figure (or multiple figures) showing attention matrices per head.
#   - (Optional) print a compact token + predicted-tag view for the same example.

# TODO: pick one example sentence and collect its tokens + model inputs

# TODO: compute the real token length (exclude padding) and define a cropping length L

# TODO: run the model to obtain attention weights for a selected layer (and optionally predictions)

# TODO: extract the attention tensor and crop it to shape [num_heads, L, L]

# TODO: plot attention heatmaps for a few heads with tokens on both axes


import seaborn as sns
import matplotlib.pyplot as plt

example_batch = next(iter(valid_loader))
idx = 0
L = example_batch['lengths'][idx].item()
input_ids = example_batch['input_ids'][idx:idx+1].to(device)
mask = example_batch['attention_mask'][idx:idx+1].to(device)

mha_model.eval()
with torch.no_grad():
    logits, attn = mha_model(input_ids, attention_mask=mask, return_attn_layer=0)

# 3. Plot heads
tokens = [itos[i.item()] for i in input_ids[0, :L]]
num_heads_to_plot = 4
fig, axes = plt.subplots(1, num_heads_to_plot, figsize=(20, 5))

for h in range(num_heads_to_plot):
    head_map = attn[0, h, :L, :L].cpu().numpy()
    sns.heatmap(head_map, xticklabels=tokens, yticklabels=tokens, ax=axes[h], cmap="magma", cbar=False)
    axes[h].set_title(f"Head {h}")
    axes[h].set_xticklabels(tokens, rotation=90)

plt.tight_layout()
plt.show()

## Concept Checks (answer in Markdown, no code)

1. In multi-head attention, we typically split the representation into subspaces of size:
   $$
   d_{\mathrm{head}} = \frac{d_{\mathrm{model}}}{H}.
   $$
   What is the intuition behind using smaller per-head dimensions instead of keeping one full-size attention?



Answer=The intuition is parallel specialization. Instead of one large head trying to capture every relationship, multiple smaller heads act as an ensemble of experts. One head can focus on syntax (e.g., matching verbs to subjects), while another focuses on semantics (e.g., identifying entity types). A single head would likely "average out" these distinct signals.

2. Multi-head attention increases modeling capacity, but it also changes compute/memory usage.  
   **Which part of attention has the dominant cost in sequence length $T$**, and why?

Answer=The dominant cost is calculating the attention matrix ($Q \times K^T$).Why: This operation results in a $T \times T$ matrix, leading to quadratic complexity $O(T^2)$. If you double the sentence length, the memory and computation required quadruple.

3. When we visualize attention heads, we might see different patterns per head.  
   Give **two plausible reasons** why heads can look different (even within the same layer).


Answer=Even in the same layer, heads diverge because they develop unique functional roles:

Local vs. Global Focus: One head might attend to immediate neighbors (diagonal pattern) for grammar, while another attends to special tokens like periods (vertical pattern) for global context.

Feature Specialization: One head may specialize in visual patterns (like capitalization in "New York"), while another focuses on relational cues (like identifying that a word following "at" is likely a location).

# Part 6 — Going Deeper: A Tiny Transformer Encoder + Depth Ablation

In Parts 4–5, we saw that **positional information** makes attention usable for order-sensitive tagging, and that **multi-head attention** can represent multiple interaction patterns in parallel.

In this part we push one more core “Transformer/BERT idea”:

- instead of *one* encoder layer, we stack **multiple encoder blocks** (attention + feed-forward + residual + layer norm),
- and we treat the notebook like a small empirical study: **how does depth change NER performance?**

We will run a **depth ablation**: train several models that differ only in the **number of encoder layers**, while keeping the rest of the setup as controlled as possible (data, tokenizer/vocab, metric computation, optimization budget, etc.).

### What we should keep fixed (to make the ablation meaningful)

To interpret results correctly, we should try to change **one variable at a time**:

- keep the same dataset splits and evaluation function,
- keep the same embedding size and number of heads,
- keep positional encoding **on** (because Part 4 established it as necessary),
- keep the training recipe comparable (optimizer family, early stopping logic, etc.).

If we decide to change any of these, we should document it briefly (and explain why).

## Encoder Block (BERT-style): Attention + FFN + Residual + LayerNorm

In this cell we implement the **basic building block** of a Transformer encoder.  
The block has two sublayers—**multi-head self-attention** and a **position-wise feedforward network**—each wrapped with a **residual connection** and **LayerNorm**.

**Guiding questions:**
- What must stay shape-preserving so we can stack many layers cleanly?
- Where exactly do we apply masking so padding tokens do not affect attention?

In [ ]:
# TODO: Implement a single BERT-style encoder block (a Transformer encoder layer).
#
# Requirements / expectations:
# - Inputs:
#   - x: Tensor of shape [B, T, D]
#   - attention_mask: optional Tensor of shape [B, T] (1 for real tokens, 0 for padding)
# - Core components (standard "encoder layer" recipe):
#   1) Multi-head self-attention sublayer (already available as MultiHeadSelfAttention)
#   2) Residual connection + LayerNorm
#   3) Position-wise feedforward network (MLP) sublayer
#   4) Residual connection + LayerNorm
# - Optional: support returning attention weights (return_attn=True)
#
# Notes:
# - Keep tensor shapes consistent across the block.
# - Masking must ensure padding tokens do not act as keys that receive attention.
# - Do NOT change the forward signature; later parts will rely on it.

class BertStyleEncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1, ff_mult: int = 4):
        super().__init__()
        self.mha = MultiHeadSelfAttention(d_model, num_heads, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model)
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, return_attn: bool = False):

        if return_attn:
            attn_out, attn_weights = self.mha(x, attention_mask, return_attn=True)
        else:
            attn_out = self.mha(x, attention_mask)

        x = self.norm1(x + self.dropout1(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        if return_attn:
            return x, attn_weights
        return x

## Tiny Transformer Encoder for NER (BERT-style stack)

In this cell we define a **small Transformer encoder** for token classification.  
We stack multiple encoder blocks, optionally add **positional encoding**, and attach a **per-token classifier**.

**Guiding questions:**
- What tensor shapes must remain unchanged so the blocks can be stacked safely?
- Where should positional information be injected, and why *before* the blocks?
- What is the purpose of returning attention from a single selected layer instead of all layers?

In [ ]:
# TODO: Implement a tiny Transformer-style encoder for NER.
#
# Goal:
# - Build a stack of BERT-style encoder blocks and attach a token-level classifier on top.
#
# Inputs / outputs:
# - input_ids: [B, T]
# - attention_mask: [B, T] (1 real token, 0 padding)
# - return logits: [B, T, num_labels]
# - Optional: return attention from a chosen layer (return_attn_layer)
#
# Design constraints:
# - Use nn.Embedding for token embeddings.
# - (Optionally) add positional encoding before the encoder blocks.
# - Use a ModuleList of BertStyleEncoderBlock.
# - Keep shapes consistent across layers: [B, T, D] in / out.
#
# Notes:
# - For stability, consider scaling embeddings by sqrt(d_model) (common in Transformers).
# - Masking should be applied inside attention blocks (keys must ignore padding).
# - return_attn_layer:
#     * if set to a valid layer index, that block should return attention weights too.
#     * if no attention requested, forward should return only logits.

class TinyEncoderForNER(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        d_model: int = 192,
        num_heads: int = 4,
        num_layers: int = 4,
        dropout: float = 0.1,
        max_len: int = 512,
        use_positional_encoding: bool = True,
    ):
        super().__init__()
        self.d_model = d_model
        self.use_pe = use_positional_encoding

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.emb_dropout = nn.Dropout(dropout)

        if self.use_pe:
            self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)

        self.blocks = nn.ModuleList([
            BertStyleEncoderBlock(d_model, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.out_dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)

        nn.init.xavier_uniform_(self.classifier.weight)

    def forward(
        self,
        input_ids: torch.Tensor,
        lengths: torch.Tensor = None,
        attention_mask: torch.Tensor = None,
        return_attn_layer: int = -1,
    ):
        x = self.embedding(input_ids) * math.sqrt(self.d_model)
        x = self.emb_dropout(x)

        if self.use_pe:
            x = self.pos_encoding(x)

        saved_attn = None
        for i, block in enumerate(self.blocks):
            if i == return_attn_layer:
                x, saved_attn = block(x, attention_mask, return_attn=True)
            else:
                x = block(x, attention_mask)

        logits = self.classifier(self.out_dropout(x))

        if return_attn_layer >= 0:
            return logits, saved_attn
        return logits

## Depth Ablation Runner (Fair Comparison Across Encoder Depths)

This cell defines a **single reusable training routine** used to compare models of different depth under the **same protocol** (same data, same metrics, same stopping rule).

In [ ]:
# TODO: Implement a reusable training runner for the depth ablation study.
#
# Goal:
# - Train the same architecture while varying ONLY the number of encoder layers (depth).
# - Keep the training/evaluation protocol consistent across depths so the comparison is fair.
#
# Requirements / expectations:
# - Re-seed at the start (reproducibility).
# - Build a model whose depth is controlled by `num_layers`.
#   (You may use your Part 5/6 model class, or a TinyEncoder variant.)
# - Use a standard optimizer (e.g., AdamW) with provided lr/weight_decay.
# - Track history per epoch: train_loss, val_loss, val_token_acc, val_f1.
# - Use early stopping based on validation F1 (or a clearly justified alternative).
# - At the end, evaluate once on the test set and return:
#     (best_model, history_dict, best_val_f1, test_metrics_dict)
#
# Notes:
# - The core of this lab is controlled experimentation:
#   depth is the variable, everything else should be held fixed.

ddef train_mha_depth(
    num_layers: int,
    num_heads: int = 4,
    d_model: int = 192,
    lr: float = 5e-4,
    weight_decay: float = 0.02,
    max_epochs: int = 12,
    patience: int = 3,
):
    seed_everything(cfg.seed)

    model = TinyEncoderForNER(
        vocab_size=len(stoi),
        num_labels=num_labels,
        pad_id=stoi[cfg.pad_token],
        d_model=d_model,
        num_heads=num_heads,
        num_layers=num_layers
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_f1 = -1.0
    best_state = None
    bad_epochs = 0
    hist = {"train_loss": [], "val_loss": [], "val_f1": [], "val_token_acc": []}

    print(f"\n>>> Training Depth L={num_layers} ({count_params(model):,} params)")

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch_attn(model, train_loader, optimizer)
        val_m = evaluate_attn(model, valid_loader)

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(val_m["loss"])
        hist["val_f1"].append(val_m["f1"])
        hist["val_token_acc"].append(val_m["token_acc"])

        if val_m["f1"] > best_f1:
            best_f1 = val_m["f1"]
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                break

    model.load_state_dict(best_state)
    test_metrics = evaluate_attn(model, test_loader)
    return model, hist, best_f1, test_metrics
    # TODO: seed for reproducibility (same seed for every depth run)
    # seed_everything(cfg.seed)

    # TODO: build a depth-controlled model
    # model = ...
    # model = model.to(device)

    # TODO: create optimizer
    # optimizer = ...

    # TODO: initialize early-stopping bookkeeping
    # best_f1 = ...
    # best_state = ...
    # bad = 0

    # TODO: history dict (exact keys not mandatory, but keep it consistent across depths)
    # hist = {"train_loss": [], "val_loss": [], "val_f1": [], "val_token_acc": []}

    # TODO: training loop
    # for ep in range(1, max_epochs + 1):
    #   - train for one epoch
    #   - evaluate on validation
    #   - log metrics
    #   - early stop on F1
    #   - save best model state

    # TODO: load best checkpoint
    # model.load_state_dict(best_state)

    # TODO: final test evaluation + print a clean summary (optional)
    # test_metrics = ...

    # TODO: return (model, hist, best_f1, test_metrics)

## Run the Depth Sweep (Ablation Driver)

In this cell we run the **depth ablation**: the same encoder design trained multiple times while varying **only** the number of stacked layers.

In [ ]:

depths_to_test = [1, 2, 4, 6]
results = {}

for L in depths_to_test:
    model, hist, best_val, test_m = train_mha_depth(num_layers=L)
    results[L] = {"model": model, "history": hist, "val_f1": best_val, "test_metrics": test_m}

import pandas as pd
summary_data = []
for L, res in results.items():
    summary_data.append({
        "Layers (Depth)": L,
        "Params": count_params(res["model"]),
        "Best Val F1": res["val_f1"],
        "Test F1": res["test_metrics"]["f1"],
        "Test Loss": res["test_metrics"]["loss"]
    })

res_df = pd.DataFrame(summary_data)
display(res_df)



## Plot Curves Across Depths (Train/Val Loss + Val F1)

In this cell we compare **training dynamics** across different encoder depths by plotting learning curves.  
We are not trying to “win” with one model yet — we are trying to understand *how depth changes behavior*.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(res_df["Layers (Depth)"], res_df["Test F1"], marker='o', linestyle='-', linewidth=2)
for i, row in res_df.iterrows():
    plt.annotate(f"{row['Test F1']:.3f}", (row['Layers (Depth)'], row['Test F1'] + 0.005))
plt.title("Depth Ablation: Impact of Layers on Entity F1")
plt.xlabel("Number of Encoder Layers")
plt.ylabel("Test Entity-Level F1")
plt.grid(True, alpha=0.3)
plt.show()

## Depth Ablation Summary Table

In this cell we summarize the depth experiment in a **single table** so we can compare depths without scanning logs.  
The point is to connect *capacity* (depth/parameters) with *generalization* (validation/test entity-level F1).

In [ ]:
depths_to_test = [1, 2, 4, 6]
results = {}

for L in depths_to_test:
    model, hist, best_val, test_m = train_mha_depth(num_layers=L)
    results[L] = {"model": model, "history": hist, "val_f1": best_val, "test_metrics": test_m}

import pandas as pd
summary_data = []
for L, res in results.items():
    summary_data.append({
        "Depth (Layers)": L,
        "Params": count_params(res["model"]),
        "Best Val F1": res["val_f1"],
        "Test F1": res["test_metrics"]["f1"],
        "Test Loss": res["test_metrics"]["loss"]
    })

res_df = pd.DataFrame(summary_data)
print("\n=== Depth Ablation Results ===")
display(res_df)

## Visualizing the Depth Ablation

In this cell we visualize the most important ablation trend: how **test entity-level F1** changes as we increase the **number of encoder layers**.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(res_df["Depth (Layers)"], res_df["Test F1"], marker='o', linestyle='-', linewidth=2)
for i, txt in enumerate(res_df["Test F1"]):
    plt.annotate(f"{txt:.3f}", (res_df["Depth (Layers)"][i], res_df["Test F1"][i]), textcoords="offset points", xytext=(0,10), ha='center')

plt.title("Effect of Encoder Depth on NER Performance")
plt.xlabel("Number of Transformer Layers")
plt.ylabel("Test Entity-Level F1")
plt.grid(True, alpha=0.3)
plt.xticks(depths_to_test)
plt.show()



## Choosing the Best Depth and Inspecting What the Model Learned

Now that we have trained multiple Transformer depths, we do two “research-lab” steps:

1) **Model selection:** we pick the depth that performs best on **test entity-level F1** (with a clear tie-break if needed).  
2) **Qualitative inspection:** we look at one validation sentence, compare predicted tags to tokens, and visualize attention maps if our model exposes them.

In [ ]:
best_row = res_df.loc[res_df["Test F1"].idxmax()]
best_L = int(best_row["Depth (Layers)"])
best_model = results[best_L]["model"]
best_test_f1 = best_row["Test F1"]

print(f"Selected Best Depth: L={best_L} with Test F1={best_test_f1:.4f}")

In [ ]:
import seaborn as sns

model = best_model
model.eval()
batch = next(iter(test_loader))
input_ids = batch["input_ids"].to(device)
lengths = batch["lengths"].to(device)
mask = batch["attention_mask"].to(device)

last_layer_idx = best_L - 1
with torch.no_grad():
    logits, attn_weights = model(input_ids, lengths, attention_mask=mask, return_attn_layer=last_layer_idx)

preds = logits.argmax(dim=-1)
idx = 0
true_len = lengths[idx].item()
tokens = [itos[i.item()] for i in input_ids[idx][:true_len]]
pred_tags = [id2label[p.item()] for p in preds[idx][:true_len]]

print(f"Example Sentence (Depth {best_L}):")
for t, p in zip(tokens, pred_tags):
    print(f"{t:<15} {p}")

if attn_weights is not None:
    att_mat = attn_weights[idx, 0, :true_len, :true_len].cpu().numpy()

    plt.figure(figsize=(10, 8))
    sns.heatmap(att_mat, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
    plt.title(f"Layer {last_layer_idx+1} Head 0 Attention")
    plt.xlabel("Key (Source)")
    plt.ylabel("Query (Target)")
    plt.show()

## Concept Checks (answer in Markdown, no code)

1) **Depth vs. capacity:**  
   What changes when we increase the number of encoder layers? Is it only “more parameters,” or also “more computation steps for composition”?

Answer=Crucially, depth adds computational hops. In a single layer, a token only "sees" its neighbors once. In $N$ layers, a token can aggregate information that has already been processed and contextualized $N-1$ times. This allows the model to build hierarchical abstractions (e.g., Layer 1 processes local syntax, while Layer 6 resolves global semantic dependencies).

2) **Qualitative inspection:**  
   When we inspect one sentence, what kinds of NER errors should we categorize?  
   Which error types are most likely to be improved by “more context mixing” across layers?

Answer=

Boundary Errors

Type Errors

False Negatives


False Positives



Type Errors see the most improvement. Resolving whether "Washington" is a PERSON, LOC, or ORG often requires looking at distant verbs or prepositions (e.g., "signed the law" vs. "traveled to"). More layers allow these distant "clues" to be mixed into the token's representation more effectively. Boundary errors also improve as the model better understands the syntactic "chunk" the word belongs to.

# Part 7 — A Tiny BERT-Like Encoder

In Parts 3–6 we explored attention-only models and the effect of positional information.  
In this part, we move one step closer to **BERT-style inputs and embeddings**:

- We extend our **word-level vocabulary** with special tokens: `[CLS]`, `[SEP]`, `[MASK]`.
- We build a dataset and collate function that produce the typical BERT inputs:
  - `input_ids`
  - `token_type_ids` (segment IDs)
  - `position_ids`
  - `attention_mask`
  - `labels` aligned with the tokens (and ignored on special tokens)
- We implement a **TinyBERT-like** encoder:
  - token + position + segment embeddings
  - stacked Transformer encoder blocks
  - token-level classifier head for NER

The goal is not to reproduce pretrained BERT performance, but to understand **the full input pipeline and architectural ingredients** that make BERT “BERT”.

## Add BERT Special Tokens to the Vocabulary

We ensure `[CLS]`, `[SEP]`, and `[MASK]` exist in our word-level vocabulary so later “BERT-style” dataset and model code can reference them reliably.

In [ ]:
BERT_CLS = "[CLS]"
BERT_SEP = "[SEP]"
BERT_MASK = "[MASK]"


def add_special_tokens_to_vocab(
    stoi: Dict[str, int],
    special_tokens: List[str],
) -> Tuple[Dict[str, int], Dict[int, str]]:
    """
    Ensure required special tokens exist in the word-level vocabulary.

    Args:
        stoi: token -> id mapping (will be updated in-place)
        special_tokens: list of tokens to add if missing

    Returns:
        (stoi, itos) where itos is id -> token mapping
    """
    for tok in special_tokens:
        if tok not in stoi:
            stoi[tok] = len(stoi)

    itos = {i: s for s, i in stoi.items()}
    return stoi, itos


stoi, itos = add_special_tokens_to_vocab(stoi, [BERT_CLS, BERT_SEP, BERT_MASK])

print("Vocab size:", len(stoi))
print(
    "IDs:",
    {t: stoi[t] for t in [cfg.pad_token, cfg.unk_token, BERT_CLS, BERT_SEP, BERT_MASK]},
)

## Build a BERT-Style NER Dataset

We wrap the raw NER data into a BERT-like input format by inserting `[CLS]`/`[SEP]`, truncating to a maximum length, converting tokens to ids, and aligning labels so that special tokens use `ignore_index`.

In [ ]:
class NERBertInputDataset(Dataset):
    """
    A dataset that converts word-level NER examples into a BERT-style format:

      tokens  -> [CLS] tokens ... [SEP]
      labels  -> ignore_index for special tokens, original labels for real tokens
      ids     -> token ids from `stoi`
      token_type_ids -> all zeros for single-sequence inputs

    Notes:
      - `max_len` counts *all* tokens, including [CLS] and [SEP].
      - Truncation must keep tokens/labels aligned.
    """

    def __init__(self, hf_split, stoi: Dict[str, int], max_len: int = 256):
        self.split = hf_split
        self.stoi = stoi
        self.max_len = max_len  # includes [CLS] and [SEP]

    def __len__(self) -> int:
        return len(self.split)

    ef __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ex = self.split[idx]
        tokens = ex["tokens"]
        labels = ex["ner_tags"]

        max_tokens = self.max_len - 2
        tokens = tokens[:max_tokens]
        labels = labels[:max_tokens]

        bert_tokens = [BERT_CLS] + tokens + [BERT_SEP]

        unk_id = self.stoi.get(cfg.unk_token)
        input_ids = [self.stoi.get(t, unk_id) for t in bert_tokens]

        bert_labels = [cfg.ignore_index] + labels + [cfg.ignore_index]

        token_type_ids = [0] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "labels": torch.tensor(bert_labels, dtype=torch.long),
            "tokens": bert_tokens,
        }

## BERT-Style Collation for NER (Padding, Masks, Positions)

We implement a custom `collate_fn` that pads variable-length BERT-formatted sequences, builds an `attention_mask`, generates `position_ids`, and pads labels using `ignore_index` so loss is computed only on real tokens.

In [ ]:
def bert_ner_collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    lengths = torch.tensor([len(ex["input_ids"]) for ex in batch])
    max_batch_len = lengths.max().item()

    pad_id = stoi[cfg.pad_token]
    input_ids = torch.full((len(batch), max_batch_len), pad_id, dtype=torch.long)
    token_type_ids = torch.zeros((len(batch), max_batch_len), dtype=torch.long)
    labels = torch.full((len(batch), max_batch_len), cfg.ignore_index, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_batch_len), dtype=torch.long)
    position_ids = torch.zeros((len(batch), max_batch_len), dtype=torch.long)

    batch_tokens = []

    for i, ex in enumerate(batch):
        L = lengths[i]
        input_ids[i, :L] = ex["input_ids"]
        token_type_ids[i, :L] = ex["token_type_ids"]
        labels[i, :L] = ex["labels"]
        attention_mask[i, :L] = 1
        position_ids[i, :L] = torch.arange(L)
        batch_tokens.append(ex["tokens"])

    return {
        "input_ids": input_ids,
        "token_type_ids": token_type_ids,
        "attention_mask": attention_mask,
        "position_ids": position_ids,
        "labels": labels,
        "tokens": batch_tokens,
        "lengths": lengths
    }

## BERT-Style Embeddings (Token + Position + Segment)

We implement the embedding layer used in BERT-like encoders by combining token embeddings with positional and segment/type embeddings, then applying normalization and dropout before passing representations to the encoder blocks.

In [ ]:
class BertEmbeddings(nn.Module):
    """
    TODO (students): Implement BERT-style input embeddings.

    Goal:
      Build the embedding representation X for each token position by combining:
        1) token embeddings (lookup by input_ids)
        2) position embeddings (lookup by position_ids)
        3) segment/type embeddings (lookup by token_type_ids)

      Then apply:
        - LayerNorm
        - Dropout

    Inputs:
      - input_ids:      LongTensor [B, T]
      - token_type_ids: LongTensor [B, T] (optional; if None, treat as all zeros)
      - position_ids:   LongTensor [B, T] (optional; if None, build 0..T-1 for each row)

    Output:
      - embeddings: FloatTensor [B, T, d_model]

    Design requirements:
      - Use nn.Embedding for token, position, and segment embeddings.
      - Support variable sequence lengths up to `max_len`.
      - Do NOT assume token_type_ids exist in the dataset (must handle None).
      - Keep the interface compatible with later encoder blocks.

    Optional (if you want BERT-like initialization):
      - Initialize embedding weights with Normal(mean=0, std=0.02).
    """
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        pad_id: int,
        max_len: int = 512,
        type_vocab_size: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embeddings = nn.Embedding(max_len, d_model)
        self.token_type_embeddings = nn.Embedding(type_vocab_size, d_model)

        self.LayerNorm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input_ids: torch.Tensor,
        token_type_ids: torch.Tensor = None,
        position_ids: torch.Tensor = None,
    ) :
        seq_length = input_ids.size(1)
        if position_ids is None:
            position_ids = torch.arange(seq_length, dtype=torch.long, device=input_ids.device)
            position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        if token_type_ids is None:
            token_type_ids = torch.zeros_like(input_ids)

        words_emb = self.word_embeddings(input_ids)
        pos_emb = self.position_embeddings(position_ids)
        type_emb = self.token_type_embeddings(token_type_ids)

        embeddings = words_emb + pos_emb + type_emb
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

## Tiny BERT-Like Encoder for NER

We define a small BERT-style model for token classification: BERT-like embeddings, a stack of Transformer encoder blocks, and a token-level classifier head. The forward pass may optionally return an attention matrix from a chosen layer for interpretability.

In [ ]:
class TinyBertLikeForNER(nn.Module):
    """
    TODO (students): Implement a tiny BERT-like encoder for NER.

    High-level structure:
      1) BERT-style embeddings (token + position + segment/type)
      2) Stack of encoder blocks (multi-head self-attention + FFN + residuals + LayerNorm)
      3) Token-level classifier: Linear(d_model -> num_labels)

    Forward should optionally return attention from one chosen layer:
      - If return_attn_layer >= 0, return (logits, attn)
      - Otherwise return logits only
    """
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        pad_id: int,
        d_model: int = 192,
        num_heads: int = 4,
        num_layers: int = 4,
        dropout: float = 0.1,
        max_len: int = 512,
    ):
        super().__init__()
        self.embeddings = BertEmbeddings(vocab_size, d_model, pad_id, max_len, dropout=dropout)

        self.encoders = nn.ModuleList([
            BertStyleEncoderBlock(d_model, num_heads, d_ff=d_model*4, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor = None,
        token_type_ids: torch.Tensor = None,
        position_ids: torch.Tensor = None,
        return_attn_layer: int = -1,
    ):
        x = self.embeddings(input_ids, token_type_ids, position_ids)


        attn_out = None
        for i, layer in enumerate(self.encoders):
            if i == return_attn_layer:
                x, attn_out = layer(x, mask=attention_mask, return_attn=True)
            else:
                x = layer(x, mask=attention_mask)

        x = self.dropout(x)
        logits = self.classifier(x)

        return (logits, attn_out) if return_attn_layer >= 0 else logits

## Inspect a BERT-Style Batch Example

We print one example from `train_bert_loader` to confirm that tokens, labels, attention masks, token-type ids, and position ids are aligned correctly—especially around `[CLS]`, `[SEP]`, padding, and `ignore_index`.

In [ ]:
batch = next(iter(train_bert_loader))

bs = len(batch["tokens"])
idx = 1 if bs > 1 else 0
print(f"BATCH SIZE: {bs} | showing example index: {idx}")

tokens = batch["tokens"][idx]
labels = batch["labels"][idx].tolist()
token_type_ids = batch["token_type_ids"][idx].tolist()
position_ids = batch["position_ids"][idx].tolist()
attention_mask = batch["attention_mask"][idx].tolist()

L = int(sum(attention_mask))  # number of real (non-pad) tokens
print("TOKENS         :", tokens[:L])
print("TOKEN_TYPE_IDS :", token_type_ids[:L])
print("POSITION_IDS   :", position_ids[:L])
print("LABELS         :", labels[:L], f"(ignore_index = {cfg.ignore_index})")

## Sanity Check: Forward Pass Shapes

We instantiate the tiny BERT-like model and run one batch through it to confirm tensor shapes and device placement. This helps catch silent bugs early (especially with padding masks and sequence length handling).

In [ ]:
# ---- Instantiate TinyBertLikeForNER + sanity-check shapes ----
tiny_bert = TinyBertLikeForNER(
    vocab_size=len(stoi),
    num_labels=num_labels,
    pad_id=stoi[cfg.pad_token],
    d_model=192,
    num_heads=4,
    num_layers=4,
    dropout=0.1,
    max_len=MAX_LEN_BERT,
).to(device)

# `batch` is assumed to come from train_bert_loader in a previous cell
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
token_type_ids = batch["token_type_ids"].to(device)
position_ids = batch["position_ids"].to(device)

logits = tiny_bert(
    input_ids,
    attention_mask=attention_mask,
    token_type_ids=token_type_ids,
    position_ids=position_ids,
)

print("input_ids:", input_ids.shape)
print("logits   :", logits.shape, "(expected [B, T, num_labels])")

## Training and Evaluation Loops for the TinyBERT-like Model

We implement the core learning logic: a training loop (forward → loss → backward → update) and an evaluation loop that reports masked loss, token accuracy, and entity-level NER metrics (precision/recall/F1). These functions must correctly ignore padding and special-token positions using `cfg.ignore_index`.

In [ ]:
# TODO: Training + Evaluation loops for TinyBERT-like
#
# Goal:
#   - Implement one epoch of training for a BERT-style model
#   - Implement evaluation that returns loss + token accuracy + NER F1
#
# Notes:
#   - Your batch contains: input_ids, labels, attention_mask,
#     token_type_ids, position_ids (plus tokens/lengths).
#   - Use cfg.ignore_index to ignore padding/[CLS]/[SEP] labels in loss + metrics.
#   - Keep your implementation modular so you can reuse it later.

def train_one_epoch_bert(model, loader, optimizer, grad_clip: float = 1.0):
    model.train()
    total_loss = 0
    total_tokens = 0

    for batch in tqdm(loader, desc="Training", leave=False):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["attention_mask"].to(device)
        tt_ids = batch["token_type_ids"].to(device)
        p_ids = batch["position_ids"].to(device)

        logits = model(input_ids, attention_mask=mask, token_type_ids=tt_ids, position_ids=p_ids)

        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=cfg.ignore_index)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        num_active = (labels != cfg.ignore_index).sum().item()
        total_loss += loss.item() * num_active
        total_tokens += num_active

    return total_loss / total_tokens


@torch.no_grad()
def evaluate_bert(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, total_tokens = 0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask=mask)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=cfg.ignore_index)

        num_active = (labels != cfg.ignore_index).sum().item()
        total_loss += loss.item() * num_active
        total_tokens += num_active

        preds = logits.argmax(dim=-1)
        for i in range(len(labels)):
            valid_mask = labels[i] != cfg.ignore_index
            p_list = [id2label[p.item()] for p in preds[i][valid_mask]]
            l_list = [id2label[l.item()] for l in labels[i][valid_mask]]
            all_preds.append(p_list)
            all_labels.append(l_list)

    return {
        "loss": total_loss / total_tokens,
        "f1": f1_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds),
        "recall": recall_score(all_labels, all_preds)
    }

## Training Loop with Early Stopping and Checkpointing

We set up an optimizer and run multi-epoch training for the TinyBERT-like model. During training, we record learning curves (loss, token accuracy, and NER F1) and implement early stopping on a validation metric (typically F1) by saving and reloading the best model checkpoint.


In [ ]:
# Choose an optimizer (AdamW is standard for BERT-style models)
optimizer = torch.optim.AdamW(tiny_bert.parameters(), lr=5e-4, weight_decay=0.01)

# Training hyperparameters
EPOCHS = 10
patience = 3

# Initialize tracking variables
best_f1 = -1.0
best_state = None
bad_epochs = 0
hist7 = {
    "train_loss": [],
    "val_loss": [],
    "val_token_acc": [],
    "val_f1": [],
}

for ep in range(1, EPOCHS + 1):
    train_loss = train_one_epoch_bert(tiny_bert, train_bert_loader, optimizer)

    val_metrics = evaluate_bert(tiny_bert, val_bert_loader)

    hist7["train_loss"].append(train_loss)
    hist7["val_loss"].append(val_metrics["loss"])
    hist7["val_token_acc"].append(val_metrics["token_acc"])
    hist7["val_f1"].append(val_metrics["f1"])

    print(f"Epoch {ep:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
          f"Val Acc: {val_metrics['token_acc']:.4f} | Val F1: {val_metrics['f1']:.4f}")

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        best_state = {k: v.cpu().clone() for k, v in tiny_bert.state_dict().items()}
        bad_epochs = 0
        print(f"  --> New best F1! Saving checkpoint...")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            break

if best_state:
    tiny_bert.load_state_dict(best_state)
    print("best model state.")

In [ ]:

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(hist7["train_loss"], label="Train Loss")
plt.plot(hist7["val_loss"], label="Val Loss")
plt.title("Loss Curves")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist7["val_f1"], label="Val F1")
plt.plot(hist7["val_token_acc"], label="Val Token Acc")
plt.title("Score Curves")
plt.legend()
plt.show()



## Final Test Evaluation

We run the finalized TinyBERT-like model on the **test set** and report the core metrics (loss, token accuracy, and entity-level F1). This is the main quantitative summary of how well our model generalizes to unseen data.

In [ ]:

test7 = evaluate_bert(tiny_bert, test_bert_loader)
print(f"FINAL TEST F1: {test7['f1']:.4f}")

## Concept Checks (answer in Markdown, no code)

1. **Special tokens & supervision**
   - Why do we set labels for `[CLS]` and `[SEP]` to `ignore_index`? because they are structural markers, not linguistic tokens. They don't carry NER tags like B-PER or I-ORG.
   - What would go wrong if we trained on those positions?Semantic Noise-Context Corruption-Bias

2. **Truncation effects**
   - We truncate tokens to `max_len - 2`. Why exactly “minus 2”?It’s a simple "VIP reservation." If your model’s hard limit is max_len, and BERT requires [CLS] at the start and [SEP] at the end, you only have max_len - 2 slots available for your actual data. If you didn't truncate, the special tokens would be pushed outside the window and the model would fail to process the sequence correctly.
   - What is the trade-off between a larger `max_len` and training speed/memory?Larger max_len: Captures more long-range dependencies.The Cost: Doubling the length quadruples the memory usage and computational time.

3. **Architecture reflection**
   - Compare TinyBERT-like (this part) to the “attention-only with PE” model (Part 4):
     - What is the same?The Core Engine-Position Awareness
     - What is different?Embedding "Sandwich-Positioning Method-Input Formatting

## Masked Language Modeling (MLM): Pretraining Objective

In this part we switch from **supervised NER** to a **self-supervised pretraining task**: **Masked Language Modeling (MLM)** (the core objective used in BERT-style models).  
Instead of predicting entity tags, the model sees sentences where some tokens are hidden and must **predict the original tokens** using context from both sides.

By the end of this part, you will:
- Build an MLM dataset and dynamic masking pipeline (the classic **80/10/10** masking rule).
- Train a small BERT-like encoder + MLM head using only masked positions for the loss.
- Evaluate the model using MLM loss (and optionally top-k masked accuracy).
- Visualize attention maps inside the encoder to see what patterns it learns during pretraining.

## Sanity Check
* This cell is a **sanity check**: it ensures all special tokens were added earlier and have valid IDs.
* If any key is missing, it means the notebook skipped the “add special tokens” step.

In [ ]:
STOI = stoi
ITOS = itos

CLS_ID  = STOI[BERT_CLS]
SEP_ID  = STOI[BERT_SEP]
MASK_ID = STOI[BERT_MASK]
PAD_ID  = STOI[cfg.pad_token]
UNK_ID  = STOI[cfg.unk_token]

print("Shared vocab size:", len(STOI))
print({t: STOI[t] for t in [cfg.pad_token, cfg.unk_token, BERT_CLS, BERT_SEP, BERT_MASK]})

## MLM Dataset (Sentence → BERT-style Input)

In this step, we build a dataset that wraps each sentence as **[CLS] tokens [SEP]** and returns `input_ids` + `token_type_ids`; **masking is NOT done here** (it will be applied later inside the `collate_fn` for dynamic MLM masking).

In [ ]:
class MLMSentenceDataset(Dataset):
    """
    Dataset for MLM pretraining:
    returns a single sentence wrapped with [CLS] ... [SEP].
    (Masking + MLM labels will be created later in the collate_fn.)
    """
    def __init__(self, hf_split, stoi: Dict[str, int], max_len: int = 256):
        self.split = hf_split
        self.stoi = stoi
        self.max_len = max_len  # includes [CLS] and [SEP]

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        ex = self.split[idx]
        tokens = ex["tokens"]

        max_tokens = self.max_len - 2
        tokens = tokens[:max_tokens]

        bert_tokens = [BERT_CLS] + tokens + [BERT_SEP]

        unk_id = self.stoi[cfg.unk_token]
        input_ids = [self.stoi.get(t, unk_id) for t in bert_tokens]

        token_type_ids = [0] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "tokens": bert_tokens,
        }

mlm_train_ds = MLMSentenceDataset(ds["train"], STOI, max_len=256) # Adjust max_len if needed
mlm_valid_ds = MLMSentenceDataset(ds["validation"], STOI, max_len=256)

# Sanity checks
print("MLM train/valid:", len(mlm_train_ds), len(mlm_valid_ds))
print("Example tokens:", mlm_train_ds[0]["tokens"][:15])

## Dynamic MLM Masking + Collate Function

In this step, we implement **BERT-style dynamic masking (15% with 80/10/10 rule)** inside the `collate_fn`, so each epoch sees *different masked positions* without changing the stored dataset.

In [ ]:
def mask_tokens_bert_style(
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    mlm_prob: float = 0.15,
) -> tuple[torch.Tensor, torch.Tensor]:
    labels = input_ids.clone()
    probability_matrix = torch.full(labels.shape, mlm_prob, device=input_ids.device)

    special_tokens_mask = [
        STOI[cfg.pad_token],
        STOI[BERT_CLS],
        STOI[BERT_SEP],
        STOI[BERT_MASK]
    ]

    for special_id in special_tokens_mask:
        probability_matrix.masked_fill_(input_ids == special_id, value=0.0)

    if attention_mask is not None:
        probability_matrix.masked_fill_(attention_mask == 0, value=0.0)

    masked_indices = torch.bernoulli(probability_matrix).bool()

    mlm_labels = torch.full(input_ids.shape, cfg.ignore_index, dtype=torch.long, device=input_ids.device)
    mlm_labels[masked_indices] = input_ids[masked_indices]

    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8, device=input_ids.device)).bool() & masked_indices
    input_ids[indices_replaced] = STOI[BERT_MASK]


    indices_random = torch.bernoulli(torch.full(labels.shape, 0.5, device=input_ids.device)).bool() & masked_indices & ~indices_replaced

    random_words = torch.randint(len(STOI), labels.shape, device=input_ids.device, dtype=torch.long)
    input_ids[indices_random] = random_words[indices_random]


    return input_ids, mlm_labels

In [ ]:
def mlm_collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    lengths = [len(x["input_ids"]) for x in batch]
    max_len = max(lengths)

    input_ids = torch.full((len(batch), max_len), fill_value=PAD_ID, dtype=torch.long)
    token_type_ids = torch.zeros((len(batch), max_len), dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

    tokens_list = []
    for i, ex in enumerate(batch):
        L = len(ex["input_ids"])
        input_ids[i, :L] = ex["input_ids"]
        token_type_ids[i, :L] = ex["token_type_ids"]
        attention_mask[i, :L] = 1
        tokens_list.append(ex["tokens"])

    # Position ids: [0..T-1] for each row
    position_ids = torch.arange(max_len, dtype=torch.long).unsqueeze(0).repeat(len(batch), 1)

    # TODO: apply dynamic masking here (NOT in the dataset)
    masked_input_ids, mlm_labels = mask_tokens_bert_style(
        input_ids=input_ids,
        attention_mask=attention_mask,
        mlm_prob=0.15,
    )

    return {
        "input_ids": masked_input_ids,
        "labels": mlm_labels,
        "token_type_ids": token_type_ids,
        "position_ids": position_ids,
        "attention_mask": attention_mask,
        "tokens": tokens_list,
        "lengths": torch.tensor(lengths, dtype=torch.long),
    }


mlm_train_loader = DataLoader(
    mlm_train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=mlm_collate_fn,
)
mlm_valid_loader = DataLoader(
    mlm_valid_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=mlm_collate_fn,
)

b = next(iter(mlm_train_loader))
{k: (v.shape if torch.is_tensor(v) else type(v)) for k, v in b.items()}

## TinyBERT Encoder + MLM Head (with Weight Tying)

Here we build a small BERT-like encoder and an MLM prediction head, including **weight tying** between the input token embeddings and the output decoder matrix.

In [ ]:
class TinyBertEncoder(nn.Module):
    """
    A tiny BERT-style encoder: embeddings + a stack of encoder blocks.
    Returns contextual token representations h in R^{B×T×D}.
    """
    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        d_model: int = 192,
        num_heads: int = 4,
        num_layers: int = 4,
        dropout: float = 0.1,
        max_len: int = 512,
    ):
        super().__init__()
        self.emb = BertEmbeddings(vocab_size, d_model, pad_id=pad_id, max_len=max_len, dropout=dropout)

        # Stack of Transformer encoder blocks (BERT-style)
        self.blocks = nn.ModuleList([
            BertStyleEncoderBlock(d_model=d_model, num_heads=num_heads, dropout=dropout, ff_mult=4)
            for _ in range(num_layers)
        ])

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, position_ids=None):
        x = self.emb(input_ids, token_type_ids=token_type_ids, position_ids=position_ids)
        for blk in self.blocks:
            x = blk(x, attention_mask=attention_mask, return_attn=False)
        return x  # [B, T, D]


class MLMHead(nn.Module):
    """
    BERT-style MLM head:
      - linear -> GELU -> LayerNorm -> vocabulary projection
    Optionally ties decoder weights to the input embedding matrix.
    """
    def __init__(self, d_model: int, vocab_size: int, tie_weights: nn.Embedding = None):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(d_model)

        # Decoder (no bias; BERT keeps bias as a separate parameter)
        self.decoder = nn.Linear(d_model, vocab_size, bias=False)
        self.bias = nn.Parameter(torch.zeros(vocab_size))

        # TODO (concept): weight tying helps reduce parameters and often improves MLM training
        # If tying is enabled, make decoder weights share storage with token embedding weights.
        if tie_weights is not None:
            self.decoder.weight = tie_weights.weight  # weight tying

    def forward(self, x):
        x = self.dense(x)
        x = self.act(x)
        x = self.ln(x)
        logits = self.decoder(x) + self.bias
        return logits  # [B, T, V]


class TinyBertForMLM(nn.Module):
    """
    Full MLM model: Encoder + MLMHead.
    """
    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        d_model: int = 192,
        num_heads: int = 4,
        num_layers: int = 4,
        dropout: float = 0.1,
        max_len: int = 512,
    ):
        super().__init__()
        self.encoder = TinyBertEncoder(vocab_size, pad_id, d_model, num_heads, num_layers, dropout, max_len)

        # TODO: tie MLM decoder weights to token embedding matrix (like BERT)
        tied = self.encoder.emb.token_embeddings
        self.mlm_head = MLMHead(d_model=d_model, vocab_size=vocab_size, tie_weights=tied)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, position_ids=None):
        h = self.encoder(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
        )
        logits = self.mlm_head(h)
        return logits  # [B, T, V]

## MLM Loss (masked positions only)

We compute the MLM loss **only over masked tokens** (labels ≠ ignore_index), using summed cross-entropy divided by the number of masked tokens.

In [ ]:
def mlm_loss_mean(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    B, T, V = logits.shape
    logits_flat = logits.view(-1, V)
    labels_flat = labels.view(-1)
    loss_sum = F.cross_entropy(logits_flat, labels_flat, ignore_index=cfg.ignore_index, reduction="sum")

    n_masked = (labels_flat != cfg.ignore_index).sum().item()

    return loss_sum / max(n_masked, 1)


## MLM Evaluation Loop

Evaluate the MLM model by averaging the **summed masked loss** across a limited number of batches (to keep evaluation fast).

In [ ]:

@torch.no_grad()
def evaluate_mlm(model, loader, max_batches: int = None):
    model.eval()
    total_loss_sum = 0.0
    total_masked = 0

    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        position_ids = batch["position_ids"].to(device)

        logits = model(input_ids, attention_mask=attention_mask,
                       token_type_ids=token_type_ids, position_ids=position_ids)

        loss_sum = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1),
                                   ignore_index=cfg.ignore_index, reduction="sum")
        n_masked = (labels != cfg.ignore_index).sum().item()

        total_loss_sum += loss_sum.item()
        total_masked += n_masked

    return total_loss_sum / max(total_masked, 1)

## MLM Training Loop (dynamic masking)

Train the MLM model with AdamW, log training loss every `log_every` steps, and periodically evaluate on validation batches to track progress.

In [ ]:
def train_mlm(
    model,
    train_loader,
    valid_loader,
    epochs: int = 1,
    lr: float = 5e-4,
    weight_decay: float = 0.02,
    max_steps: int = 3000,
    log_every: int = 50,
    eval_batches: int = 50,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"step": [], "train_loss": [], "valid_loss": []}
    global_step = 0

    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        seen = 0

        for batch in tqdm(train_loader, desc=f"MLM Train (epoch {ep})", leave=False):
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            position_ids = batch["position_ids"].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids, position_ids=position_ids)
            loss = mlm_loss_mean(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running += loss.item()
            seen += 1
            global_step += 1

            if global_step % log_every == 0:
                train_loss_avg = running / max(seen, 1)
                val_loss = evaluate_mlm(model, valid_loader, max_batches=eval_batches)

                hist["step"].append(global_step)
                hist["train_loss"].append(train_loss_avg)
                hist["valid_loss"].append(val_loss)

                # math.exp(loss) gives Perplexity
                ppl = math.exp(min(val_loss, 20))
                print(f"Step {global_step:4d} | Train Loss: {train_loss_avg:.4f} | Val Loss: {val_loss:.4f} | Val PPL: {ppl:.1f}")

            if global_step >= max_steps:
                break

        if global_step >= max_steps:
            break

    return model, hist

seed_everything(cfg.seed)
mlm_model, mlm_hist = train_mlm(
    mlm_model,
    mlm_train_loader,
    mlm_valid_loader,
    epochs=2,
    lr=5e-4,
    max_steps=1000,
    log_every=100
)

## Train TinyBERT for MLM (dynamic masking)

We initialize the MLM model, measure an initial validation loss, then train for a small number of steps/epochs and re-evaluate to confirm learning progress.

In [ ]:
seed_everything(cfg.seed)

# 1) Build the MLM model
mlm_model = TinyBertForMLM(
    vocab_size=len(STOI),
    pad_id=PAD_ID,
    d_model=192,
    num_heads=4,
    num_layers=4,
    dropout=0.1,
    max_len=MAX_LEN_BERT,
).to(device)

# 2) Sanity check: initial validation loss (masked tokens only)
# For a random model with vocab size V, initial loss is roughly ln(V)
print("Initial valid MLM loss:", evaluate_mlm(mlm_model, mlm_valid_loader, max_batches=50))

# 3) Train
# Hyperparameters:
# - 1-2 epochs is usually enough for homework-scale datasets.
# - 2000 steps covers a significant chunk of data without taking hours.
mlm_model, mlm_hist = train_mlm(
    mlm_model,
    mlm_train_loader,
    mlm_valid_loader,
    epochs=1,
    lr=5e-4,
    weight_decay=0.02,
    max_steps=2000,
    log_every=100,
    eval_batches=50,
)

# 4) Final validation loss (after training)
print("Final valid MLM loss:", evaluate_mlm(mlm_model, mlm_valid_loader, max_batches=50))

## Visualize MLM training

Plot the training loss (running average) and validation loss over steps, and convert validation loss to an approximate perplexity via `exp(loss)` (clipped for numerical safety).

In [ ]:
# Plot MLM curves only if you have already run training and produced `mlm_hist`.
# TODO: Run the training cell above to create `mlm_hist`, then re-run this cell.

df_mlm = pd.DataFrame(mlm_hist)

plt.figure()
plt.plot(df_mlm["step"], df_mlm["train_loss"], label="train (avg since epoch start)")
plt.plot(df_mlm["step"], df_mlm["valid_loss"], label="valid (dynamic masking)")
plt.xlabel("Step")
plt.ylabel("MLM loss (masked positions only)")
plt.title("MLM training curve")
plt.legend()
plt.show()

plt.figure()
plt.plot(
    df_mlm["step"],
    [math.exp(min(x, 20)) for x in df_mlm["valid_loss"]],
    label="valid ppl ≈ exp(loss)"
)
plt.xlabel("Step")
plt.ylabel("Perplexity (approx, masked positions)")
plt.title("MLM validation perplexity (approx)")
plt.legend()
plt.show()

## TODO: Evaluate MLM Top-k Accuracy (masked positions only)

Compute masked-token Top-1 and Top-5 accuracy on the validation set to quantify how often the true token appears in the model’s top predictions.

In [ ]:
@torch.no_grad()
def mlm_topk_acc(model, loader, k: int = 1, max_batches: int = 50):
    model.eval()
    correct = 0
    total = 0

    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask=attention_mask)

        # Build mask for masked positions
        mask = (labels != cfg.ignore_index)
        if not mask.any():
            continue

        # Get top-k indices
        _, topk = logits[mask].topk(k, dim=-1) # [N_masked, k]
        targets = labels[mask].unsqueeze(-1)   # [N_masked, 1]

        correct += (topk == targets).any(dim=-1).sum().item()
        total += mask.sum().item()

    return correct / max(total, 1)

top1 = mlm_topk_acc(mlm_model, mlm_valid_loader, k=1)
top5 = mlm_topk_acc(mlm_model, mlm_valid_loader, k=5)
print(f"Valid MLM masked top-1 acc: {top1:.4f}")
print(f"Valid MLM masked top-5 acc: {top5:.4f}")

## TODO: Visualize MLM Encoder Attention (single layer/head)

Extract and plot one attention head from a chosen encoder layer to inspect what tokens the model attends to during MLM pretraining.


In [ ]:
@torch.no_grad()
def encoder_attention_map(model, loader, layer_to_show: int = 0, head_to_show: int = 0, ex_idx: int = 0, max_tokens: int = 40):
    model.eval()
    batch = next(iter(loader))

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    emb_out = model.encoder.emb(input_ids)
    _, attn = model.encoder.blocks[layer_to_show](emb_out, attention_mask=attention_mask, return_attn=True)

    L = int(batch["lengths"][ex_idx])
    L = min(L, max_tokens)

    matrix = attn[ex_idx, head_to_show, :L, :L].cpu().numpy()
    tokens = batch["tokens"][ex_idx][:L]

    plt.figure(figsize=(10, 8))
    sns.heatmap(matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis", annot=False)
    plt.title(f"Attention Map: Layer {layer_to_show}, Head {head_to_show}")
    plt.xlabel("Key Tokens")
    plt.ylabel("Query Tokens")
    plt.show()

encoder_attention_map(mlm_model, mlm_valid_loader, layer_to_show=0, head_to_show=0)
encoder_attention_map(mlm_model, mlm_valid_loader, layer_to_show=0, head_to_show=1)

## Concept Checks (answer in Markdown, no code)

1) **What is MLM actually optimizing?**  
Explain in words what the model is asked to predict, and why this is a self-supervised objective.
Answer=Masked Language Modeling (MLM) is a cloze-style task where the model is asked to predict the original identity of tokens that have been "hidden" or corrupted in an input sequence.What the model predicts: Specifically, the model outputs a probability distribution over the entire vocabulary for each masked position. It is optimizing the cross-entropy loss between its prediction and the actual ground-truth token that was originally there.The Objective: It forces the model to learn bidirectional context. To guess a missing word correctly, the model must understand the syntax (grammar) and semantics (meaning) of both the words that come before and the words that come after the mask.Why is this "Self-Supervised"?It is self-supervised because the labels are inherent in the data itself. You do not need humans to manually label sentences. You can take any raw text (like Wikipedia or books), programmatically hide $15\%$ of the words, and use the original words as the labels. This allows the model to learn from massive, unannotated datasets

2) **Dynamic masking vs. static masking:**  
What is dynamic masking, and why can it help training compared to precomputing a fixed masked dataset?
Answer=Dynamic Masking
The masking is applied on-the-fly every time a sequence is fed into the model (usually inside the collate_fn or the training loop).

Why it helps:

Data Augmentation: The model effectively sees different "versions" of the same sentence across different epochs. A word that is masked in Epoch 1 might be visible in Epoch 2, providing a much richer variety of contextual clues.

Better Generalization: It prevents the model from overfitting to a fixed set of masked tokens, forcing it to learn a more robust representation of the language that doesn't depend on which specific words happen to be missing in a single precomputed file.


## Part 9 — Pretraining Objective #2: Next Sentence Prediction (NSP)

So far we trained a **masked language model (MLM)** that learns to recover missing tokens from context.  
In this part we add a second classic pretraining signal from the original BERT recipe: **Next Sentence Prediction (NSP)**.

### What we build
We will construct **sentence pairs**:
- **Sentence A** = a real sentence from the dataset
- **Sentence B** = either the *true next* sentence (**IsNext = 1**) or a *random* sentence from elsewhere (**NotNext = 0**)

Then we train a single encoder to solve **two tasks at once**:
1. **MLM**: predict masked tokens (token-level classification over the vocabulary)
2. **NSP**: predict whether B follows A (binary classification from the `[CLS]` representation)

### Why this matters
- MLM encourages **local, token-level semantic understanding**.
- NSP encourages the model to capture **sentence-to-sentence coherence** and discourse-level signals.
- Multi-task training forces the encoder to learn representations useful at different granularity levels.

## Build NSP sentence pairs
We create positive (next-sentence) and negative (random-sentence) pairs for the NSP objective.

In [ ]:
class NSPPairDataset(Dataset):
    def __init__(self, hf_split, stoi: Dict[str, int], max_len: int = 256, p_is_next: float = 0.5):
        self.split = hf_split
        self.stoi = stoi
        self.max_len = max_len
        self.p_is_next = p_is_next
        self.n = len(hf_split)

    def _truncate_pair(self, a_tokens, b_tokens):
        # Total length must fit: [CLS] A [SEP] B [SEP]
        max_pair = self.max_len - 3
        a = list(a_tokens)
        b = list(b_tokens)

        # Truncate by removing tokens from the longer sequence until they fit
        while len(a) + len(b) > max_pair:
            if len(a) > len(b):
                a.pop()
            else:
                b.pop()
        return a, b

    def __len__(self):
        return self.n - 1

    def __getitem__(self, idx):
        tokens_a = self.split[idx]["tokens"]

        if random.random() < self.p_is_next:
            # Case 1: True next sentence
            tokens_b = self.split[idx + 1]["tokens"]
            is_next = 1
        else:
            # Case 2: Random sentence
            # Pick a random index j that is not idx or idx+1
            j = random.randrange(self.n)
            while j == idx or j == idx + 1:
                j = random.randrange(self.n)
            tokens_b = self.split[j]["tokens"]
            is_next = 0

        tokens_a, tokens_b = self._truncate_pair(tokens_a, tokens_b)

        return {
            "tokens_a": tokens_a,
            "tokens_b": tokens_b,
            "is_next": torch.tensor(is_next, dtype=torch.long)
        }

## Collate NSP pairs for MLM + NSP
Pack sentence pairs into one BERT-style sequence and prepare tensors needed for joint pretraining.

In [ ]:
def nsp_mlm_collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    input_ids_list = []
    token_type_list = []
    is_next_list = []
    tokens_debug = []

    for ex in batch:
        a = ex["tokens_a"]
        b = ex["tokens_b"]
        is_next_list.append(ex["is_next"])

        combined = [BERT_CLS] + a + [BERT_SEP] + b + [BERT_SEP]
        tokens_debug.append(combined)

        ids = [STOI.get(t, UNK_ID) for t in combined]
        input_ids_list.append(torch.tensor(ids, dtype=torch.long))

        tt_ids = [0] * (len(a) + 2) + [1] * (len(b) + 1)
        token_type_list.append(torch.tensor(tt_ids, dtype=torch.long))

    lengths = torch.tensor([len(x) for x in input_ids_list])
    max_batch_len = lengths.max().item()

    B = len(batch)
    input_ids = torch.full((B, max_batch_len), fill_value=PAD_ID, dtype=torch.long)
    token_type_ids = torch.zeros((B, max_batch_len), dtype=torch.long)
    attention_mask = torch.zeros((B, max_batch_len), dtype=torch.long)

    for i in range(B):
        L = lengths[i]
        input_ids[i, :L] = input_ids_list[i]
        token_type_ids[i, :L] = token_type_list[i]
        attention_mask[i, :L] = 1

    position_ids = torch.arange(max_batch_len).expand(B, max_batch_len)

    # Apply MLM masking (reusing logic from Part 8)
    masked_input_ids, mlm_labels = mask_tokens_bert_style(
        input_ids=input_ids.clone(),
        attention_mask=attention_mask,
        mlm_prob=0.15
    )

    return {
        "input_ids": masked_input_ids,
        "mlm_labels": mlm_labels,
        "nsp_labels": torch.stack(is_next_list),
        "token_type_ids": token_type_ids,
        "position_ids": position_ids,
        "attention_mask": attention_mask,
        "tokens": tokens_debug,
        "lengths": lengths
    }

## Add an NSP head for joint MLM + NSP pretraining
Extend the MLM encoder with a small classifier on the `[CLS]` representation to predict whether sentence B follows sentence A.

In [ ]:
# Note: You can modify this part if you need

class NSPHead(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.act = nn.Tanh()
        self.classifier = nn.Linear(d_model, 2)  # 0 = NotNext, 1 = IsNext
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, cls_hidden: torch.Tensor):
        x = self.act(self.dense(cls_hidden))
        return self.classifier(x)

class TinyBertForPretraining(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        d_model: int = 192,
        num_heads: int = 4,
        num_layers: int = 4,
        dropout: float = 0.1,
        max_len: int = 512,
    ):
        super().__init__()
        self.encoder = TinyBertEncoder(vocab_size, pad_id, d_model, num_heads, num_layers, dropout, max_len)

        tied = self.encoder.emb.token_embeddings
        self.mlm_head = MLMHead(d_model=d_model, vocab_size=vocab_size, tie_weights=tied)

        self.nsp_head = NSPHead(d_model=d_model)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, position_ids=None):
        h = self.encoder(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
        )
        mlm_logits = self.mlm_head(h)
        cls_hidden = h[:, 0, :]
        nsp_logits = self.nsp_head(cls_hidden)
        return mlm_logits, nsp_logits

## Initialize the joint pretraining model (optional warm start)
Create the MLM+NSP model and (optionally) reuse the encoder/MLM weights learned in Part 8.

In [ ]:
seed_everything(cfg.seed)

pretrain_model = TinyBertForPretraining(
    vocab_size=len(STOI),
    pad_id=PAD_ID,
    d_model=192,
    num_heads=4,
    num_layers=4,
    dropout=0.1,
    max_len=MAX_LEN_BERT,
).to(device)

if "mlm_model" in globals():
    print("Loading weights from Part 8 MLM model...")
    pretrain_model.encoder.load_state_dict(mlm_model.encoder.state_dict())
    pretrain_model.mlm_head.load_state_dict(mlm_model.mlm_head.state_dict(), strict=False)

print("Ready.")

## Joint Pretraining: Evaluation + Training Loop (MLM + NSP)
Implement the evaluation and training routines for the combined MLM+NSP objective, then track validation metrics during training.

In [ ]:
@torch.no_grad()
def eval_pretrain(model, loader, max_batches: int = 100):
    model.eval()
    mlm_loss_sum, n_masked = 0.0, 0
    nsp_loss_sum, n_examples, nsp_correct = 0.0, 0, 0

    for bi, batch in enumerate(loader):
        if bi >= max_batches: break

        input_ids = batch["input_ids"].to(device)
        mlm_labels = batch["mlm_labels"].to(device)
        nsp_labels = batch["nsp_labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        tt_ids = batch["token_type_ids"].to(device)
        pos_ids = batch["position_ids"].to(device)

        mlm_logits, nsp_logits = model(input_ids, attention_mask, tt_ids, pos_ids)

        # MLM Loss
        mlm_loss = F.cross_entropy(mlm_logits.view(-1, mlm_logits.size(-1)),
                                   mlm_labels.view(-1), ignore_index=cfg.ignore_index, reduction="sum")
        mlm_loss_sum += mlm_loss.item()
        n_masked += (mlm_labels != cfg.ignore_index).sum().item()

        # NSP Loss & Acc
        nsp_loss = F.cross_entropy(nsp_logits, nsp_labels, reduction="sum")
        nsp_loss_sum += nsp_loss.item()
        n_examples += nsp_labels.size(0)
        nsp_correct += (nsp_logits.argmax(dim=-1) == nsp_labels).sum().item()

    return {
        "mlm_loss": mlm_loss_sum / max(n_masked, 1),
        "nsp_loss": nsp_loss_sum / max(n_examples, 1),
        "nsp_acc": nsp_correct / max(n_examples, 1)
    }



In [ ]:
def train_pretrain(
    model, train_loader, valid_loader,
    lr: float = 5e-4, weight_decay: float = 0.02,
    max_steps: int = 300, log_every: int = 50,
    grad_clip: float = 1.0, valid_batches: int = 100
):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.train()

    global_step = 0
    running_mlm, running_nsp = 0.0, 0.0

    while global_step < max_steps:
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            mlm_labels = batch["mlm_labels"].to(device)
            nsp_labels = batch["nsp_labels"].to(device)

            opt.zero_grad()
            mlm_logits, nsp_logits = model(input_ids, batch["attention_mask"].to(device),
                                           batch["token_type_ids"].to(device), batch["position_ids"].to(device))

            mlm_loss = mlm_loss_mean(mlm_logits, mlm_labels)
            nsp_loss = F.cross_entropy(nsp_logits, nsp_labels)
            total_loss = mlm_loss + nsp_loss

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            running_mlm += mlm_loss.item()
            running_nsp += nsp_loss.item()
            global_step += 1

            if global_step % log_every == 0:
                val = eval_pretrain(model, valid_loader)
                print(f"Step {global_step} | MLM Loss: {val['mlm_loss']:.3f} | NSP Acc: {val['nsp_acc']:.2%}")
                running_mlm, running_nsp = 0.0, 0.0

            if global_step >= max_steps: break
    return model

In [ ]:
# Evaluate before training
print("Initial valid metrics:", eval_pretrain(pretrain_model, nsp_valid_loader, max_batches=100))

# Train for ~1 epoch worth of steps (or as configured)
pretrain_model, hist9 = train_pretrain(
    pretrain_model,
    nsp_train_loader,
    nsp_valid_loader,
    max_steps=len(nsp_train_loader) + 1,
    log_every=50,
)

# Evaluate after training
print("Final valid metrics:", eval_pretrain(pretrain_model, nsp_valid_loader, max_batches=100))

## Visualize MLM + NSP Pretraining Curves
Plot the logged training/validation MLM loss, NSP loss/accuracy, and MLM perplexity as a function of update steps.

In [ ]:
import matplotlib.pyplot as plt

# Plotting the pretraining history
steps = pretrain_hist["step"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) MLM Loss Curve
axes[0, 0].plot(steps, pretrain_hist["train_mlm"], label="Train MLM Loss", marker='o', markersize=4)
axes[0, 0].plot(steps, pretrain_hist["valid_mlm"], label="Valid MLM Loss", marker='x', markersize=4)
axes[0, 0].set_title("MLM Loss (Cross-Entropy)")
axes[0, 0].set_xlabel("Step")
axes[0, 0].legend()
axes[0, 0].grid(True)

# 2) NSP Loss Curve
axes[0, 1].plot(steps, pretrain_hist["train_nsp"], label="Train NSP Loss", color='orange', marker='o', markersize=4)
axes[0, 1].plot(steps, pretrain_hist["valid_nsp"], label="Valid NSP Loss", color='red', marker='x', markersize=4)
axes[0, 1].set_title("NSP Loss (Cross-Entropy)")
axes[0, 1].set_xlabel("Step")
axes[0, 1].legend()
axes[0, 1].grid(True)

# 3) NSP Accuracy Curve
axes[1, 0].plot(steps, pretrain_hist["valid_nsp_acc"], label="Valid NSP Acc", color='green', marker='s')
axes[1, 0].set_title("NSP Accuracy")
axes[1, 0].set_xlabel("Step")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(True)

# 4) MLM Perplexity Curve
# Perplexity = exp(Loss)
axes[1, 1].plot(steps, pretrain_hist["valid_mlm_ppl"], label="Valid MLM PPL", color='purple', marker='^')
axes[1, 1].set_title("MLM Perplexity")
axes[1, 1].set_xlabel("Step")
axes[1, 1].set_ylabel("PPL")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## Inspect One NSP Pair (Qualitative Check)
Take one batch from the NSP validation loader, split it into sentence A/B using `[SEP]`, and print the NSP label + tokens.

In [ ]:

batch = next(iter(nsp_valid_loader))

ex_idx = 0
tokens = batch["tokens"][ex_idx]
label = batch["nsp_labels"][ex_idx].item()
L = batch["lengths"][ex_idx].item()

actual_tokens = tokens[:L]

sep_indices = [i for i, t in enumerate(actual_tokens) if t == BERT_SEP]

if len(sep_indices) >= 2:
    sent_a_tokens = actual_tokens[1 : sep_indices[0]]
    sent_b_tokens = actual_tokens[sep_indices[0] + 1 : sep_indices[1]]

    sentence_a = " ".join(sent_a_tokens)
    sentence_b = " ".join(sent_b_tokens)

    print(f"NSP Label: {label} ({'IsNext' if label == 1 else 'NotNext'})")
    print("-" * 30)
    print(f"Sentence A: {sentence_a}")
    print("-" * 30)
    print(f"Sentence B: {sentence_b}")
else:
    print("Error: Could not find expected [SEP] tokens in the sequence.")

## Concept Checks (answer in Markdown, no code)

1. **Data construction:**  
   In our NSP dataset, how are positive pairs (IsNext) and negative pairs (NotNext) created?  
   What role does `p_is_next` play?

   Answer=Positive Pairs (IsNext / Label 1): Created by taking two sentences that actually appear consecutively in the original text (sentence $A$ followed by sentence $B$).Negative Pairs (NotNext / Label 0): Created by taking a real sentence $A$ and pairing it with a random sentence $B$ sampled from a completely different part of the document or a different document entirely.Role of p_is_next: This hyperparameter controls the sampling balance. It represents the probability that a given example in the dataset will be a positive pair. Typically set to $0.5$, it ensures the model sees an equal number of "correct" and "random" sequences, preventing the model from developing a bias toward one class.

2. **Loss design:**  
   We train with a combined loss:

   $$
   \mathcal{L} = \mathcal{L}_{\mathrm{MLM}} + \mathcal{L}_{\mathrm{NSP}}.
   $$

   What does each term supervise, and which model outputs are used to compute them?

   Answer=1. $\mathcal{L}_{\mathrm{MLM}}$ (Masked Language Modeling Loss)What it supervises: This term supervises the model's token-level understanding and its ability to capture bidirectional context. It forces the model to learn grammar, vocabulary, and semantic relationships between words.Model outputs used: It uses the hidden states of the masked tokens from the final layer of the encoder. These hidden states are passed through the MLMHead, which produces a probability distribution over the entire vocabulary for each masked position.Computation: It is typically computed using Cross-Entropy loss between the predicted vocabulary logits and the actual identities of the tokens that were masked out.2. $\mathcal{L}_{\mathrm{NSP}}$ (Next Sentence Prediction Loss)What it supervises: This term supervises the model's sentence-level coherence and its ability to understand the relationship between two distinct segments of text. This is crucial for downstream tasks like Question Answering (where you have a context and a question) or Natural Language Inference.Model outputs used: It uses the hidden state of the [CLS] token from the final layer. Since the [CLS] token's representation is attended to by every other token in the sequence (via the self-attention mechanism), it serves as a "pooled" summary of the entire sentence pair. This vector is passed through the NSPHead (a simple linear classifier).Computation: It is computed as a binary Cross-Entropy loss (IsNext vs. NotNext) comparing the predicted probability to the ground-truth relationship of the sentence pair.

# Part 10 — Fine-tuning Results

In Parts 8–9 we trained the same tiny BERT-style encoder with self-supervised objectives (MLM, then MLM+NSP).  
In this part we **fine-tune for NER** and compare three initializations:

- **Scratch:** encoder starts from random weights.
- **MLM-init:** encoder starts from the Part 8 MLM-pretrained weights.
- **MLM+NSP-init:** encoder starts from the Part 9 MLM+NSP-pretrained weights.

We will evaluate each run with the same NER metrics (loss, token accuracy, and entity-level precision/recall/F1) and then do a small qualitative check by printing token-level predictions for one validation example.

## Select a Compatible Vocabulary for Fine-Tuning  
We must ensure the fine-tuning vocabulary size matches the encoder’s token embedding table to avoid shape-mismatch errors.


In [ ]:
def pick_vocab_for_encoder():
    """
    Returns (FT_STOI, FT_ITOS) such that len(FT_STOI) matches the encoder vocab size.
    Priority:
      1) If pretrain_model exists -> match its encoder embedding size
      2) elif mlm_model exists -> match its encoder embedding size
      3) else fallback to (stoi, itos)
    """
    # candidates we might have
    candidates = []
    if "mlm_stoi" in globals() and "mlm_itos" in globals():
        candidates.append(("mlm", mlm_stoi, mlm_itos))
    if "stoi" in globals() and "itos" in globals():
        candidates.append(("ner", stoi, itos))

    # determine target vocab size
    target = None
    if "pretrain_model" in globals():
        target = pretrain_model.encoder.emb.token_embeddings.num_embeddings
        src = "pretrain_model"
    elif "mlm_model" in globals():
        target = mlm_model.encoder.emb.token_embeddings.num_embeddings
        src = "mlm_model"
    else:
        target = None
        src = "none"

    if target is None:
        print("No pretrained encoder found; using (stoi, itos).")
        return stoi, itos

    for name, S, I in candidates:
        if len(S) == target:
            print(f"Using vocab '{name}' because it matches encoder vocab_size={target} (source={src}).")
            return S, I

    # If none match, fail loudly with a helpful message
    raise ValueError(
        f"No available vocab matches encoder vocab size={target}. "
        f"Available sizes: {[ (name, len(S)) for name,S,_ in candidates ]}. "
        f"Fix by using the same vocab in Parts 8–10."
    )

FT_STOI, FT_ITOS = pick_vocab_for_encoder()

FT_PAD_ID  = FT_STOI[cfg.pad_token]
FT_UNK_ID  = FT_STOI[cfg.unk_token]
FT_CLS_ID  = FT_STOI[BERT_CLS]
FT_SEP_ID  = FT_STOI[BERT_SEP]

## NER Dataset with BERT-Style Special Tokens  
Wrap each sentence as `[CLS] tokens [SEP]` and align labels by ignoring the special-token positions.

In [ ]:
class NERBertInputDataset(Dataset):
    def __init__(self, hf_split, stoi: Dict[str,int], max_len: int = 256):
        self.split = hf_split
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        ex = self.split[idx]
        tokens = ex["tokens"]
        labels = ex["ner_tags"]

        # Fit: [CLS] + tokens + [SEP]
        max_tokens = self.max_len - 2
        tokens = tokens[:max_tokens]
        labels = labels[:max_tokens]

        bert_tokens = [BERT_CLS] + tokens + [BERT_SEP]
        input_ids = [self.stoi.get(t, self.stoi[cfg.unk_token]) for t in bert_tokens]

        # Labels: ignore CLS and SEP positions
        bert_labels = [cfg.ignore_index] + labels + [cfg.ignore_index]

        token_type_ids = [0] * len(input_ids)  # single segment

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "labels": torch.tensor(bert_labels, dtype=torch.long),
            "tokens": bert_tokens,
        }

## Collate Function + Fine-tuning DataLoaders  
Implement padding + attention masks (and position ids) to build BERT-style batches for NER fine-tuning.

In [ ]:
# TODO: Implement a BERT-style collate function for token classification.
# Requirements:
# 1) Pad input_ids with FT_PAD_ID to [B, T_max]
# 2) Build token_type_ids (already provided per example) and pad to [B, T_max]
# 3) Build attention_mask with 1 for real tokens, 0 for padding
# 4) Pad labels with cfg.ignore_index to [B, T_max]
# 5) Build position_ids as [0..T_max-1] repeated for each batch row
# 6) Return a dict with:
#    "input_ids", "token_type_ids", "position_ids", "attention_mask", "labels",
#    "tokens" (list of token strings per example), "lengths" (original lengths)

def bert_ner_collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    lengths = [len(ex["input_ids"]) for ex in batch]
    max_len = max(lengths)
    batch_size = len(batch)

    input_ids = torch.full((batch_size, max_len), FT_PAD_ID, dtype=torch.long)
    token_type_ids = torch.zeros((batch_size, max_len), dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
    labels = torch.full((batch_size, max_len), cfg.ignore_index, dtype=torch.long)

    for i, ex in enumerate(batch):
        l = lengths[i]
        input_ids[i, :l] = ex["input_ids"]
        token_type_ids[i, :l] = ex["token_type_ids"]
        attention_mask[i, :l] = 1
        labels[i, :l] = ex["labels"]

    position_ids = torch.arange(max_len, dtype=torch.long).unsqueeze(0).expand(batch_size, max_len)

    return {
        "input_ids": input_ids,
        "token_type_ids": token_type_ids,
        "position_ids": position_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "tokens": [ex["tokens"] for ex in batch],
        "lengths": torch.tensor(lengths, dtype=torch.long),
    }

# Build loaders
ft_train_loader = DataLoader(ft_train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=bert_ner_collate_fn)
ft_valid_loader = DataLoader(ft_valid_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=bert_ner_collate_fn)
ft_test_loader  = DataLoader(ft_test_ds,  batch_size=cfg.batch_size, shuffle=False, collate_fn=bert_ner_collate_fn)

# Sanity check
sample_batch = next(iter(ft_train_loader))
print(f"Batch shapes: IDs {sample_batch['input_ids'].shape}, Labels {sample_batch['labels'].shape}")

## Token Classification Head  
Wrap an encoder with a dropout + linear layer to predict a NER label for every token.

In [ ]:
class TinyBertForTokenClassification(nn.Module):
    def __init__(self, encoder: nn.Module, d_model: int, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, position_ids=None):
        h = self.encoder(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
        )
        logits = self.classifier(self.drop(h))
        return logits  # [B,T,C]

## Fine-tuning Loops (Train + Evaluate)  
Implement the training and evaluation routines for token-level classification during fine-tuning.

In [ ]:
@torch.no_grad()
def evaluate_ner_ft(model, loader):
    model.eval()
    total_loss = 0
    total_tokens = 0
    y_true_all = []
    y_pred_all = []
    criterion = nn.CrossEntropyLoss(ignore_index=cfg.ignore_index, reduction='sum')

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["attention_mask"].to(device)

        logits = model(input_ids, attention_mask=mask,
                       token_type_ids=batch["token_type_ids"].to(device),
                       position_ids=batch["position_ids"].to(device))

        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        num_valid_tokens = (labels != cfg.ignore_index).sum().item()
        total_loss += loss.item()
        total_tokens += num_valid_tokens

        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        labels_np = labels.cpu().numpy()

        for b in range(len(labels_np)):
            true_tags = []
            pred_tags = []
            for t in range(len(labels_np[b])):
                if labels_np[b][t] != cfg.ignore_index:
                    true_tags.append(id2label[labels_np[b][t]])
                    pred_tags.append(id2label[preds[b][t]])
            y_true_all.append(true_tags)
            y_pred_all.append(pred_tags)

    return {
        "loss": total_loss / max(1, total_tokens),
        "f1": f1_score(y_true_all, y_pred_all),
        "precision": precision_score(y_true_all, y_pred_all),
        "recall": recall_score(y_true_all, y_pred_all)
    }

def train_one_epoch_ner_ft(model, loader, optimizer, grad_clip=1.0):
    model.train()
    total_loss = 0
    total_tokens = 0
    criterion = nn.CrossEntropyLoss(ignore_index=cfg.ignore_index, reduction='mean')

    for batch in tqdm(loader, desc="Training NER", leave=False):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask=batch["attention_mask"].to(device),
                       token_type_ids=batch["token_type_ids"].to(device),
                       position_ids=batch["position_ids"].to(device))

        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        num_valid_tokens = (labels != cfg.ignore_index).sum().item()
        total_loss += loss.item() * num_valid_tokens
        total_tokens += num_valid_tokens

    return total_loss / max(1, total_tokens)

## Infer Encoder Hyperparameters from Pretraining  
Automatically detect the encoder’s shape (dimension/length/depth) so fine-tuning matches the pretrained weights.

In [ ]:
# Infer encoder config (d_model, heads, layers, max_len) from pretrained if possible
ENC_D_MODEL = 192
ENC_HEADS = 4
ENC_LAYERS = 4
ENC_MAXLEN = MAX_LEN_BERT

if "pretrain_model" in globals():
    ENC_D_MODEL = pretrain_model.encoder.emb.token_embeddings.embedding_dim
    ENC_MAXLEN  = pretrain_model.encoder.emb.position_embeddings.num_embeddings
    ENC_LAYERS  = len(pretrain_model.encoder.blocks)
elif "mlm_model" in globals():
    ENC_D_MODEL = mlm_model.encoder.emb.token_embeddings.embedding_dim
    ENC_MAXLEN  = mlm_model.encoder.emb.position_embeddings.num_embeddings
    ENC_LAYERS  = len(mlm_model.encoder.blocks)

## Fine-tuning Experiment Helper Function  
Implement one end-to-end fine-tuning run (init → train/early-stop → test) so we can compare scratch vs. pretrained encoders fairly.

In [ ]:
def finetune_experiment(init_name, encoder_init, d_model, lr=3e-4, wd=0.01, epochs=5, patience=2):
    seed_everything(cfg.seed)

    # 1) Build the encoder
    encoder = TinyBertEncoder(
        vocab_size=len(FT_STOI),
        d_model=d_model,
        n_heads=ENC_HEADS,
        n_layers=ENC_LAYERS,
        max_len=ENC_MAXLEN
    )

    if encoder_init == "mlm":
        assert "mlm_model" in globals(), "MLM model missing"
        encoder.load_state_dict(mlm_model.encoder.state_dict())
    elif encoder_init == "mlm_nsp":
        assert "pretrain_model" in globals(), "Pretrain model missing"
        encoder.load_state_dict(pretrain_model.encoder.state_dict())

    model = TinyBertForTokenClassification(encoder, d_model, num_labels=len(id2label)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    best_val_f1 = -1.0
    best_state = None
    patience_ctr = 0
    hist = {"train_loss": [], "val_loss": [], "val_f1": []}

    for ep in range(1, epochs + 1):
        train_loss = train_one_epoch_ner_ft(model, ft_train_loader, optimizer)
        metrics = evaluate_ner_ft(model, ft_valid_loader)

        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(metrics["loss"])
        hist["val_f1"].append(metrics["f1"])

        print(f"[{init_name}] Ep {ep} | Train Loss: {train_loss:.4f} | Val F1: {metrics['f1']:.4f}")

        if metrics["f1"] > best_val_f1:
            best_val_f1 = metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience: break

    model.load_state_dict(best_state)
    test_metrics = evaluate_ner_ft(model.to(device), ft_test_loader)
    return model, hist, best_val_f1, test_metrics

## Fine-tuning Comparison (Scratch vs. Pretraining)  
Run the same NER fine-tuning pipeline with different encoder initializations and summarize results in one table.

In [ ]:

D_MODEL = ENC_D_MODEL

results_data = []
model_scr, hist_scr, f1_scr, test_scr = finetune_experiment("Scratch", "scratch", D_MODEL)
results_data.append(["Scratch", f1_scr, test_scr["f1"], test_scr["loss"]])

if "mlm_model" in globals():
    model_mlm, hist_mlm, f1_mlm, test_mlm = finetune_experiment("MLM-init", "mlm", D_MODEL)
    results_data.append(["MLM-init", f1_mlm, test_mlm["f1"], test_mlm["loss"]])

if "pretrain_model" in globals():
    model_pre, hist_pre, f1_pre, test_pre = finetune_experiment("MLM+NSP-init", "mlm_nsp", D_MODEL)
    results_data.append(["MLM+NSP-init", f1_pre, test_pre["f1"], test_pre["loss"]])

df = pd.DataFrame(results_data, columns=["Init", "Best Val F1", "Test F1", "Test Loss"])
display(df.sort_values("Test F1", ascending=False))

@torch.no_grad()
def pretty_print_ner_tokens(tokens, true_ids, pred_ids, max_tokens=60):
    print(f"{'TOKEN':<15} | {'TRUE':<10} | {'PRED':<10} | {'MATCH'}")
    print("-" * 50)
    for i in range(min(len(tokens), max_tokens)):
        if true_ids[i] == cfg.ignore_index: continue
        t, tr, pr = tokens[i], id2label[true_ids[i]], id2label[pred_ids[i]]
        match = "✓" if tr == pr else "✗"
        print(f"{t:<15} | {tr:<10} | {pr:<10} | {match}")

batch = next(iter(ft_valid_loader))
model_scr.eval() # Pick one model
logits = model_scr(batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
preds = torch.argmax(logits, dim=-1)[0].cpu().numpy()
pretty_print_ner_tokens(batch["tokens"][0], batch["labels"][0].numpy(), preds)

## Training Curves  
Plot train/validation loss curves to compare convergence across different initialization strategies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_hist(hist, title):
    epochs = range(1, len(hist["train_loss"]) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, hist["train_loss"], label="Train Loss", marker='o')
    plt.plot(epochs, hist["val_loss"], label="Val Loss", marker='x')
    plt.title(f"Training Curves: {title}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_hist(hist_scr, "Scratch Initialization")

if "hist_mlm" in locals():
    plot_hist(hist_mlm, "MLM Initialization")

if "hist_pre" in locals():
    plot_hist(hist_pre, "MLM+NSP Initialization")

## Fine-tuning Comparison Plots  
Compare validation F1 curves (raw and best-so-far) across different encoder initializations.

In [ ]:

hists = {"Scratch": hist_scr}
if "hist_mlm" in locals():
    hists["MLM-init"] = hist_mlm
if "hist_pre" in locals():
    hists["MLM+NSP-init"] = hist_pre

plt.figure(figsize=(10, 6))
for name, h in hists.items():
    f1_key = "val_f1" if "val_f1" in h else "val_F1"
    epochs = range(1, len(h[f1_key]) + 1)
    plt.plot(epochs, h[f1_key], label=name, marker='o')

plt.title("Validation F1 Comparison")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
for name, h in hists.items():
    f1_key = "val_f1" if "val_f1" in h else "val_F1"
    epochs = range(1, len(h[f1_key]) + 1)
    best_so_far = np.maximum.accumulate(h[f1_key])
    plt.plot(epochs, best_so_far, label=f"{name} (Best)", linestyle='--')

plt.title("Best-so-far Validation F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.legend()
plt.grid(True)
plt.show()

## Qualitative Inspection: Token-level Predictions  
Print one validation example with true vs. predicted NER tags for the best-performing fine-tuned model.

In [ ]:
# ------------------------------------------------------------
# TODO: Qualitative inspection of NER predictions (token-level).
#
# Tasks:
#   1) Implement `pretty_print_ner_tokens(tokens, true_ids, pred_ids, max_tokens=...)`
#      to print TOKEN / TRUE / PRED and a simple correctness marker.
#   2) Choose a model to inspect WITHOUT hard-coding a specific winner.
#      Suggested options:
#        - pick by a chosen criterion (e.g., best validation F1 saved during training),
#        - or pick a fixed run name ("Scratch", "MLM-init", "MLM+NSP-init") if it exists.
#   3) Take one batch from `ft_valid_loader`, run the selected model, and print:
#        - the run name
#        - token-level true vs predicted tags for the first example in the batch
# Notes:
#   - Skip positions where label == cfg.ignore_index.
#   - Use id2label[...] to map ids to tag strings.
# ------------------------------------------------------------

@torch.no_grad()
def pretty_print_ner_tokens(tokens, true_ids, pred_ids, max_tokens=60):
    header = f"{'TOKEN':<18} | {'TRUE TAG':<10} | {'PRED TAG':<10} | {'MATCH'}"
    print(header)
    print("-" * len(header))

    count = 0
    for t, tr_id, pr_id in zip(tokens, true_ids, pred_ids):
        if tr_id == cfg.ignore_index:
            continue

        tr_tag = id2label[tr_id.item()]
        pr_tag = id2label[pr_id.item()]
        match = "✓" if tr_tag == pr_tag else "✗"

        print(f"{t:<18} | {tr_tag:<10} | {pr_tag:<10} | {match}")

        count += 1
        if count >= max_tokens:
            break

best_model_name = "Scratch"
best_model = model_scr
if "model_pre" in locals():
    best_model_name = "MLM+NSP-init"
    best_model = model_pre
elif "model_mlm" in locals():
    best_model_name = "MLM-init"
    best_model = model_mlm

print(f"Inspecting predictions for model: {best_model_name}\n")

best_model.eval()
batch = next(iter(ft_valid_loader))
input_ids = batch["input_ids"].to(device)
mask = batch["attention_mask"].to(device)
labels = batch["labels"]
logits = best_model(input_ids, attention_mask=mask)
preds = torch.argmax(logits, dim=-1).cpu()
pretty_print_ner_tokens(batch["tokens"][0], labels[0], preds[0])

## Concept Checks (answer in Markdown, no code)

1. **Why might MLM pretraining help NER?**  
   Explain the kind of information MLM forces the encoder to learn, and why that information could transfer to token-level labeling.

   Answer=What it learns: To fill in a mask, the model must learn syntax (is the missing word a verb or a noun?), morphology (should it be plural or past tense?), and semantics (does "Apple" refer to the fruit or the tech company based on the surrounding context?).

Transfer to NER: Named Entity Recognition is a token-level task that relies heavily on these same signals. For example, in the sentence "I went to Amazon to see the river," the MLM-trained encoder recognizes that the surrounding words "river" and "the" imply a geographical location. This "contextual awareness" transfers directly to NER, allowing the model to classify tokens more accurately than a scratch-initialized model that has to learn both English grammar and NER labels simultaneously.

4. **Interpret the curves:** If the pretrained model reaches higher validation F1 earlier, what does that suggest?  
   Relate your answer to optimization and sample efficiency.

   Answer=Better Optimization (Starting Point): The model doesn't start from a "random" point in the weight space. It starts with a pretrained initialization where the weights already "know" how to extract useful features from text. This places the model in a better basin of the loss landscape, making the fine-tuning process a matter of "refining" rather than "learning from scratch."

Sample Efficiency: The model is more sample efficient, meaning it requires significantly fewer labeled examples to reach a certain performance threshold. Because it already understands language structure, it can "connect the dots" between the NER labels and the text much faster, effectively doing more with less data.

# Part 11 — Fine-tuning a Pretrained Transformer for NER

In this part we switch from our **from-scratch encoders** to a **pretrained Transformer** (e.g., DistilBERT/BERT) and fine-tune it for **Named Entity Recognition** using the Hugging Face `Trainer` API.

Why this matters:
- Pretraining gives strong language representations “for free”, so fine-tuning often reaches high NER performance with much less task-specific training.
- The main technical challenge for token classification is **label alignment**: our dataset labels are word-level, but the tokenizer produces **subword tokens**.
- We will also test **data efficiency** by training on different fractions of the training set and comparing Test F1.

## Choose Pretrained Backbone

Select a HuggingFace checkpoint and load its tokenizer for the pretrained NER fine-tuning experiments.

In [ ]:
MODEL_NAME = "distilbert-base-cased"
# MODEL_NAME = "bert-base-cased"  # stronger but heavier

print("Using:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

print("num_labels:", num_labels)
print("example label:", id2label[12])

## Sanity Check: Labels Are Available

Quickly verify that the label mapping and label count are loaded from earlier parts.

In [ ]:
# Ensure these exist from earlier parts
print("num_labels:", num_labels)
print("example label:", id2label[13])

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LEN_BERT
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(cfg.ignore_index)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(cfg.ignore_index)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized

## Tokenize and Align Word-Level Labels

Convert word-level NER tags into token-level labels after subword tokenization.

In [ ]:
hf_ner = DatasetDict({
    "train": ds["train"],
    "validation": ds["validation"],
    "test": ds["test"],
})

tok_ner = hf_ner.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=hf_ner["train"].column_names,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print(tok_ner)

## Trainer Metrics (Entity-Level)

Compute precision/recall/F1 from token-level predictions while ignoring special/padded positions.

In [ ]:
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true_all, y_pred_all = [], []
    for lab_seq, pred_seq in zip(labels, preds):
        true_tags, pred_tags = [], []
        for l, p in zip(lab_seq, pred_seq):
            if l != cfg.ignore_index:
                true_tags.append(id2label[int(l)])
                pred_tags.append(id2label[int(p)])
        y_true_all.append(true_tags)
        y_pred_all.append(pred_tags)

    return {
        "precision": precision_score(y_true_all, y_pred_all),
        "recall": recall_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all),
    }



## TrainingArguments + Trainer Setup

Create version-safe HuggingFace `TrainingArguments` and `Trainer` kwargs (tokenizer/processing_class) for fine-tuning.

In [ ]:
def make_training_args(output_dir, seed, num_train_epochs, **extra_kwargs):
    kwargs = dict(
        output_dir=output_dir,
        overwrite_output_dir=True,
        logging_steps=50,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=seed,
        data_seed=seed,
        save_total_limit=1,
        per_device_train_batch_size=extra_kwargs.get("train_bs", 16),
        per_device_eval_batch_size=extra_kwargs.get("eval_bs", 32),
        num_train_epochs=num_train_epochs,
        learning_rate=extra_kwargs.get("learning_rate", 2e-5),
        weight_decay=0.01,
        warmup_ratio=0.1,
        fp16=torch.cuda.is_available(),
        eval_strategy="steps",
        eval_steps=200,
        save_steps=200,
        save_strategy="steps"
    )
    return TrainingArguments(**kwargs)

## Tokenize + align word-level labels to subword tokens

Convert word-level NER tags into token-level labels using `word_ids()` and `ignore_index` for non-first subwords.

In [ ]:
def plot_log_history(log_history, title_prefix=""):
    steps_train, loss_train = [], []
    steps_eval, f1_eval = [], []

    for item in log_history:
        if "loss" in item and "eval_loss" not in item and "step" in item:
            steps_train.append(item["step"])
            loss_train.append(item["loss"])
        if "eval_f1" in item and "step" in item:
            steps_eval.append(item["step"])
            f1_eval.append(item["eval_f1"])

    if steps_train:
        plt.figure()
        plt.plot(steps_train, loss_train)
        plt.xlabel("Step")
        plt.ylabel("Train loss")
        plt.title(f"{title_prefix} Train loss vs step")
        plt.show()

    if steps_eval:
        plt.figure()
        plt.plot(steps_eval, f1_eval, marker="o")
        plt.xlabel("Step")
        plt.ylabel("Validation F1")
        plt.title(f"{title_prefix} Val F1 vs step")
        plt.show()

## Build pretrained token-classification model

Load a pretrained backbone and adapt it to `num_labels` for NER.

In [ ]:
def build_hf_ner_model():
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    model.config.output_attentions = False
    model.config.output_hidden_states = False
    return model

## Train on a fraction of the data

Subsample the training set, fine-tune, and return validation/test metrics and logs.

In [ ]:


def run_data_fraction(frac, seed=42, num_train_epochs=3.0):
    # Subset selection
    n = len(tok_ner["train"])
    k = int(frac * n)
    sub_train = tok_ner["train"].shuffle(seed=seed).select(range(k))

    model = build_hf_ner_model()
    args = make_training_args(f"output_{frac}", seed, num_train_epochs)

    trainer = Trainer(
        **make_trainer_kwargs(model, args, sub_train, tok_ner["validation"])
    )

    trainer.train()
    test_metrics = trainer.evaluate(tok_ner["test"], metric_key_prefix="test")

    return test_metrics["test_f1"], test_metrics["test_loss"], trainer.state.log_history


## Data efficiency sweep

Run the same pretrained model on multiple training fractions and plot Test F1 vs fraction.

In [ ]:

# Execution
fractions = [0.10, 0.25, 1.00]
rows = []

for f in fractions:
    f1, loss, history = run_data_fraction(f, seed=cfg.seed)
    rows.append({"Fraction": f, "Test F1": f1, "Test Loss": loss})

frac_df = pd.DataFrame(rows)
print(frac_df)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(frac_df["Fraction"], frac_df["Test F1"], marker='o', linestyle='-', color='teal', linewidth=2)

plt.title("Scaling Analysis: Test F1 vs. Training Data Fraction", fontsize=14)
plt.xlabel("Fraction of Training Data used for Fine-tuning", fontsize=12)
plt.ylabel("Test F1 Score", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(fractions)  # Ensure the x-axis matches our experiment points [0.10, 0.25, 1.00]
plt.ylim(min(frac_df["Test F1"]) - 0.05, 1.0) # Adjust Y-axis to see details

plt.tight_layout()
plt.show()

for frac in fractions:
    if frac in logs_by_frac:
        plot_log_history(logs_by_frac[frac], title=f"DistilBERT Training (Fraction={frac:.0%})")

In [ ]:
for f in fractions:
    plot_log_history(logs_by_frac[f], title_prefix=f"{MODEL_NAME} ({int(f*100)}% data)")

## Concept Checks (answer in Markdown, no code)

1. **Subword label alignment:**  
   Suppose a word is split into subwords, e.g. `["play", "##ing"]`.  
   - Which token(s) should receive the original word label? First token ("play")
   - Which token(s) should receive `ignore_index`?  Subsequent tokens ("##ing")
   - Why?Metric Consistency: NER is traditionally a word-level task. If we labeled every subword, a word split into 5 pieces would weigh 5x more in the loss function than a short word, biasing the model toward long words.

Simplifying Predictions: During inference, we typically care about the entity type of the whole word. Labeling only the first subword allows the model to make a single "decision" for the word's meaning at its start, while the later subwords provide additional context without complicating the output space.

* * 2. **Model choice trade-off:**  
   Compare using `distilbert-base-cased` vs `bert-base-cased` in terms of:
   - expected accuracy/F1
   - training/inference cost
   - when you would prefer each one

Expected Accuracy and F1
bert-base-cased is the stronger model. It uses 12 encoder layers to build deep, hierarchical representations of text. distilbert-base-cased, which is a "distilled" version of BERT, typically retains about 95–97% of the original's performance. In complex NER tasks where subtle context (like distinguishing between a person's name and a brand name) is vital, the full BERT model will almost always yield a higher F1 score.

Training and Inference Cost
distilbert-base-cased is designed for efficiency. It has 40% fewer parameters and is roughly 60% faster than BERT. This means you can use larger batch sizes during training, and your model will respond much quicker in a production environment. bert-base-cased is significantly "heavier," requiring more VRAM and more time to process each token, which increases the cost of both training and hosting the model.

When to prefer each
Choose DistilBERT if you are deploying to environments with limited resources (like mobile devices or web browsers), if you need to minimize latency for real-time user interactions, or if your dataset is large and you need to keep training costs down. It is the "good enough" choice for most production use cases.

Choose BERT-base if your priority is squeezing every possible point out of your F1 score. This is common in academic research, Kaggle competitions, or high-stakes industries like legal or medical document processing where a 2% improvement in accuracy justifies the extra compute cost.

---

> **“In theory there is no difference between theory and practice. In practice there is.”**

This notebook is where Transformers stop being magic and start being engineering.

*Wish you luck,*  
**Sina Bayrami**

---